In [ ]:
# ==================== CELL 1 | SETUP ====================
import os, sys, importlib

# ---- EDIT THIS ONE LINE: the folder that holds the input data files (see README, "Data") ----
DATA_DIR = 'data'
# On Google Colab with the files in your Drive, use for example:
# DATA_DIR = '/content/drive/MyDrive/Ranking_Health_Facility'
# ---------------------------------------------------------------------------------------------
if DATA_DIR.startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')

PAPER_DIR = DATA_DIR                      # the name the later cells use
OUT_DIR   = f'{PAPER_DIR}/R_series_results'
os.makedirs(OUT_DIR, exist_ok=True)

# nas_core, nas_rank and build_facilities sit next to this notebook (or in DATA_DIR)
for _p in (os.path.abspath(PAPER_DIR), os.getcwd()):
    if _p not in sys.path:
        sys.path.insert(0, _p)
importlib.invalidate_caches()

import nas_core as nas, nas_rank as rk, build_facilities as bf
importlib.reload(nas); importlib.reload(rk); importlib.reload(bf)

import numpy as np, pandas as pd, matplotlib.pyplot as plt

print('libraries OK  |  PAPER_DIR =', PAPER_DIR)

## Stage 0: build the working files from the licensed weekly patterns

These five cells read the 48 weekly foot-traffic files in `DATA_DIR/NYC-2024-Weekly-Patterns/` and write `practitioners_raw.parquet`, `parent_map.parquet`, `panel_manhattan_weekly.parquet`, `catchment_annual.parquet` and `facilities_manhattan_2024_annual.parquet`. The first, second and third take roughly 5-20 minutes each. If those files are already in `DATA_DIR`, skip to the next section.

In [ ]:
# ============ STAGE 0a | practitioner table from the weekly files (about 20 minutes) ============
# ============ CELL 0 (replacement) | rebuild with working catchment ============
import re, json, importlib, numpy as np, pandas as pd
importlib.invalidate_caches()
import build_facilities as bf, nas_core as nas
importlib.reload(bf)

NYC = ('36005','36047','36061','36081','36085')

def _catchment4(js):
    """(n_cbgs, entropy, local_share, manhattan_share) -- handles '''' escaping."""
    if not isinstance(js, str): return (np.nan,)*4
    s = re.sub(r'"{2,}', '"', js.strip()).strip("'").strip()
    if not s.startswith('{'): return (np.nan,)*4
    try: d = json.loads(s)
    except Exception: return (np.nan,)*4
    it = [(str(k), float(v)) for k,v in d.items()
          if isinstance(v,(int,float)) and v > 0]
    if not it: return (0.0, 0.0, np.nan, np.nan)
    tot = sum(v for _,v in it); p = np.array([v/tot for _,v in it])
    return (float(len(it)), float(-(p*np.log(p)).sum()),
            sum(v for k,v in it if k[:5] in NYC)/tot,
            sum(v for k,v in it if k[:5]=='36061')/tot)

# patch the module's parser, then rebuild (~20 min, one pass)
def _patched(js):
    a = _catchment4(js); return (a[0], a[1])
bf._catchment = _patched

fac = bf.build(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns',
               naics=bf.NAICS_DEFAULT, annual_min_visits=1, min_weeks=1)
print('\nraw practitioners:', len(fac))
fac.to_parquet(f'{PAPER_DIR}/practitioners_raw.parquet', index=False)

In [ ]:
# ============ STAGE 0b | parent-venue map (POIs that inherit a building's traffic) ============
import gzip, pandas as pd, numpy as np, build_facilities as bf

pairs = bf.find_weekly_files(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns')
keep = []
for i,(d,p) in enumerate(pairs,1):
    w = pd.read_csv(gzip.open(p,'rt'),
                    usecols=['placekey','parent_placekey','poi_cbg'],
                    low_memory=False)
    cty = (pd.to_numeric(w.poi_cbg,errors='coerce').astype('Int64')
             .astype(str).str.zfill(12).str[:5])
    keep.append(w[cty=='36061'][['placekey','parent_placekey']])
    print(f'  [{i}/{len(pairs)}]', end='\r')

par = pd.concat(keep).drop_duplicates('placekey')
par.to_parquet(f'{PAPER_DIR}/parent_map.parquet', index=False)

is_child  = par.parent_placekey.notna()
is_parent = par.placekey.isin(par.parent_placekey.dropna())
print(f'\nplacekeys: {len(par):,}')
print(f'  children (inherit building traffic): {is_child.sum():,} ({is_child.mean():.1%})')
print(f'  parents  (are buildings)           : {is_parent.sum():,}')
print(f'  standalone (clean)                 : {(~is_child & ~is_parent).sum():,}')

In [ ]:
# ============ STAGE 0c | weekly panel and pooled annual catchment ============
import re, json, importlib, numpy as np, pandas as pd, warnings
from collections import defaultdict
importlib.invalidate_caches()
import build_facilities as bf; importlib.reload(bf)

NYC = ('36005','36047','36061','36081','36085')

def parse_cbgs(js):
    if not isinstance(js, str): return {}
    s = re.sub(r'"{2,}', '"', js.strip()).strip("'").strip()
    if not s.startswith('{'): return {}
    try: d = json.loads(s)
    except Exception: return {}
    return {str(k): float(v) for k, v in d.items()
            if isinstance(v, (int, float)) and v > 0}

pairs = bf.find_weekly_files(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns')
cols  = [c for c in bf.WANT if c in set(bf.header_of(pairs[0][1]))]

pool, frames = defaultdict(lambda: defaultdict(float)), []
for i, (d, p) in enumerate(pairs, 1):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        f = bf._read_filtered(p, cols, bf.NAICS_DEFAULT)
    if len(f):
        for pk, js in zip(f['placekey'], f['visitor_home_cbgs']):
            for k, v in parse_cbgs(js).items():
                pool[pk][k] += v
        f['_week'] = d
        frames.append(f.drop(columns=['visitor_home_cbgs']))
    print(f'  [{i:>2}/{len(pairs)}] {d}  {len(f):>6,}', end='\r')

panel = pd.concat(frames, ignore_index=True)
panel.to_parquet(f'{PAPER_DIR}/panel_manhattan_weekly.parquet', index=False)
print(f'\npanel saved: {len(panel):,} facility-weeks')

rows = []
for pk, d in pool.items():
    tot = sum(d.values()); p = np.array(list(d.values()))/tot
    rows.append({'placekey': pk,
                 'catch_breadth_annual': len(d),
                 'catch_entropy_annual': float(-(p*np.log(p)).sum()),
                 'local_share':     sum(v for k,v in d.items() if k[:5] in NYC)/tot,
                 'manhattan_share': sum(v for k,v in d.items() if k[:5]=='36061')/tot})
catch = pd.DataFrame(rows)
catch.to_parquet(f'{PAPER_DIR}/catchment_annual.parquet', index=False)

print(catch[['catch_breadth_annual','local_share','manhattan_share']].describe().round(3))

In [ ]:
# ============ STAGE 0d | building-level facility table (runs in seconds) ============
# Prereq: the setup cell has run (nas_core, nas_rank, build_facilities imported).
# Reads only saved files, so it survives a runtime restart.
import numpy as np, pandas as pd, nas_core as nas, build_facilities as bf
from sklearn.neighbors import NearestNeighbors
from scipy.spatial import cKDTree

GRID_M     = 75      # 75 m: 50 m was splitting single buildings across cells
BLD_THRESH = 500     # chosen from the sweep below, not derived by a rule

# ---- 1. load the three saved artefacts ------------------------------------
prac  = pd.read_parquet(f'{PAPER_DIR}/practitioners_raw.parquet')
par   = pd.read_parquet(f'{PAPER_DIR}/parent_map.parquet')
catch = pd.read_parquet(f'{PAPER_DIR}/catchment_annual.parquet')
print(f'practitioners {len(prac):,} | parent map {len(par):,} | catchment {len(catch):,}')

# ---- 2. drop POIs inheriting parent-venue traffic -------------------------
f = prac.merge(par, on='placekey', how='left')
n0 = len(f)
f = f[f.parent_placekey.isna()].drop(columns=['parent_placekey'])
print(f'dropped {n0-len(f):,} child POIs ({(n0-len(f))/n0:.1%}) -> {len(f):,} standalone')

f = f.drop(columns=[c for c in ['catchment_breadth','catchment_entropy'] if c in f.columns])
f = f.merge(catch, on='placekey', how='left').reset_index(drop=True)

# ---- 3. aggregate to buildings BEFORE thresholding ------------------------
xy = nas.local_xy(f.latitude.to_numpy(), f.longitude.to_numpy())
f['_bld'] = [f'{a:.0f}_{b:.0f}' for a, b in np.round(xy/GRID_M)*GRID_M]
ws = lambda s, w: np.average(s.fillna(s.mean()), weights=w) if w.sum() > 0 else np.nan
g = f.groupby('_bld')

bld = pd.DataFrame({
    'n_practices':  g.size(),
    'annual_visits': g['annual_visits'].sum(),
    'latitude':  g['latitude'].mean(),
    'longitude': g['longitude'].mean(),
    'poi_cbg':   g['poi_cbg'].first(),
    'anchor_practice': g.apply(lambda x: x.loc[x.annual_visits.idxmax(), 'location_name'], include_groups=False),
    'weeks_present': g['weeks_present'].max(),
    'mean_dwell': g.apply(lambda x: ws(x.mean_dwell, x.annual_visits), include_groups=False),
    'mean_distance_from_home': g.apply(lambda x: ws(x.mean_distance_from_home, x.annual_visits), include_groups=False),
    'catch_breadth_annual': g['catch_breadth_annual'].max(),
    'catch_entropy_annual': g.apply(lambda x: ws(x.catch_entropy_annual, x.annual_visits), include_groups=False),
    'local_share':     g.apply(lambda x: ws(x.local_share, x.annual_visits), include_groups=False),
    'manhattan_share': g.apply(lambda x: ws(x.manhattan_share, x.annual_visits), include_groups=False),
    'weekend_share':    g['weekend_share'].mean(),
    'visit_volatility': g['visit_volatility'].mean(),
}).reset_index(drop=True)
print(f'\nbuildings at {GRID_M} m grid: {len(bld):,}   median practices/building: {bld.n_practices.median():.0f}')

# ---- 4. threshold sweep (keep this table for the paper) -------------------
print('\n--- building-level threshold sensitivity ---')
rows = []
for t in [250, 500, 1000, 2000, 3000, 5000, 10000, 20000]:
    n = int((bld.annual_visits >= t).sum()); rows.append({'min_annual_visits': t, 'N': n})
    print(f'   >= {t:>6,} -> N = {n:>5,}')
sweep = pd.DataFrame(rows)

out = bld[bld.annual_visits >= BLD_THRESH].reset_index(drop=True)
print(f'\nthreshold {BLD_THRESH:,}  ->  N = {len(out):,}')

# ---- 5. geometry + integrity checks ---------------------------------------
xyb = nas.local_xy(out.latitude.to_numpy(), out.longitude.to_numpy())
d = NearestNeighbors(n_neighbors=min(11, len(out))).fit(xyb).kneighbors(xyb)[0]
print(f'\nmedian 10th-neighbour distance: {np.median(d[:, -1]):.0f} m')
print(f'  share with 10th neighbour < 50 m: {(d[:, -1] < 50).mean():.1%}  (want ~0)')

dm = out.duplicated(subset=['catch_breadth_annual','local_share'], keep=False)
tree = cKDTree(xyb)
near = [(i,j) for i,j in tree.query_pairs(80)
        if out.catch_breadth_annual.iloc[i] == out.catch_breadth_annual.iloc[j]
        and abs(out.local_share.iloc[i] - out.local_share.iloc[j]) < 1e-9]
print(f'duplicate catchment signatures: {int(dm.sum())} ({dm.mean():.1%})')
print(f'  of which within 80 m (grid splitting): {len(near)}')
if dm.mean() > 0.05 and len(near) > 0.5 * dm.sum():
    print(f'  >> still splitting buildings. Try GRID_M = {GRID_M + 25}.')

# ---- 6. spatial context + save --------------------------------------------
out = bf.add_spatial_context(out)
out.to_parquet(f'{PAPER_DIR}/facilities_manhattan_2024_annual.parquet', index=False)
sweep.to_csv(f'{PAPER_DIR}/threshold_sensitivity.csv', index=False)
print(f'\nwritten: facilities_manhattan_2024_annual.parquet  (N = {len(out):,})')
print(out[['anchor_practice','n_practices','annual_visits','local_share',
           'catch_breadth_annual','n_facilities_500m']].head(8).to_string(index=False))

## Analysis

In [ ]:
# ---- what's actually on disk, and does the panel keep home CBGs? ----
import os
for f in ['panel_manhattan_weekly.parquet','catchment_annual.parquet',
          'practitioners_raw.parquet','parent_map.parquet',
          'facilities_manhattan_2024_annual.parquet']:
    p = f'{PAPER_DIR}/{f}'
    print(f'{"OK " if os.path.exists(p) else "-- "}{f}')

pan = pd.read_parquet(f'{PAPER_DIR}/panel_manhattan_weekly.parquet')
print('\npanel columns:', list(pan.columns))
print('has visitor_home_cbgs:', 'visitor_home_cbgs' in pan.columns)

catch = pd.read_parquet(f'{PAPER_DIR}/catchment_annual.parquet')
print('catchment columns:', list(catch.columns))

In [ ]:
# ==================== CELL | recover per-facility CBG distributions ====================
import re, json, gzip, warnings
from collections import defaultdict

def parse_cbgs(js):
    """Handles the quadruple-escaped quotes in this vendor's CSV export."""
    if not isinstance(js, str): return {}
    s = re.sub(r'"{2,}', '"', js.strip()).strip("'").strip()
    if not s.startswith('{'): return {}
    try: d = json.loads(s)
    except Exception: return {}
    return {str(k): float(v) for k, v in d.items()
            if isinstance(v, (int, float)) and v > 0}

pairs = bf.find_weekly_files(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns')
cols  = [c for c in bf.WANT if c in set(bf.header_of(pairs[0][1]))]

pool = defaultdict(lambda: defaultdict(float))
for i, (d, p) in enumerate(pairs, 1):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        f = bf._read_filtered(p, cols, bf.NAICS_DEFAULT)
    if len(f) and 'visitor_home_cbgs' in f.columns:
        for pk, js in zip(f['placekey'], f['visitor_home_cbgs']):
            for k, v in parse_cbgs(js).items():
                pool[pk][k] += v
    print(f'  [{i:>2}/{len(pairs)}] {d}', end='\r')

# long format: one row per (facility, home CBG) with the annual visit weight
rows = [{'placekey': pk, 'home_cbg': cbg, 'visits': v}
        for pk, d in pool.items() for cbg, v in d.items()]
flows = pd.DataFrame(rows)
flows.to_parquet(f'{PAPER_DIR}/facility_cbg_flows.parquet', index=False)

print(f'\n\nflows: {len(flows):,} facility-CBG pairs')
print(f'  facilities: {flows.placekey.nunique():,}')
print(f'  distinct home CBGs: {flows.home_cbg.nunique():,}')
print(f'  median CBGs per facility: {flows.groupby("placekey").size().median():.0f}')
print('\nCBG format check (need 12-digit block groups):')
print(flows.home_cbg.head(5).tolist())
print('  length distribution:', flows.home_cbg.str.len().value_counts().head().to_dict())

In [ ]:
svi = pd.read_csv(f'{PAPER_DIR}/NEWYORK.csv', low_memory=False)
print('shape:', svi.shape)
print('\nkey columns present:', [c for c in svi.columns
      if any(k in c.upper() for k in ['FIPS','RPL_THEME','E_TOTPOP','COUNTY','LOCATION'])])
print('\nFIPS sample:', svi.FIPS.astype(str).head(3).tolist())
print('FIPS length distribution:', svi.FIPS.astype(str).str.len().value_counts().to_dict())
print('\nRPL_THEMES range:', svi.RPL_THEMES.min(), 'to', svi.RPL_THEMES.max())
print('  -999 count:', int((svi.RPL_THEMES == -999).sum()))
print('\nManhattan tracts (36061):',
      int(svi.FIPS.astype(str).str.zfill(11).str[:5].eq('36061').sum()))

In [ ]:
# ==================== CELL | SVI join: patient vulnerability per facility ====================
svi = pd.read_csv(f'{PAPER_DIR}/NEWYORK.csv', low_memory=False)
svi['FIPS'] = svi.FIPS.astype(str).str.zfill(11)
for c in ['RPL_THEMES','RPL_THEME1','RPL_THEME2','RPL_THEME3','RPL_THEME4']:
    svi[c] = svi[c].replace(-999, np.nan)
svi_t = svi.set_index('FIPS')[['RPL_THEMES','RPL_THEME1','RPL_THEME2',
                               'RPL_THEME3','RPL_THEME4','E_TOTPOP']]

flows = pd.read_parquet(f'{PAPER_DIR}/facility_cbg_flows.parquet')
flows = flows[flows.home_cbg.str.len() == 12].copy()
flows['tract'] = flows.home_cbg.str[:11]

n_all = flows.visits.sum()
flows = flows.join(svi_t, on='tract', how='left')
matched = flows.RPL_THEMES.notna()
print(f'flow rows: {len(flows):,}   matched to NY SVI: {matched.mean():.1%}')
print(f'visit weight matched: {flows.loc[matched,"visits"].sum()/n_all:.1%}')
print('  (unmatched = out-of-state catchment; see Methods note)')

m = flows[matched]
def wmean(g, col):
    w = g.visits.to_numpy()
    return np.average(g[col].to_numpy(), weights=w) if w.sum() > 0 else np.nan

g = m.groupby('placekey')
pv = pd.DataFrame({
    'patient_svi':        g.apply(lambda x: wmean(x,'RPL_THEMES'), include_groups=False),
    'patient_svi_socio':  g.apply(lambda x: wmean(x,'RPL_THEME1'), include_groups=False),
    'patient_svi_house':  g.apply(lambda x: wmean(x,'RPL_THEME2'), include_groups=False),
    'patient_svi_minor':  g.apply(lambda x: wmean(x,'RPL_THEME3'), include_groups=False),
    'patient_svi_trans':  g.apply(lambda x: wmean(x,'RPL_THEME4'), include_groups=False),
    'ny_catchment_visits': g.visits.sum(),
    'n_ny_tracts':         g.tract.nunique(),
}).reset_index()
pv.to_parquet(f'{PAPER_DIR}/patient_vulnerability.parquet', index=False)
print(f'\npatient vulnerability computed for {len(pv):,} facilities')
print(pv[['patient_svi','n_ny_tracts']].describe().round(3))

# ---- the diagnostic: do facilities in vulnerable areas serve vulnerable patients? ----
fac = pd.read_parquet(f'{PAPER_DIR}/facilities_manhattan_2024_annual.parquet')
prac = pd.read_parquet(f'{PAPER_DIR}/practitioners_raw.parquet')
prac['tract'] = (pd.to_numeric(prac.poi_cbg, errors='coerce').astype('Int64')
                 .astype(str).str.zfill(12).str[:11])
prac = prac.merge(svi_t[['RPL_THEMES']].rename(columns={'RPL_THEMES':'site_svi'}),
                  left_on='tract', right_index=True, how='left')
prac = prac.merge(pv[['placekey','patient_svi']], on='placekey', how='left')

d = prac.dropna(subset=['site_svi','patient_svi'])
print(f'\n--- DIAGNOSTIC (n={len(d):,} practitioners) ---')
print(f'site SVI    : median {d.site_svi.median():.3f}')
print(f'patient SVI : median {d.patient_svi.median():.3f}')
print(f'correlation site vs patient SVI: {nas.spearman(d.site_svi, d.patient_svi):.3f}')

fig, ax = plt.subplots(figsize=(5,5))
ax.scatter(d.site_svi, d.patient_svi, s=4, alpha=.25)
ax.plot([0,1],[0,1],'k--',lw=.8)
ax.set_xlabel('SVI where the facility IS'); ax.set_ylabel('SVI of who it SERVES')
ax.set_title('Does location predict patient vulnerability?')
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/svi_site_vs_patient.png', dpi=200); plt.show()

In [ ]:
d2 = d.dropna(subset=['patient_svi'])
for lo, hi in [(1,3),(4,10),(11,30),(31,100),(101,10**6)]:
    s = pv[(pv.n_ny_tracts >= lo) & (pv.n_ny_tracts <= hi)]
    if len(s):
        print(f'  {lo:>4}-{hi if hi<10**6 else "inf":>5} tracts  n={len(s):>6,}  '
              f'patient_svi sd={s.patient_svi.std():.3f}  mean={s.patient_svi.mean():.3f}')

In [ ]:
MIN_TRACTS = 10
pv_clean = pv[pv.n_ny_tracts >= MIN_TRACTS].reset_index(drop=True)
pv_clean.to_parquet(f'{PAPER_DIR}/patient_vulnerability.parquet', index=False)
print(f'screened at >={MIN_TRACTS} NY tracts: {len(pv):,} -> {len(pv_clean):,} practitioners')

# how many of the 1,231 buildings survive?
prac2 = pd.read_parquet(f'{PAPER_DIR}/practitioners_raw.parquet')
par   = pd.read_parquet(f'{PAPER_DIR}/parent_map.parquet')
p2 = (prac2.merge(par, on='placekey', how='left')
           .query('parent_placekey.isna()')
           .merge(pv_clean, on='placekey', how='inner'))
xy2 = nas.local_xy(p2.latitude.to_numpy(), p2.longitude.to_numpy())
p2['_bld'] = [f'{a:.0f}_{b:.0f}' for a,b in np.round(xy2/75)*75]
bsvi = p2.groupby('_bld').apply(
    lambda x: np.average(x.patient_svi, weights=x.annual_visits), include_groups=False)
print(f'buildings with a patient_svi estimate: {len(bsvi):,} (of 1,231)')

In [ ]:
# ==================== CELL | provisional administrative target ====================
svi = pd.read_csv(f'{PAPER_DIR}/NEWYORK.csv', low_memory=False)
svi['FIPS'] = svi.FIPS.astype(str).str.zfill(11)
svi['RPL_THEMES'] = svi.RPL_THEMES.replace(-999, np.nan)
svi['E_TOTPOP']   = svi.E_TOTPOP.replace(-999, np.nan)
man = svi[svi.FIPS.str[:5] == '36061'][['FIPS','RPL_THEMES','E_TOTPOP']].dropna()
print(f'Manhattan tracts with SVI + population: {len(man)}   '
      f'total pop {man.E_TOTPOP.sum():,.0f}')

fac = pd.read_parquet(f'{PAPER_DIR}/facilities_manhattan_2024_annual.parquet')
fac['tract'] = (pd.to_numeric(fac.poi_cbg, errors='coerce').astype('Int64')
                .astype(str).str.zfill(12).str[:11])
fac = fac.merge(man.rename(columns={'FIPS':'tract','RPL_THEMES':'site_svi',
                                    'E_TOTPOP':'tract_pop'}), on='tract', how='left')

# demand = vulnerability-weighted population;  supply = local facility count
fac['admin_demand'] = fac.site_svi * fac.tract_pop
fac['need_admin']   = np.log1p(fac.admin_demand) - np.log1p(5 + fac.n_facilities_1000m)

print(f'\nmatched to a tract: {fac.need_admin.notna().mean():.1%}')
print(fac.need_admin.describe().round(3))
print(f'\ndistinct demand values: {fac.admin_demand.nunique()} across {len(fac)} buildings')
print('  (tract-level resolution: buildings in the same tract share a demand value)')

# how does it relate to the mobility-derived target?
print(f'\nrank corr with mobility need_index: {nas.spearman(fac.need_admin, fac.need_index):.3f}')
print(f'rank corr with annual_visits      : {nas.spearman(fac.need_admin, fac.annual_visits):.3f}')

fac.to_parquet(f'{PAPER_DIR}/facilities_manhattan_2024_annual.parquet', index=False)
print('\nsaved with need_admin')

In [ ]:
d = fac.dropna(subset=['need_admin']).reset_index(drop=True)
pv = pd.read_parquet(f'{PAPER_DIR}/patient_vulnerability.parquet')

# mobility features only; nothing administrative, nothing from the target
X_m, fn = nas.feature_matrix(d, exclude=[
    'need_admin','need_index','local_visits','annual_visits','local_share',
    'admin_demand','site_svi','tract_pop',
    'n_facilities_500m','n_facilities_1000m','n_facilities_2000m',
    'dist_nearest_facility_m','dist_5th_nearest_m','poi_cbg','weeks_present'])

y_a = d.need_admin.to_numpy(float)
xy_a = nas.local_xy(d.latitude.to_numpy(), d.longitude.to_numpy())
ok = np.isfinite(y_a) & np.isfinite(xy_a).all(1)
X_m, y_a, xy_a = X_m[ok], y_a[ok], xy_a[ok]

t = nas.staircase(X_m, y_a, xy_a, k=10)
print(f'N={len(y_a)}  d={X_m.shape[1]}  features={fn}')
print(f'\nmobility features -> administrative need')
print(f'  rho own-features    = {t.rho["Ridge"]:.4f}')
print(f'  rho neighbour-aware = {t.rho["Ridge-Spatial"]:.4f}')
print(f'  gain                = {t.rho["Ridge-Spatial"]-t.rho["Ridge"]:+.4f}')

print('\nper-feature correlation with administrative need:')
print(pd.Series({c: nas.spearman(X_m[:,i], y_a) for i,c in enumerate(fn)}
                ).round(3).sort_values(key=abs, ascending=False).to_string())

In [ ]:
# how much of the target is between-tract vs within-tract?
dd = d[ok].reset_index(drop=True)
grp = dd.groupby('tract').need_admin
print(f'tracts: {dd.tract.nunique()}  buildings: {len(dd)}')
print(f'within-tract SD: {grp.transform("std").mean():.4f}   overall SD: {dd.need_admin.std():.4f}')

# control 1: does a plain tract-mean predictor do just as well?
tm = grp.transform('mean').to_numpy()
print(f'\ntract-mean alone: rho = {nas.spearman(y_a, tm):.4f}')

# control 2: coordinates only, no facility features at all
print(f'coordinates only : rho = {nas.spearman(y_a, nas.ridge_oof(xy_a, y_a, nas.spatial_folds(xy_a,5,0))):.4f}')

# control 3: the noise benchmark, as before
folds = nas.spatial_folds(xy_a, 5, 0)
nr = np.array([nas.spearman(y_a, nas.ridge_oof(
    np.hstack([X_m, np.random.default_rng(s).normal(0,1,(len(X_m), 2*X_m.shape[1]))]),
    y_a, folds)) for s in range(20)])
print(f'\nnoise control: {nr.mean():.4f} (sd {nr.std():.4f})   neighbourhood: {t.rho["Ridge-Spatial"]:.4f}')
print(f'  neighbourhood is {(t.rho["Ridge-Spatial"]-nr.mean())/nr.std():+.1f} SD vs noise')

In [ ]:
tr = dd.groupby('tract').agg(
    need_admin=('need_admin','first'),
    lat=('latitude','mean'), lon=('longitude','mean'),
    **{c: (c,'mean') for c in fn}).reset_index()
tr = tr[np.isfinite(tr.need_admin)].reset_index(drop=True)

Xt  = tr[list(fn)].to_numpy(float)
yt  = tr.need_admin.to_numpy(float)
xyt = nas.local_xy(tr.lat.to_numpy(), tr.lon.to_numpy())
tt  = nas.staircase(Xt, yt, xyt, k=10)

print(f'N={len(tr)} tracts')
print(f'  rho own-features    = {tt.rho["Ridge"]:.4f}')
print(f'  rho neighbour-aware = {tt.rho["Ridge-Spatial"]:.4f}')
print(f'  gain                = {tt.rho["Ridge-Spatial"]-tt.rho["Ridge"]:+.4f}')

In [ ]:
folds_t = nas.spatial_folds(xyt, 5, 0)

# 1. seed stability
g = np.array([nas.staircase(Xt, yt, xyt, k=10, seed=s).rho['Ridge-Spatial'] -
              nas.staircase(Xt, yt, xyt, k=10, seed=s).rho['Ridge'] for s in range(10)])
print(f'[1] gain across 10 seeds: mean={g.mean():+.4f} sd={g.std():.4f} positive {(g>0).sum()}/10')

# 2. noise control — does 14 random columns gain anything?
nr = np.array([nas.spearman(yt, nas.ridge_oof(
    np.hstack([Xt, np.random.default_rng(s).normal(0,1,(len(Xt), 2*Xt.shape[1]))]),
    yt, folds_t)) for s in range(20)])
print(f'[2] noise: {nr.mean():.4f} (sd {nr.std():.4f})  neighbourhood: {tt.rho["Ridge-Spatial"]:.4f}'
      f'  -> {(tt.rho["Ridge-Spatial"]-nr.mean())/nr.std():+.1f} SD')

# 3. coordinates only — is this just geography?
print(f'[3] coordinates only: rho = {nas.spearman(yt, nas.ridge_oof(xyt, yt, folds_t)):.4f}')

# 4. neighbour-mean of the TARGET — the strongest alternative explanation:
#    if adjacent tracts have similar need, a spatial lag alone may explain it
idx_t, w_t = nas.knn_graph(xyt, 10)
ylag = np.einsum('nk,nk->n', w_t, yt[idx_t])
print(f'[4] neighbour-mean of target alone: rho = {nas.spearman(yt, ylag):.4f}')
print('    (high value = the target is spatially autocorrelated and x_bar is a proxy for it)')

# 5. bootstrap the gain
rng, n = np.random.default_rng(0), len(yt)
pr, pn = tt.pred['Ridge'], tt.pred['Ridge-Spatial']
b = np.array([nas.spearman(yt[i], pn[i]) - nas.spearman(yt[i], pr[i])
              for i in (rng.integers(0,n,n) for _ in range(3000))])
print(f'[5] gain 95% CI [{np.percentile(b,2.5):+.4f}, {np.percentile(b,97.5):+.4f}]  P(>0)={(b>0).mean():.3f}')

In [ ]:
svi = pd.read_csv(f'{PAPER_DIR}/NEWYORK.csv', low_memory=False)
cand = [c for c in svi.columns if any(k in c.upper()
        for k in ['LAT','LON','LNG','X','Y','CENTROID','GEO','AREA','SQMI'])]
print('possible geometry columns:', cand)
print(svi[['LOCATION']].head(2).to_string(index=False))

In [ ]:
# ==================== building-level administrative target ====================
LAMBDA_M = 800     # decay length in metres

svi = pd.read_csv(f'{PAPER_DIR}/NEWYORK.csv', low_memory=False)
svi['FIPS'] = svi.FIPS.astype(str).str.zfill(11)
for c in ['RPL_THEMES','E_TOTPOP']:
    svi[c] = svi[c].replace(-999, np.nan)
man = svi[svi.FIPS.str[:5]=='36061'][['FIPS','RPL_THEMES','E_TOTPOP']].dropna()

# tract centroids from practitioner positions (no shapefile, no network)
prac = pd.read_parquet(f'{PAPER_DIR}/practitioners_raw.parquet')
prac['tract'] = (pd.to_numeric(prac.poi_cbg, errors='coerce').astype('Int64')
                 .astype(str).str.zfill(12).str[:11])
cen = prac.groupby('tract').agg(lat=('latitude','mean'), lon=('longitude','mean'),
                                n_poi=('placekey','size')).reset_index()
cen = cen.merge(man.rename(columns={'FIPS':'tract'}), on='tract', how='inner')
cen = cen[cen.n_poi >= 3].reset_index(drop=True)
print(f'tracts with centroid + SVI: {len(cen)}  (median {cen.n_poi.median():.0f} POIs each)')

fac  = pd.read_parquet(f'{PAPER_DIR}/facilities_manhattan_2024_annual.parquet')
xy_f = nas.local_xy(fac.latitude.to_numpy(), fac.longitude.to_numpy())
xy_t = nas.local_xy(cen.lat.to_numpy(), cen.lon.to_numpy())

D = np.sqrt(((xy_f[:,None,:] - xy_t[None,:,:])**2).sum(-1))
W = np.exp(-D / LAMBDA_M)
demand = cen.RPL_THEMES.to_numpy() * cen.E_TOTPOP.to_numpy()

fac['admin_demand_decay'] = W @ demand
fac['need_admin_bld'] = (np.log1p(fac.admin_demand_decay)
                         - np.log1p(5 + fac.n_facilities_1000m))

t_id = (pd.to_numeric(fac.poi_cbg, errors='coerce').astype('Int64')
        .astype(str).str.zfill(12).str[:11])
wsd  = fac.assign(t=t_id).groupby('t').need_admin_bld.transform('std').mean()
print(f'\ndistinct values: {fac.need_admin_bld.nunique()} of {len(fac)} buildings  (was 177)')
print(f'within-tract SD: {wsd:.4f}   overall SD: {fac.need_admin_bld.std():.4f}')
print(f'  ratio {wsd/fac.need_admin_bld.std():.3f}  (was 0.047; higher = real within-tract variation)')

fac.to_parquet(f'{PAPER_DIR}/facilities_manhattan_2024_annual.parquet', index=False)
print('saved')

In [ ]:
d = fac.dropna(subset=['need_index','need_admin_bld']).reset_index(drop=True)
Xc, fnc = nas.feature_matrix(d, exclude=[
    'need_index','need_admin_bld','need_admin',          # <- need_admin was missing
    'local_visits','annual_visits','local_share',
    'admin_demand','admin_demand_decay','site_svi','tract_pop',
    'n_facilities_500m','n_facilities_1000m','n_facilities_2000m',
    'dist_nearest_facility_m','dist_5th_nearest_m','poi_cbg','weeks_present'])
xyc = nas.local_xy(d.latitude.to_numpy(), d.longitude.to_numpy())
print('features:', fnc)     # must be the same 7 as before

for name, col in [('contrast  (need_index)', 'need_index'),
                  ('field     (need_admin_bld)', 'need_admin_bld')]:
    yv = d[col].to_numpy(float)
    g = np.array([nas.staircase(Xc, yv, xyc, k=10, seed=s).rho['Ridge-Spatial'] -
                  nas.staircase(Xc, yv, xyc, k=10, seed=s).rho['Ridge'] for s in range(10)])
    t = nas.staircase(Xc, yv, xyc, k=10)
    print(f'{name:<28} own={t.rho["Ridge"]:.4f}  nbr={t.rho["Ridge-Spatial"]:.4f}  '
          f'gain={g.mean():+.4f} (sd {g.std():.4f})')

# also report the within-tract SD ratio for the field target — I never saw it print
t_id = (pd.to_numeric(d.poi_cbg, errors='coerce').astype('Int64')
        .astype(str).str.zfill(12).str[:11])
wsd = d.assign(t=t_id).groupby('t').need_admin_bld.transform('std').mean()
print(f'\nfield target within-tract SD ratio: {wsd/d.need_admin_bld.std():.3f}')

In [ ]:
for lam in (400, 800, 1600):
    W = np.exp(-np.sqrt(((xy_f[:,None,:]-xy_t[None,:,:])**2).sum(-1)) / lam)
    yv = np.log1p(W @ demand) - np.log1p(5 + fac.n_facilities_1000m.to_numpy())
    yv = yv[d.index] if len(yv) != len(d) else yv
    g = np.array([nas.staircase(Xc, yv, xyc, k=10, seed=s).rho['Ridge-Spatial'] -
                  nas.staircase(Xc, yv, xyc, k=10, seed=s).rho['Ridge'] for s in range(5)])
    print(f'lambda={lam:>5} m   gain = {g.mean():+.4f}')

In [ ]:
fac = pd.read_parquet(f'{PAPER_DIR}/facilities_manhattan_2024_annual.parquet')
missing = [c for c in ['need_index','need_admin_bld','admin_demand_decay'] if c not in fac.columns]
print('missing:', missing or 'none', '| N =', len(fac))

In [ ]:
# ==================== PHASE 1 | gap analysis + disagreement quadrants ====================
# PRE-REGISTERED (fixed before results are seen):
#   quadrants: split at the study-wide MEDIAN of each axis
#   hidden-high = site_svi < median(site) AND patient_svi > median(patient)
#   sensitivity: quintile version (site in Q1-Q2 AND patient in Q4-Q5)
#   large gap  = (patient_svi - site_svi) > 0.20        (20 percentile points)

svi = pd.read_csv(f'{PAPER_DIR}/NEWYORK.csv', low_memory=False)
svi['FIPS'] = svi.FIPS.astype(str).str.zfill(11)
svi['RPL_THEMES'] = svi.RPL_THEMES.replace(-999, np.nan)
svi_t = svi.set_index('FIPS')['RPL_THEMES']

prac = pd.read_parquet(f'{PAPER_DIR}/practitioners_raw.parquet')
par  = pd.read_parquet(f'{PAPER_DIR}/parent_map.parquet')
pv   = pd.read_parquet(f'{PAPER_DIR}/patient_vulnerability.parquet')   # screened >=10 tracts

p = (prac.merge(par, on='placekey', how='left').query('parent_placekey.isna()')
         .merge(pv[['placekey','patient_svi']], on='placekey', how='inner'))
p['tract'] = (pd.to_numeric(p.poi_cbg, errors='coerce').astype('Int64')
              .astype(str).str.zfill(12).str[:11])
p['site_svi'] = p.tract.map(svi_t)
xyp = nas.local_xy(p.latitude.to_numpy(), p.longitude.to_numpy())
p['_bld'] = [f'{a:.0f}_{b:.0f}' for a, b in np.round(xyp/75)*75]

g = p.groupby('_bld')
B = pd.DataFrame({
    'patient_svi': g.apply(lambda x: np.average(x.patient_svi, weights=x.annual_visits), include_groups=False),
    'site_svi':    g.apply(lambda x: np.average(x.site_svi.dropna(),
                        weights=x.annual_visits[x.site_svi.notna()]) if x.site_svi.notna().any() else np.nan,
                        include_groups=False),
    'visits':      g.annual_visits.sum(),
    'lat': g.latitude.mean(), 'lon': g.longitude.mean(),
}).dropna().reset_index()
print(f'buildings with both measures: {len(B)}')

# ---- 1. quintile agreement (the Hendrix-comparable number) ----------------
qs = lambda s: pd.qcut(s, 5, labels=False)
B['q_site'], B['q_pat'] = qs(B.site_svi), qs(B.patient_svi)
same = (B.q_site == B.q_pat).mean()
print(f'\n[1] same SVI quintile (location vs visitors): {same:.1%}')
print(pd.crosstab(B.q_site, B.q_pat, normalize='all').round(3).to_string())

# ---- 2. the gap distribution ----------------------------------------------
B['gap'] = B.patient_svi - B.site_svi
print(f'\n[2] gap = visitor SVI - site SVI')
print(B.gap.describe().round(3).to_string())
print(f'    share with gap > +0.20: {(B.gap > .20).mean():.1%}   < -0.20: {(B.gap < -.20).mean():.1%}')
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(B.gap, bins=50); ax.axvline(0, c='k', lw=.8); ax.axvline(.20, c='r', ls='--', lw=.8)
ax.set_xlabel('visitor SVI minus site SVI'); ax.set_ylabel('sites')
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/gap_histogram.png', dpi=200); plt.show()

# ---- 3. disagreement quadrants (pre-registered medians) --------------------
ms, mp = B.site_svi.median(), B.patient_svi.median()
B['quad'] = np.select(
    [(B.site_svi>=ms)&(B.patient_svi>=mp), (B.site_svi<ms)&(B.patient_svi<mp),
     (B.site_svi<ms)&(B.patient_svi>=mp), (B.site_svi>=ms)&(B.patient_svi<mp)],
    ['concordant-high','concordant-low','hidden-high','hidden-low'],
    default='unclassified')
print(f'\n[3] quadrants (median split):')
qt = B.groupby('quad').agg(n=('quad','size'), visit_share=('visits',lambda v: v.sum()/B.visits.sum()),
                           mean_gap=('gap','mean')).round(3)
print(qt.to_string())
hh5 = ((B.q_site<=1)&(B.q_pat>=3)).sum()
print(f'    quintile sensitivity: hidden-high (site Q1-Q2, visitors Q4-Q5) = {hh5}')

xyB = nas.local_xy(B.lat.to_numpy(), B.lon.to_numpy())
cmap = {'concordant-high':'tab:red','concordant-low':'tab:gray',
        'hidden-high':'tab:blue','hidden-low':'tab:orange'}
fig, ax = plt.subplots(figsize=(5.5,6.5))
for q,c in cmap.items():
    m = B.quad==q; ax.scatter(xyB[m,0]/1000, xyB[m,1]/1000, c=c, s=10, label=f'{q} ({m.sum()})')
ax.set_aspect('equal'); ax.legend(fontsize=8); ax.set_xlabel('km E'); ax.set_ylabel('km N')
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/disagreement_map.png', dpi=200); plt.show()
B.to_parquet(f'{PAPER_DIR}/building_svi_quadrants.parquet', index=False)

# ---- 4. single-feature baselines against the field target ------------------
d2 = fac.dropna(subset=['need_admin_bld']).reset_index(drop=True)
X7, fn7 = nas.feature_matrix(d2, exclude=[
    'need_index','need_admin_bld','need_admin','local_visits','annual_visits','local_share',
    'admin_demand','admin_demand_decay','site_svi','tract_pop',
    'n_facilities_500m','n_facilities_1000m','n_facilities_2000m',
    'dist_nearest_facility_m','dist_5th_nearest_m','poi_cbg','weeks_present'])
y2  = d2.need_admin_bld.to_numpy(float)
xy2 = nas.local_xy(d2.latitude.to_numpy(), d2.longitude.to_numpy())
fl  = nas.spatial_folds(xy2, 5, 0)
print(f'\n[4] single-feature baselines vs need_admin_bld  (7-feature model = 0.4839)')
for i, c in enumerate(fn7):
    r = nas.spearman(y2, nas.ridge_oof(X7[:, [i]], y2, fl))
    print(f'    {c:<22} rho = {r:+.4f}')
print(f'    raw annual_visits      rho = {nas.spearman(y2, np.log1p(d2.annual_visits)):+.4f}  (in-sample corr)')
print(f'\n    target decomposition: demand rho = '
      f'{nas.spearman(np.log1p(d2.admin_demand_decay), nas.ridge_oof(X7, np.log1p(d2.admin_demand_decay.to_numpy()), fl)):.4f}'
      f' | supply rho = {nas.spearman(np.log1p(5+d2.n_facilities_1000m), nas.ridge_oof(X7, np.log1p(5+d2.n_facilities_1000m.to_numpy()), fl)):.4f}')

**Needs three state SVI files in `DATA_DIR`:** `NEWYORK.csv`, `NEWJERSEY.csv` and `CONNECTICUT.csv` (CDC/ATSDR SVI 2022, census tracts, one file per state). This cell is a robustness check; the main analysis uses the national file `SVI_US_2022.csv`.

In [ ]:
# ============ TRI-STATE REJOIN | coverage lift + Phase 1 recheck + dwell probe ============
# Note: state SVI files rank percentiles WITHIN each state. Mixing NY/NJ/CT
# percentiles assumes state-relative vulnerability is comparable; flagged for Methods.

frames = []
for st in ['NEWYORK', 'NEWJERSEY', 'CONNECTICUT']:
    s = pd.read_csv(f'{PAPER_DIR}/{st}.csv', low_memory=False)
    s['FIPS'] = s.FIPS.astype(str).str.zfill(11)
    s['RPL_THEMES'] = s.RPL_THEMES.replace(-999, np.nan)
    frames.append(s[['FIPS', 'RPL_THEMES']].assign(state=st))
    print(f'{st:<12} tracts: {len(s):,}')
svi3 = pd.concat(frames).dropna().set_index('FIPS')

flows = pd.read_parquet(f'{PAPER_DIR}/facility_cbg_flows.parquet')
flows = flows[flows.home_cbg.str.len() == 12].copy()
flows['tract'] = flows.home_cbg.str[:11]
flows['st2']   = flows.tract.str[:2]
flows = flows.join(svi3['RPL_THEMES'], on='tract')

W = flows.visits.sum()
print('\nmatch rate by home state (visit-weighted):')
for st2, name in [('36','NY'), ('34','NJ'), ('09','CT')]:
    m = flows[flows.st2 == st2]
    print(f'  {name}: {m.RPL_THEMES.notna().mean():.1%} of rows, '
          f'{m.visits.sum()/W:.1%} of all visit weight')
print(f'TOTAL visit weight matched: {flows.loc[flows.RPL_THEMES.notna(),"visits"].sum()/W:.1%}  (was 70.7%)')

m = flows[flows.RPL_THEMES.notna()]
g = m.groupby('placekey')
pv3 = pd.DataFrame({
    'patient_svi': g.apply(lambda x: np.average(x.RPL_THEMES, weights=x.visits), include_groups=False),
    'n_tracts':    g.tract.nunique(),
}).reset_index()
pv3 = pv3[pv3.n_tracts >= 10].reset_index(drop=True)          # same pre-registered screen
pv3.to_parquet(f'{PAPER_DIR}/patient_vulnerability_tristate.parquet', index=False)
print(f'\npractitioners with tri-state patient_svi (>=10 tracts): {len(pv3):,}')

# ---- rebuild building table and re-run the three headline measures --------
svi_ny = pd.read_csv(f'{PAPER_DIR}/NEWYORK.csv', low_memory=False)
svi_ny['FIPS'] = svi_ny.FIPS.astype(str).str.zfill(11)
site_map = svi_ny.assign(RPL_THEMES=svi_ny.RPL_THEMES.replace(-999, np.nan)).set_index('FIPS')['RPL_THEMES']

prac = pd.read_parquet(f'{PAPER_DIR}/practitioners_raw.parquet')
par  = pd.read_parquet(f'{PAPER_DIR}/parent_map.parquet')
p = (prac.merge(par, on='placekey', how='left').query('parent_placekey.isna()')
         .merge(pv3[['placekey','patient_svi']], on='placekey', how='inner'))
p['site_svi'] = (pd.to_numeric(p.poi_cbg, errors='coerce').astype('Int64')
                 .astype(str).str.zfill(12).str[:11]).map(site_map)
xyp = nas.local_xy(p.latitude.to_numpy(), p.longitude.to_numpy())
p['_bld'] = [f'{a:.0f}_{b:.0f}' for a, b in np.round(xyp/75)*75]
g = p.groupby('_bld')
B3 = pd.DataFrame({
    'patient_svi': g.apply(lambda x: np.average(x.patient_svi, weights=x.annual_visits), include_groups=False),
    'site_svi':    g.apply(lambda x: np.average(x.site_svi.dropna(),
                        weights=x.annual_visits[x.site_svi.notna()]) if x.site_svi.notna().any() else np.nan,
                        include_groups=False),
    'visits':      g.annual_visits.sum(),
}).dropna().reset_index()

qs = lambda s: pd.qcut(s, 5, labels=False)
B3['q_site'], B3['q_pat'] = qs(B3.site_svi), qs(B3.patient_svi)
B3['gap'] = B3.patient_svi - B3.site_svi
ms, mp = B3.site_svi.median(), B3.patient_svi.median()
hh  = ((B3.site_svi < ms) & (B3.patient_svi >= mp)).sum()
hh5 = ((B3.q_site <= 1) & (B3.q_pat >= 3)).sum()

print(f'\n--- PHASE 1 RECHECK WITH TRI-STATE COVERAGE (N={len(B3)} buildings) ---')
print(f'{"":24}{"NY-only":>10}{"tri-state":>11}')
print(f'{"same quintile":<24}{"39.7%":>10}{(B3.q_site==B3.q_pat).mean():>10.1%}')
print(f'{"median gap":<24}{"+0.152":>10}{B3.gap.median():>+10.3f}')
print(f'{"gap > +0.20":<24}{"39.1%":>10}{(B3.gap>.20).mean():>10.1%}')
print(f'{"site vs patient corr":<24}{"0.462":>10}{nas.spearman(B3.site_svi, B3.patient_svi):>10.3f}')
print(f'{"hidden-high (median)":<24}{"155":>10}{hh:>10}')
print(f'{"hidden-high (quintile)":<24}{"80":>10}{hh5:>10}')
B3.to_parquet(f'{PAPER_DIR}/building_svi_quadrants_tristate.parquet', index=False)

# ---- dwell-bucket probe (plan step 2c feasibility) -------------------------
pairs = bf.find_weekly_files(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns')
hdr = bf.header_of(pairs[0][1])
print(f'\n--- DWELL PROBE | weekly file has {len(hdr)} columns ---')
print('dwell/bucket-related:', [c for c in hdr if 'dwell' in c.lower() or 'bucket' in c.lower()] or 'NONE')

In [ ]:
# ============ NATIONAL SVI REJOIN + DWELL EXTRACTION ============
# PRE-REGISTERED for the worker-contamination sensitivity (fixed before results):
#   long visit = dwell > 240 min (Brelsford 4-hour split)
#   sensitivity A: exclude sites with long-visit share > 0.30
#   sensitivity B: exclude sites in the top decile of long-visit share

# ---- PART A | national-percentile rejoin ----------------------------------
us = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False)
us['FIPS'] = us.FIPS.astype(str).str.zfill(11)
us['RPL_THEMES'] = us.RPL_THEMES.replace(-999, np.nan)
us_map = us.set_index('FIPS')['RPL_THEMES']
print(f'national SVI: {len(us):,} tracts   (CT present: '
      f'{(us.FIPS.str[:2]=="09").sum():,})')

flows = pd.read_parquet(f'{PAPER_DIR}/facility_cbg_flows.parquet')
flows = flows[flows.home_cbg.str.len() == 12].copy()
flows['tract'] = flows.home_cbg.str[:11]
flows['svi']   = flows.tract.map(us_map)
W = flows.visits.sum()
print(f'visit weight matched nationally: {flows.loc[flows.svi.notna(),"visits"].sum()/W:.1%}  (was 70.7% NY-only, 75.8% tri-state)')
top = (flows.assign(st=flows.tract.str[:2]).groupby('st')
       .agg(w=('visits','sum'), match=('svi', lambda s: s.notna().mean()))
       .sort_values('w', ascending=False).head(6))
print('top home states by visit weight:'); print((top.assign(w=top.w/W)).round(3).to_string())

m = flows[flows.svi.notna()]
g = m.groupby('placekey')
pvN = pd.DataFrame({'patient_svi': g.apply(lambda x: np.average(x.svi, weights=x.visits), include_groups=False),
                    'n_tracts': g.tract.nunique()}).reset_index()
pvN = pvN[pvN.n_tracts >= 10].reset_index(drop=True)
pvN.to_parquet(f'{PAPER_DIR}/patient_vulnerability_national.parquet', index=False)

prac = pd.read_parquet(f'{PAPER_DIR}/practitioners_raw.parquet')
par  = pd.read_parquet(f'{PAPER_DIR}/parent_map.parquet')
p = (prac.merge(par, on='placekey', how='left').query('parent_placekey.isna()')
         .merge(pvN[['placekey','patient_svi']], on='placekey', how='inner'))
p['site_svi'] = (pd.to_numeric(p.poi_cbg, errors='coerce').astype('Int64')
                 .astype(str).str.zfill(12).str[:11]).map(us_map)      # site also national
xyp = nas.local_xy(p.latitude.to_numpy(), p.longitude.to_numpy())
p['_bld'] = [f'{a:.0f}_{b:.0f}' for a, b in np.round(xyp/75)*75]
g = p.groupby('_bld')
BN = pd.DataFrame({
    'patient_svi': g.apply(lambda x: np.average(x.patient_svi, weights=x.annual_visits), include_groups=False),
    'site_svi':    g.apply(lambda x: np.average(x.site_svi.dropna(),
                        weights=x.annual_visits[x.site_svi.notna()]) if x.site_svi.notna().any() else np.nan,
                        include_groups=False),
    'visits': g.annual_visits.sum(),
    'lat': g.latitude.mean(), 'lon': g.longitude.mean(),
}).dropna().reset_index()

qs = lambda s: pd.qcut(s, 5, labels=False)
BN['q_site'], BN['q_pat'] = qs(BN.site_svi), qs(BN.patient_svi)
BN['gap'] = BN.patient_svi - BN.site_svi
ms, mp = BN.site_svi.median(), BN.patient_svi.median()
BN['hidden_high'] = (BN.site_svi < ms) & (BN.patient_svi >= mp)
print(f'\n--- HEADLINE NUMBERS, NATIONAL PERCENTILES (N={len(BN)}) ---')
print(f'median site SVI {BN.site_svi.median():.3f} | median patient SVI {BN.patient_svi.median():.3f}')
print(f'same quintile {(BN.q_site==BN.q_pat).mean():.1%} | corr {nas.spearman(BN.site_svi,BN.patient_svi):.3f}')
print(f'median gap {BN.gap.median():+.3f} | gap>+0.20 {(BN.gap>.20).mean():.1%} | gap<-0.20 {(BN.gap<-.20).mean():.1%}')
print(f'hidden-high: median-split {BN.hidden_high.sum()} | quintile {((BN.q_site<=1)&(BN.q_pat>=3)).sum()}')
BN.to_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet', index=False)

# ---- PART B | dwell-bucket extraction (48 files, ~15-25 min) ---------------
import re, json, warnings
from collections import defaultdict
def parse_j(js):
    if not isinstance(js, str): return {}
    s = re.sub(r'"{2,}', '"', js.strip()).strip("'").strip()
    try: d = json.loads(s)
    except Exception: return {}
    return {str(k): float(v) for k, v in d.items() if isinstance(v, (int, float))}

pairs = bf.find_weekly_files(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns')
cols  = ['placekey', 'naics_code', 'bucketed_dwell_times']
acc   = defaultdict(lambda: [0.0, 0.0])                     # placekey -> [long, total]
for i, (d_, pth) in enumerate(pairs, 1):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        f = bf._read_filtered(pth, cols, bf.NAICS_DEFAULT)
    for pk, js in zip(f['placekey'], f['bucketed_dwell_times']):
        b = parse_j(js)
        if b:
            tot = sum(b.values()); lng = b.get('>240', 0.0)
            acc[pk][0] += lng; acc[pk][1] += tot
    print(f'  [{i:>2}/{len(pairs)}] {d_}', end='\r')

dw = pd.DataFrame([{'placekey': k, 'long_share': v[0]/v[1]}
                   for k, v in acc.items() if v[1] > 0])
dw.to_parquet(f'{PAPER_DIR}/dwell_long_share.parquet', index=False)
print(f'\n\nlong-visit share computed for {len(dw):,} practices')
print(dw.long_share.describe().round(3).to_string())

p2 = p.merge(dw, on='placekey', how='left')
gs = p2.groupby('_bld').apply(lambda x: np.average(x.long_share.fillna(x.long_share.mean()),
                              weights=x.annual_visits), include_groups=False)
BN = BN.merge(gs.rename('long_share'), left_on='_bld', right_index=True, how='left')
print(f'\nbuilding long-visit share: median {BN.long_share.median():.3f}, p90 {BN.long_share.quantile(.9):.3f}')
print(f'corr(long_share, gap) = {nas.spearman(BN.long_share.fillna(0), BN.gap):.3f}')
for lbl, keep in [('A: long_share <= 0.30', BN.long_share.fillna(0) <= .30),
                  ('B: below top decile   ', BN.long_share.fillna(0) <= BN.long_share.quantile(.9))]:
    s = BN[keep]
    print(f'  sens {lbl}  N={len(s):>5}  median gap {s.gap.median():+.3f}  '
          f'gap>0.20 {(s.gap>.20).mean():.1%}  hidden-high {s.hidden_high.sum()}')
BN.to_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet', index=False)

In [ ]:
# ---- DIAGNOSE | what does bucketed_dwell_times actually look like? ----
import warnings
pairs = bf.find_weekly_files(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    f1 = bf._read_filtered(pairs[0][1], ['placekey','naics_code','bucketed_dwell_times'],
                           bf.NAICS_DEFAULT)
print('rows:', len(f1), '| columns returned:', list(f1.columns))
if 'bucketed_dwell_times' in f1.columns:
    v = f1['bucketed_dwell_times'].dropna()
    print('non-null:', len(v))
    for x in v.head(3):
        print('\nRAW:', repr(x)[:300])
    import re, json
    s = re.sub(r'"{2,}', '"', str(v.iloc[0]).strip()).strip("'").strip()
    print('\nCLEANED:', s[:300])
    try:
        d = json.loads(s); print('\nPARSED OK — keys:', list(d.keys()))
    except Exception as e:
        print('\nPARSE FAILED:', e)
else:
    print('column was dropped by _read_filtered — it restricts to bf.WANT')

In [ ]:
# ---- DIAGNOSE v2 | read the raw weekly file directly ----
import gzip
pairs = bf.find_weekly_files(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns')
path = pairs[0][1]
print('file:', path)

raw = pd.read_csv(path, usecols=['placekey','naics_code','bucketed_dwell_times'],
                  nrows=5000, low_memory=False)
print('rows read:', len(raw))
raw['naics4'] = raw.naics_code.astype(str).str[:4]
hc = raw[raw.naics4.isin(['6211','6213','6214'])]
print('healthcare rows in first 5000:', len(hc))

v = hc['bucketed_dwell_times'].dropna() if len(hc) else raw['bucketed_dwell_times'].dropna()
print('non-null dwell values:', len(v), 'of', len(hc) if len(hc) else len(raw))
if len(v):
    print('\nRAW value:', repr(v.iloc[0])[:400])
    import re, json
    s = re.sub(r'"{2,}', '"', str(v.iloc[0]).strip()).strip("'").strip()
    print('\nCLEANED:', s[:400])
    try:
        d = json.loads(s)
        print('\nPARSED OK — keys:', list(d.keys()))
    except Exception as e:
        print('\nPARSE FAILED:', e)
else:
    print('\ncolumn is empty even in the raw file for these rows')

In [ ]:
# ---- PART B (corrected) | dwell extraction via direct reads ----------------
# PRE-REGISTERED: long visit = '>240' bucket; sens A: long_share <= 0.30;
#                 sens B: below top decile of long_share.
import re, json, warnings
from collections import defaultdict

def parse_j(js):
    if not isinstance(js, str): return {}
    s = re.sub(r'"{2,}', '"', js.strip()).strip("'").strip()
    try: d = json.loads(s)
    except Exception: return {}
    return {str(k): float(v) for k, v in d.items() if isinstance(v, (int, float))}

pairs = bf.find_weekly_files(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns')
acc = defaultdict(lambda: [0.0, 0.0])                    # placekey -> [long, total]
for i, (d_, pth) in enumerate(pairs, 1):
    for ch in pd.read_csv(pth, usecols=['placekey','naics_code','bucketed_dwell_times'],
                          chunksize=200_000, low_memory=False):
        ch = ch[ch.naics_code.astype(str).str[:4].isin(['6211','6213','6214'])]
        for pk, js in zip(ch['placekey'], ch['bucketed_dwell_times']):
            b = parse_j(js)
            if b:
                acc[pk][0] += b.get('>240', 0.0); acc[pk][1] += sum(b.values())
    print(f'  [{i:>2}/{len(pairs)}] {d_}   practices so far: {len(acc):,}', end='\r')

dw = pd.DataFrame([{'placekey': k, 'long_share': v[0]/v[1]}
                   for k, v in acc.items() if v[1] > 0])
dw.to_parquet(f'{PAPER_DIR}/dwell_long_share.parquet', index=False)
print(f'\n\nlong-visit share computed for {len(dw):,} practices')
print(dw.long_share.describe().round(3).to_string())

# ---- sensitivity: does worker traffic drive the gap? (self-contained) ------
BN  = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet')
pvN = pd.read_parquet(f'{PAPER_DIR}/patient_vulnerability_national.parquet')
prac = pd.read_parquet(f'{PAPER_DIR}/practitioners_raw.parquet')
par  = pd.read_parquet(f'{PAPER_DIR}/parent_map.parquet')
p = (prac.merge(par, on='placekey', how='left').query('parent_placekey.isna()')
         .merge(pvN[['placekey']], on='placekey', how='inner')
         .merge(dw, on='placekey', how='left'))
xyp = nas.local_xy(p.latitude.to_numpy(), p.longitude.to_numpy())
p['_bld'] = [f'{a:.0f}_{b:.0f}' for a, b in np.round(xyp/75)*75]
ls = p.groupby('_bld').apply(lambda x: np.average(x.long_share.dropna(),
        weights=x.annual_visits[x.long_share.notna()]) if x.long_share.notna().any() else np.nan,
        include_groups=False)
BN = BN.drop(columns=['long_share'], errors='ignore').merge(
        ls.rename('long_share'), left_on='_bld', right_index=True, how='left')

print(f'\nbuilding long-visit share: median {BN.long_share.median():.3f}  '
      f'p90 {BN.long_share.quantile(.9):.3f}  missing {BN.long_share.isna().mean():.1%}')
print(f'corr(long_share, gap) = {nas.spearman(BN.long_share.fillna(0), BN.gap):.3f}')
print(f'\n{"":32}{"N":>6}{"med gap":>9}{"gap>0.20":>10}{"hidden-high":>12}')
print(f'{"full sample":<32}{len(BN):>6}{BN.gap.median():>+9.3f}{(BN.gap>.20).mean():>10.1%}{BN.hidden_high.sum():>12}')
for lbl, keep in [('A: long_share <= 0.30', BN.long_share.fillna(0) <= .30),
                  ('B: below top decile',    BN.long_share.fillna(0) <= BN.long_share.quantile(.9))]:
    s = BN[keep]
    print(f'{lbl:<32}{len(s):>6}{s.gap.median():>+9.3f}{(s.gap>.20).mean():>10.1%}{s.hidden_high.sum():>12}')
BN.to_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet', index=False)
print('\nsaved')

In [ ]:
# ---- REPRINT Part A headline block from saved artifacts ----
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet')
us = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False,
                 usecols=['FIPS','RPL_THEMES'])
us['FIPS'] = us.FIPS.astype(str).str.zfill(11)
us_map = us.assign(R=us.RPL_THEMES.replace(-999, np.nan)).set_index('FIPS')['R']

flows = pd.read_parquet(f'{PAPER_DIR}/facility_cbg_flows.parquet')
flows = flows[flows.home_cbg.str.len()==12].copy()
flows['tract'] = flows.home_cbg.str[:11]
flows['svi'] = flows.tract.map(us_map)
W = flows.visits.sum()
print(f'visit weight matched nationally: {flows.loc[flows.svi.notna(),"visits"].sum()/W:.1%}')
top = (flows.assign(st=flows.tract.str[:2]).groupby('st')
       .agg(w=('visits','sum'), match=('svi', lambda s: s.notna().mean()))
       .sort_values('w', ascending=False).head(6))
print('top home states (share of weight, match rate):')
print(top.assign(w=top.w/W).round(3).to_string())

print(f'\n--- HEADLINE, NATIONAL PERCENTILES (N={len(BN)}) ---')
print(f'median site SVI    {BN.site_svi.median():.3f}')
print(f'median patient SVI {BN.patient_svi.median():.3f}')
print(f'same quintile      {(BN.q_site==BN.q_pat).mean():.1%}')
print(f'site-patient corr  {nas.spearman(BN.site_svi, BN.patient_svi):.3f}')
print(f'median gap {BN.gap.median():+.3f} | >+0.20 {(BN.gap>.20).mean():.1%} | <-0.20 {(BN.gap<-.20).mean():.1%}')
print(f'hidden-high: median-split {BN.hidden_high.sum()} | quintile {((BN.q_site<=1)&(BN.q_pat>=3)).sum()}')

In [ ]:
# ==================== EXHIBIT CELL | national figures + Table 5 + Table S2 ==========
import numpy as np, pandas as pd, matplotlib.pyplot as plt, nas_core as nas
FIG = f'{OUT_DIR}/figures'; import os; os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})

BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet')
ms, mp = BN.site_svi.median(), BN.patient_svi.median()
BN['quad'] = np.select(
    [(BN.site_svi>=ms)&(BN.patient_svi>=mp), (BN.site_svi<ms)&(BN.patient_svi<mp),
     (BN.site_svi<ms)&(BN.patient_svi>=mp), (BN.site_svi>=ms)&(BN.patient_svi<mp)],
    ['concordant-high','concordant-low','hidden-high','hidden-low'], default='x')

# ---- Table 5: full national quadrant profile --------------------------------
T5 = BN.groupby('quad').agg(n=('quad','size'),
        visit_share=('visits', lambda v: v.sum()/BN.visits.sum()),
        mean_gap=('gap','mean'), med_site=('site_svi','median'),
        med_pat=('patient_svi','median')).round(3)
print('TABLE 5 | national quadrant profile'); print(T5.to_string())
T5.to_csv(f'{FIG}/T5_quadrants_national.csv')

# ---- Table S2: national quintile cross-tab ----------------------------------
S2 = pd.crosstab(BN.q_site, BN.q_pat, normalize='all').round(3)
print('\nTABLE S2 | quintile cross-tab (share of sites)'); print(S2.to_string())
S2.to_csv(f'{FIG}/TS2_quintile_crosstab_national.csv')

# ---- Fig 1: national gap histogram ------------------------------------------
fig, ax = plt.subplots(figsize=(5.2, 3.0))
ax.hist(BN.gap, bins=50, color='#7f9fbf', edgecolor='white', lw=.3)
ax.axvline(0, c='k', lw=.8); ax.axvline(.20, c='#d62728', ls='--', lw=.9)
ax.axvline(BN.gap.median(), c='#1f77b4', lw=1.2)
ax.text(BN.gap.median(), ax.get_ylim()[1]*.95, f' median {BN.gap.median():+.3f}',
        fontsize=8, color='#1f77b4', va='top')
ax.text(.21, ax.get_ylim()[1]*.75, f'{(BN.gap>.2).mean():.0%} of sites\n> +20 pts',
        fontsize=8, color='#d62728')
ax.set_xlabel('visitor SVI minus site SVI (national percentiles)')
ax.set_ylabel('sites')
fig.tight_layout(); fig.savefig(f'{FIG}/F1_gap_histogram_national.png', dpi=300); plt.show()

# ---- Fig 2: disagreement map + 2b scatter -----------------------------------
xyB = nas.local_xy(BN.lat.to_numpy(), BN.lon.to_numpy())
cmap = {'concordant-high':'#d62728','concordant-low':'0.7',
        'hidden-high':'#1f77b4','hidden-low':'#ff7f0e'}
fig, ax = plt.subplots(figsize=(4.6, 6.2))
for q, c in cmap.items():
    m = (BN.quad==q).to_numpy()
    ax.scatter(xyB[m,0]/1000, xyB[m,1]/1000, c=c, s=9,
               label=f'{q} (n={m.sum()})', linewidths=0)
ax.set_aspect('equal'); ax.legend(fontsize=7, loc='upper left', frameon=False)
ax.set_xlabel('km east'); ax.set_ylabel('km north')
fig.tight_layout(); fig.savefig(f'{FIG}/F2_disagreement_map_national.png', dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(4.4, 4.4))
for q, c in cmap.items():
    m = (BN.quad==q).to_numpy()
    ax.scatter(BN.site_svi[m], BN.patient_svi[m], c=c, s=8, linewidths=0,
               label=q if q=='hidden-high' else None)
ax.plot([0,1],[0,1],'k--',lw=.7)
ax.axvline(ms, c='0.6', lw=.6); ax.axhline(mp, c='0.6', lw=.6)
ax.set_xlabel('site location SVI (national pctile)')
ax.set_ylabel('service-population SVI (national pctile)')
ax.set_xlim(0,1); ax.set_ylim(0,1)
fig.tight_layout(); fig.savefig(f'{FIG}/F2b_scatter_national.png', dpi=300); plt.show()

print(f"\nManuscript fill-ins -> hidden-high visit share: "
      f"{BN.loc[BN.quad=='hidden-high','visits'].sum()/BN.visits.sum():.1%}; "
      f"mean gap: {BN.loc[BN.quad=='hidden-high','gap'].mean():+.3f}")

In [ ]:
# ========== TARGET REBUILD | national percentiles, remove the basis mismatch ==========
us = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False,
                 usecols=['FIPS','RPL_THEMES','E_TOTPOP'])
us['FIPS'] = us.FIPS.astype(str).str.zfill(11)
for c in ['RPL_THEMES','E_TOTPOP']:
    us[c] = us[c].replace(-999, np.nan)
man = us[us.FIPS.str[:5]=='36061'].dropna().rename(columns={'FIPS':'tract'})
print(f'Manhattan tracts, national file: {len(man)}')

prac = pd.read_parquet(f'{PAPER_DIR}/practitioners_raw.parquet')
prac['tract'] = (pd.to_numeric(prac.poi_cbg, errors='coerce').astype('Int64')
                 .astype(str).str.zfill(12).str[:11])
cen = (prac.groupby('tract').agg(lat=('latitude','mean'), lon=('longitude','mean'),
                                 n_poi=('placekey','size')).reset_index()
       .merge(man, on='tract').query('n_poi >= 3').reset_index(drop=True))
print(f'tracts with centroid + national SVI: {len(cen)}')

fac  = pd.read_parquet(f'{PAPER_DIR}/facilities_manhattan_2024_annual.parquet')
xy_f = nas.local_xy(fac.latitude.to_numpy(), fac.longitude.to_numpy())
xy_t = nas.local_xy(cen.lat.to_numpy(), cen.lon.to_numpy())
D = np.sqrt(((xy_f[:,None,:] - xy_t[None,:,:])**2).sum(-1))
demand = cen.RPL_THEMES.to_numpy() * cen.E_TOTPOP.to_numpy()
fac['need_admin_natl'] = (np.log1p(np.exp(-D/800) @ demand)
                          - np.log1p(5 + fac.n_facilities_1000m))
print(f'rank corr, state-basis vs national-basis target: '
      f'{nas.spearman(fac.need_admin_bld, fac.need_admin_natl):.4f}')
fac.to_parquet(f'{PAPER_DIR}/facilities_manhattan_2024_annual.parquet', index=False)

# ---- re-run the controlled comparison on the rebuilt target ----------------
d = fac.dropna(subset=['need_index','need_admin_natl']).reset_index(drop=True)
Xc, fnc = nas.feature_matrix(d, exclude=[
    'need_index','need_admin_bld','need_admin','need_admin_natl',
    'local_visits','annual_visits','local_share',
    'admin_demand','admin_demand_decay','site_svi','tract_pop',
    'n_facilities_500m','n_facilities_1000m','n_facilities_2000m',
    'dist_nearest_facility_m','dist_5th_nearest_m','poi_cbg','weeks_present'])
xyc = nas.local_xy(d.latitude.to_numpy(), d.longitude.to_numpy())
print('features:', fnc)
for name, col in [('contrast (need_index)     ', 'need_index'),
                  ('field NATIONAL target     ', 'need_admin_natl')]:
    yv = d[col].to_numpy(float)
    g = np.array([nas.staircase(Xc, yv, xyc, k=10, seed=s).rho['Ridge-Spatial'] -
                  nas.staircase(Xc, yv, xyc, k=10, seed=s).rho['Ridge'] for s in range(10)])
    t = nas.staircase(Xc, yv, xyc, k=10)
    print(f'{name} own={t.rho["Ridge"]:.4f}  nbr={t.rho["Ridge-Spatial"]:.4f}  '
          f'gain={g.mean():+.4f} (sd {g.std():.4f})')

In [ ]:
# ============ STEP 1 (corrected) | canonical site crosswalk, then B1 + B2 ============
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet')
MS, MP = BN.site_svi.median(), BN.patient_svi.median()
cohort = set(BN._bld)

# --- rebuild the grid with the SAME input the national run used, so IDs reproduce ---
pvN  = pd.read_parquet(f'{PAPER_DIR}/patient_vulnerability_national.parquet')
prac = pd.read_parquet(f'{PAPER_DIR}/practitioners_raw.parquet')
par  = pd.read_parquet(f'{PAPER_DIR}/parent_map.parquet')
p = (prac.merge(par, on='placekey', how='left').query('parent_placekey.isna()')
         .merge(pvN[['placekey']], on='placekey', how='inner')).copy()
xyp = nas.local_xy(p.latitude.to_numpy(), p.longitude.to_numpy())
p['_bld'] = [f'{a:.0f}_{b:.0f}' for a, b in np.round(xyp/75)*75]
hit = p._bld.isin(cohort).sum()
print(f'crosswalk check: {hit:,} of {len(p):,} practitioners map into the 1,249 cohort sites; '
      f'{p[p._bld.isin(cohort)]._bld.nunique()} distinct cohort sites recovered (expect 1249)')
# record the projection origin so the grid is reproducible without this trick
lat0, lon0 = np.nanmean(p.latitude), np.nanmean(p.longitude)
xw = p[['placekey','_bld','annual_visits','latitude','longitude']].copy()
xw.attrs = {}
xw.to_parquet(f'{PAPER_DIR}/site_crosswalk.parquet', index=False)
pd.Series({'grid_m':75,'origin_lat':lat0,'origin_lon':lon0,'projection':'equirectangular about origin',
           'snapping':'round(xy/75)*75'}).to_json(f'{PAPER_DIR}/site_grid_spec.json')
print(f'saved site_crosswalk.parquet and site_grid_spec.json (origin {lat0:.6f}, {lon0:.6f})')

pk2bld = xw.set_index('placekey')['_bld']
pk_w   = xw.set_index('placekey')['annual_visits']

us = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False, usecols=['FIPS','RPL_THEMES'])
us['FIPS'] = us.FIPS.astype(str).str.zfill(11)
in_file = set(us.FIPS)
svi_map = us.assign(R=us.RPL_THEMES.replace(-999, np.nan)).set_index('FIPS')['R']

# ---- B2 | reconciliation on cohort rows -------------------------------------------
fl = pd.read_parquet(f'{PAPER_DIR}/facility_cbg_flows.parquet')
fl['_bld'] = fl.placekey.map(pk2bld)
fl = fl[fl._bld.isin(cohort)].copy()
W = fl.visits.sum()
fl['cbg'] = fl.home_cbg.astype(str)
fl['valid12'] = fl.cbg.str.len().eq(12) & fl.cbg.str.isdigit()
fl['tract'] = np.where(fl.valid12, fl.cbg.str[:11], None)
fl['st'] = np.where(fl.valid12, fl.cbg.str[:2], None)
fl['in_file'] = fl.tract.isin(in_file)
fl['svi'] = fl.tract.map(svi_map)
fl['cat'] = np.select(
    [~fl.valid12, fl.st.isin(['60','66','69','72','78']), ~fl.in_file, fl.svi.isna()],
    ['invalid identifier','US territory','tract absent from 2022 file','SVI missing code (-999)'],
    default='matched')
print(f'\nB2 | ORIGIN-WEIGHT RECONCILIATION  (cohort rows: {len(fl):,}; device-weeks: {W:,.0f})')
print((fl.groupby('cat').visits.sum().sort_values(ascending=False) / W).map('{:.1%}'.format).to_string())
ab = fl[fl.cat.eq('tract absent from 2022 file')]
print('  absent-from-file weight by state:'); print((ab.groupby('st').visits.sum().sort_values(ascending=False).head(6)/W).map('{:.2%}'.format).to_string())
site_tot = fl.groupby('_bld').visits.sum(); site_mat = fl[fl.cat.eq('matched')].groupby('_bld').visits.sum()
c_i = (site_mat / site_tot).reindex(BN._bld).fillna(0)
print(f'  site coverage c_i: median {c_i.median():.3f}, p10 {c_i.quantile(.1):.3f}, p90 {c_i.quantile(.9):.3f}; '
      f'sites with c_i>=0.80: {(c_i>=.8).mean():.1%}')
pd.DataFrame({'_bld': c_i.index, 'coverage_c': c_i.values}).to_parquet(f'{PAPER_DIR}/site_coverage.parquet', index=False)

# ---- B1 | fixed cohort, national scale, frozen thresholds --------------------------
m = fl[fl.cat.eq('matched')].copy()
def run(label, states):
    sub = m if states is None else m[m.st.isin(states)]
    g = sub.groupby('placekey')
    pv = pd.DataFrame({'V': g.apply(lambda x: np.average(x.svi, weights=x.visits), include_groups=False),
                       'n_tr': g.tract.nunique()})
    pv['_bld'] = pv.index.map(pk2bld); pv['w'] = pv.index.map(pk_w)
    gb = pv.groupby('_bld')
    d = BN.set_index('_bld').assign(V=gb.apply(lambda x: np.average(x.V, weights=x.w), include_groups=False),
                                    n_tr=gb.n_tr.sum())
    d['D'] = d.V - d.site_svi
    d['cls'] = np.select([(d.site_svi>=MS)&(d.V>=MP),(d.site_svi<MS)&(d.V<MP),
                          (d.site_svi<MS)&(d.V>=MP),(d.site_svi>=MS)&(d.V<MP)],
                         ['CH','CL','HH','HL'], default='NA')
    ok = d.V.notna()
    return dict(run=label, sites_with_V=int(ok.sum()), sites_lt10tr=int((d.n_tr<10).sum()),
                rho=nas.spearman(d.site_svi[ok], d.V[ok]), medD=d.D.median(), gt20=(d.D>.20).mean(),
                lt20=(d.D<-.20).mean(), Dpos=(d.D>0).mean(), HH=int((d.cls=='HH').sum()),
                HL=int((d.cls=='HL').sum()), HH_wt=d.loc[d.cls=='HH','visits'].sum()/d.visits.sum()), d.cls
rows, classes = [], {}
for label, states in [('NY only',['36']),('NY+NJ',['36','34']),('all matched',None)]:
    r, cls = run(label, states); rows.append(r); classes[label] = cls
res = pd.DataFrame(rows).set_index('run')
print('\nB1 | FIXED COHORT (N=1249), NATIONAL SCALE, FROZEN THRESHOLDS'); print(res.round(3).to_string())
res.to_csv(f'{OUT_DIR}/B1_fixed_cohort_coverage.csv')
print('\n  class transitions NY-only -> all matched:'); print(pd.crosstab(classes['NY only'], classes['all matched']).to_string())

In [ ]:
# ============ STEP 2a | regional-composition null (B3a) ============
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
xw = pd.read_parquet(f'{PAPER_DIR}/site_crosswalk.parquet')
pk2bld, pk_w = xw.set_index('placekey')['_bld'], xw.set_index('placekey')['annual_visits']
us = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False, usecols=['FIPS','RPL_THEMES'])
us['FIPS'] = us.FIPS.astype(str).str.zfill(11)
svi_map = us.assign(R=us.RPL_THEMES.replace(-999, np.nan)).set_index('FIPS')['R']

fl = pd.read_parquet(f'{PAPER_DIR}/facility_cbg_flows.parquet')
fl['_bld'] = fl.placekey.map(pk2bld); fl = fl[fl._bld.isin(BN.index)].copy()
fl['tract'] = fl.home_cbg.astype(str).str[:11]; fl['svi'] = fl.tract.map(svi_map)
m = fl[fl.svi.notna()]
V_reg = np.average(m.svi, weights=m.visits)
L, V, D = BN.site_svi, BN.patient_svi, BN.gap
D_null = V_reg - L
print(f'regional weighted-mean origin SVI: {V_reg:.3f}')
print(f'{"":30}{"observed":>10}{"null (a)":>10}')
for lab, a, b in [('median Delta', D.median(), D_null.median()),
                  ('share Delta > +0.20', (D>.2).mean(), (D_null>.2).mean()),
                  ('share Delta < -0.20', (D<-.2).mean(), (D_null<-.2).mean()),
                  ('share Delta > 0', (D>0).mean(), (D_null>0).mean()),
                  ('SD of Delta', D.std(), D_null.std())]:
    print(f'{lab:<30}{a:>10.3f}{b:>10.3f}')
excess = V - V_reg                                   # what site-specific flows add
print(f'\nsite-specific excess V_i - V_reg: median |excess| {excess.abs().median():.3f}; '
      f'share |excess| > 0.10: {(excess.abs()>.1).mean():.1%}')
print(f'corr(excess, L): {nas.spearman(excess, L):.3f}   (positive = flows track location; '
      f'pure averaging would give 0)')
# hidden-high under the null: which observed HH sites would be HH anyway?
MS, MP = L.median(), V.median()
hh_obs = (L<MS)&(V>=MP); hh_null = (L<MS)&(V_reg>=MP)
print(f'hidden-high observed {hh_obs.sum()}; under null (a) {hh_null.sum()} '
      f'({"all below-median-L sites" if V_reg>=MP else "none"}, since V_reg {"≥" if V_reg>=MP else "<"} median V)')
# per-site: how much of the observed gap is explained by the null
print(f'share of observed Delta explained by null, median over sites with D>0: '
      f'{(D_null[D>0]/D[D>0]).clip(0,2).median():.2f}')
pd.DataFrame({'_bld':BN.index,'D_obs':D,'D_null_regional':D_null,'excess_V':excess}).to_parquet(f'{PAPER_DIR}/null_a.parquet', index=False)

In [ ]:
rel0 = pd.read_csv(rl[0], sep='|', dtype=str, encoding='utf-8-sig', nrows=3)
print(list(rel0.columns)); print(rel0.T)

In [ ]:
# ============ STEP 2b (corrected) | block-group vintage crosswalk + distance-constrained null ============
import glob, os
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
MS, MP = BN.site_svi.median(), BN.patient_svi.median()
xw = pd.read_parquet(f'{PAPER_DIR}/site_crosswalk.parquet')
pk2bld, pk_w = xw.set_index('placekey')['_bld'], xw.set_index('placekey')['annual_visits']
us = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False, usecols=['FIPS','RPL_THEMES'])
us['FIPS'] = us.FIPS.astype(str).str.zfill(11)
svi_map = us.assign(R=us.RPL_THEMES.replace(-999, np.nan)).set_index('FIPS')['R']

gz = [f for f in glob.glob(f'{PAPER_DIR}/*') if 'gaz' in os.path.basename(f).lower() and 'tract' in os.path.basename(f).lower()]
rl = []
for f in glob.glob(f'{PAPER_DIR}/*.txt'):
    try:
        hdr = pd.read_csv(f, sep='|', nrows=0, encoding='utf-8-sig').columns
        if 'GEOID_BLKGRP_20' in hdr: rl.append(f)
    except Exception: pass
print('gazetteer:', [os.path.basename(f) for f in gz]); print('relationship:', [os.path.basename(f) for f in rl])
gaz = pd.read_csv(gz[0], sep='\t', dtype=str); gaz.columns = [c.strip() for c in gaz.columns]
gaz['GEOID'] = gaz.GEOID.str.zfill(11)
cent = gaz.set_index('GEOID')[['INTPTLAT','INTPTLONG']].astype(float)
print(f'tract centroids: {len(cent):,}')
rel = pd.concat([pd.read_csv(f, sep='|', dtype=str, encoding='utf-8-sig') for f in rl])
rel = rel[['GEOID_BLKGRP_20','GEOID_BLKGRP_10','AREALAND_PART','AREALAND_BLKGRP_10']].copy()
for c in ['AREALAND_PART','AREALAND_BLKGRP_10']: rel[c] = pd.to_numeric(rel[c], errors='coerce')
rel['share'] = (rel.AREALAND_PART / rel.AREALAND_BLKGRP_10).clip(0, 1).fillna(0)
rel = rel[rel.share > 0]
print(f'relationship rows: {len(rel):,}; 2010 block groups covered: {rel.GEOID_BLKGRP_10.nunique():,}; '
      f'states: {sorted(rel.GEOID_BLKGRP_20.str[:2].unique())}')

fl = pd.read_parquet(f'{PAPER_DIR}/facility_cbg_flows.parquet')
fl['_bld'] = fl.placekey.map(pk2bld); fl = fl[fl._bld.isin(BN.index)].copy()
fl['home_cbg'] = fl.home_cbg.astype(str); fl = fl[fl.home_cbg.str.len().eq(12)]
fl['tract'] = fl.home_cbg.str[:11]
W = fl.visits.sum()
in_file = fl.tract.isin(svi_map.index)
mat = fl[in_file].assign(svi=lambda d: d.tract.map(svi_map)).dropna(subset=['svi'])
absent = fl[~in_file]
print(f'\nbefore crosswalk: matched {mat.visits.sum()/W:.1%}; absent-from-file {absent.visits.sum()/W:.1%}')

xr = absent.merge(rel, left_on='home_cbg', right_on='GEOID_BLKGRP_10', how='inner')
xr['visits'] = xr.visits * xr.share
xr['tract'] = xr.GEOID_BLKGRP_20.str[:11]
xr['svi'] = xr.tract.map(svi_map)
rec = xr.dropna(subset=['svi'])[['placekey','_bld','tract','visits','svi']]
print(f'recovered via block-group crosswalk: {rec.visits.sum()/W:.1%} of cohort weight; '
      f'still absent (other states / no match): {(absent.visits.sum()-rec.visits.sum())/W:.1%}')
mat2 = pd.concat([mat[['placekey','_bld','tract','visits','svi']], rec])
print(f'after crosswalk: matched {mat2.visits.sum()/W:.1%}')

def V_by_site(m):
    g = m.groupby('placekey')
    pv = pd.DataFrame({'V': g.apply(lambda x: np.average(x.svi, weights=x.visits), include_groups=False)})
    pv['_bld'] = pv.index.map(pk2bld); pv['w'] = pv.index.map(pk_w)
    return pv.groupby('_bld').apply(lambda x: np.average(x.V, weights=x.w), include_groups=False).reindex(BN.index)
def summarise(V, label):
    D = V - BN.site_svi; hh = ((BN.site_svi<MS)&(V>=MP))
    return dict(run=label, rho=nas.spearman(BN.site_svi, V), medD=D.median(), gt20=(D>.2).mean(),
                lt20=(D<-.2).mean(), Dpos=(D>0).mean(), HH=int(hh.sum()), sdD=D.std())
V_before, V_after = V_by_site(mat), V_by_site(mat2)
rows = [summarise(V_before,'matched, pre-crosswalk'), summarise(V_after,'matched + vintage crosswalk')]

# ---- distance-constrained null (b) ----
m = mat2[mat2.tract.isin(cent.index)].copy()
T = m.groupby('tract').visits.sum()
S = m.groupby('_bld').visits.sum().reindex(BN.index).fillna(0)
tr = T.index.to_numpy(); tv = svi_map.reindex(tr).to_numpy()
tlat, tlon = cent.loc[tr,'INTPTLAT'].to_numpy(), cent.loc[tr,'INTPTLONG'].to_numpy()
slat, slon = BN.lat.to_numpy(), BN.lon.to_numpy()
def hav(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2); dl = np.radians(lon2 - lon1)
    a = np.sin((p2-p1)/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*6371.0*np.arcsin(np.sqrt(a))
D_km = np.empty((len(slat), len(tr)), dtype=np.float32)
for i in range(len(slat)): D_km[i] = hav(slat[i], slon[i], tlat, tlon)
mm = m.merge(pd.DataFrame({'tract':tr,'ti':np.arange(len(tr))}), on='tract')
mm['si'] = mm._bld.map({b:i for i,b in enumerate(BN.index)})
d_obs = D_km[mm.si.to_numpy(), mm.ti.to_numpy()]
obs_mean_logd = np.average(np.log(d_obs + 0.5), weights=mm.visits)
print(f'\nobserved: tracts {len(tr):,}; weighted mean log(d+0.5) {obs_mean_logd:.3f}; '
      f'weighted median distance {np.median(np.repeat(d_obs, np.maximum(1,(mm.visits/mm.visits.min()).astype(int).clip(upper=50)))):.1f} km')
def ipf(beta, iters=30):
    K = (D_km + 0.5) ** (-beta) * T.to_numpy()[None, :]
    for _ in range(iters):
        K *= (S.to_numpy() / np.maximum(K.sum(1), 1e-12))[:, None]
        K *= (T.to_numpy() / np.maximum(K.sum(0), 1e-12))[None, :]
    return K
mean_logd = lambda K: np.average(np.log(D_km + 0.5), weights=K)
fit = [(b, mean_logd(ipf(b, 10))) for b in np.linspace(0.5, 3.0, 11)]
beta = min(fit, key=lambda x: abs(x[1] - obs_mean_logd))[0]
K = ipf(beta, 40)
print(f'kernel exponent beta = {beta:.2f}; null mean log(d+0.5) {mean_logd(K):.3f}')
V_null = pd.Series((K * np.nan_to_num(tv)[None, :]).sum(1) / np.maximum(K.sum(1), 1e-12), index=BN.index)
rows.append(summarise(V_null, 'null (b): distance-constrained, marginals preserved'))
res = pd.DataFrame(rows).set_index('run'); print('\n', res.round(3).to_string())

ex = V_after - V_null
hh_obs = (BN.site_svi<MS)&(V_after>=MP); hh_null = (BN.site_svi<MS)&(V_null>=MP)
print(f'\nexcess V_obs - V_null(b): median {ex.median():+.3f}; median |excess| {ex.abs().median():.3f}; '
      f'share |excess|>0.10: {(ex.abs()>.1).mean():.1%}; share |excess|>0.05: {(ex.abs()>.05).mean():.1%}')
print(f'corr(V_obs, V_null) {nas.spearman(V_after, V_null):.3f}   corr(excess, L) {nas.spearman(ex, BN.site_svi):.3f}')
print(f'hidden-high: observed {hh_obs.sum()}, null(b) {hh_null.sum()}, both {(hh_obs&hh_null).sum()}, '
      f'observed-only {(hh_obs&~hh_null).sum()}, null-only {(~hh_obs&hh_null).sum()}')
pd.DataFrame({'_bld':BN.index,'L':BN.site_svi,'V_obs':V_after,'V_null_b':V_null,'excess':ex,
              'D_obs':V_after-BN.site_svi,'D_null_b':V_null-BN.site_svi}).to_parquet(f'{PAPER_DIR}/null_b.parquet', index=False)
res.to_csv(f'{OUT_DIR}/B3_null_models.csv'); print('saved')

In [ ]:
# ============ STEP 3 | split-half temporal stability of V, Delta, excess, and hidden-high (B5) ============
import re, json, glob, os
from collections import defaultdict
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
MS, MP = BN.site_svi.median(), BN.patient_svi.median()
xw = pd.read_parquet(f'{PAPER_DIR}/site_crosswalk.parquet')
pk2bld, pk_w = xw.set_index('placekey')['_bld'], xw.set_index('placekey')['annual_visits']
cohort_pk = set(xw.placekey)
nb = pd.read_parquet(f'{PAPER_DIR}/null_b.parquet').set_index('_bld')
us = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False, usecols=['FIPS','RPL_THEMES'])
us['FIPS'] = us.FIPS.astype(str).str.zfill(11)
svi_map = us.assign(R=us.RPL_THEMES.replace(-999, np.nan)).set_index('FIPS')['R']
rl = [f for f in glob.glob(f'{PAPER_DIR}/*.txt') if 'GEOID_BLKGRP_20' in pd.read_csv(f, sep='|', nrows=0, encoding='utf-8-sig').columns]
rel = pd.concat([pd.read_csv(f, sep='|', dtype=str, encoding='utf-8-sig') for f in rl])
rel['share'] = (pd.to_numeric(rel.AREALAND_PART)/pd.to_numeric(rel.AREALAND_BLKGRP_10)).clip(0,1).fillna(0)
rel = rel[rel.share>0][['GEOID_BLKGRP_10','GEOID_BLKGRP_20','share']]

def parse_j(js):
    if not isinstance(js, str): return {}
    s = re.sub(r'"{2,}', '"', js.strip()).strip("'").strip()
    try: return {str(k): float(v) for k, v in json.loads(s).items() if isinstance(v,(int,float))}
    except Exception: return {}

pairs = bf.find_weekly_files(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns')
acc = defaultdict(float)                                  # (placekey, period, cbg) -> weight
for i, (d_, pth) in enumerate(pairs, 1):
    half = 'H1' if i <= 24 else 'H2'; q = f'Q{(i-1)//12 + 1}'
    for ch in pd.read_csv(pth, usecols=['placekey','visitor_home_cbgs'], chunksize=200_000, low_memory=False):
        ch = ch[ch.placekey.isin(cohort_pk)]
        for pk, js in zip(ch.placekey, ch.visitor_home_cbgs):
            for cbg, w in parse_j(js).items():
                acc[(pk, half, cbg)] += w; acc[(pk, q, cbg)] += w
    print(f'  [{i:>2}/48] {d_}', end='\r')
fl = pd.DataFrame([(k[0],k[1],k[2],v) for k,v in acc.items()], columns=['placekey','period','home_cbg','w'])
fl['home_cbg'] = fl.home_cbg.astype(str).str.zfill(12); fl['tract'] = fl.home_cbg.str[:11]
fl['svi'] = fl.tract.map(svi_map)
# vintage crosswalk for absent NY block groups
ab = fl[fl.svi.isna() & ~fl.tract.isin(svi_map.index)]
xr = ab.merge(rel, left_on='home_cbg', right_on='GEOID_BLKGRP_10', how='inner')
xr['w'] = xr.w * xr.share; xr['tract'] = xr.GEOID_BLKGRP_20.str[:11]; xr['svi'] = xr.tract.map(svi_map)
fl = pd.concat([fl.dropna(subset=['svi']), xr.dropna(subset=['svi'])[fl.columns]])
fl['_bld'] = fl.placekey.map(pk2bld)
fl.to_parquet(f'{PAPER_DIR}/flows_by_period.parquet', index=False)
print(f'\n\nperiod flows saved: {len(fl):,} rows')

def V_site(sub):
    g = sub.groupby('placekey')
    pv = pd.DataFrame({'V': g.apply(lambda x: np.average(x.svi, weights=x.w), include_groups=False),
                       'n_tr': g.tract.nunique()})
    pv['_bld'] = pv.index.map(pk2bld); pv['wt'] = pv.index.map(pk_w)
    gb = pv.groupby('_bld')
    return (gb.apply(lambda x: np.average(x.V, weights=x.wt), include_groups=False).reindex(BN.index),
            gb.n_tr.sum().reindex(BN.index).fillna(0))
P = {}
for per in ['H1','H2','Q1','Q2','Q3','Q4']:
    V, ntr = V_site(fl[fl.period==per]); P[per] = pd.DataFrame({'V':V,'n_tr':ntr,'D':V-BN.site_svi,'ex':V-nb.V_null_b,
                                                                  'HH':(BN.site_svi<MS)&(V>=MP)})
ok = (P['H1'].n_tr>=10)&(P['H2'].n_tr>=10)
print(f'sites with >=10 matched tracts in BOTH halves: {ok.sum()} of {len(BN)}')
print(f'{"":36}{"split-half rho":>16}')
for lab, col in [('visitor-origin SVI (V)','V'),('paired difference (Delta)','D'),('excess over distance null','ex')]:
    print(f'{lab:<36}{nas.spearman(P["H1"].loc[ok,col], P["H2"].loc[ok,col]):>16.3f}')
hh1, hh2 = P['H1'].HH[ok], P['H2'].HH[ok]
print(f'\nhidden-high: H1 {hh1.sum()}, H2 {hh2.sum()}, both {(hh1&hh2).sum()}, either {(hh1|hh2).sum()}, '
      f'full-period {BN.hidden_high[ok].sum()}')
full_hh = BN.hidden_high[ok]
print(f'of full-period hidden-high: in both halves {(full_hh&hh1&hh2).sum()}, in one {(full_hh&(hh1^hh2)).sum()}, in neither {(full_hh&~hh1&~hh2).sum()}')
qs = [P[q].HH for q in ['Q1','Q2','Q3','Q4']]; nq = sum(q.astype(int) for q in qs)
stab = pd.DataFrame({'_bld':BN.index,'HH_full':BN.hidden_high,'HH_H1':P['H1'].HH,'HH_H2':P['H2'].HH,'HH_quarters':nq,
                     'ex_H1':P['H1'].ex,'ex_H2':P['H2'].ex,'ok_both_halves':ok})
stab['flag'] = np.select([stab.HH_full & stab.HH_H1 & stab.HH_H2 & (stab.HH_quarters>=3),
                          stab.HH_full & (stab.HH_H1 | stab.HH_H2)], ['robust','uncertain'], default='nonqualifying')
print('\nhidden-high stability flags:'); print(stab[stab.HH_full].flag.value_counts().to_string())
stab.to_parquet(f'{PAPER_DIR}/site_stability.parquet', index=False); print('saved site_stability.parquet')

In [ ]:
# ============ STEP 4 | per-site origin bootstrap + aggregate block bootstrap + H1/H2 diagnostic (B4) ============
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
MS, MP = BN.site_svi.median(), BN.patient_svi.median()
nb = pd.read_parquet(f'{PAPER_DIR}/null_b.parquet').set_index('_bld')
fp = pd.read_parquet(f'{PAPER_DIR}/flows_by_period.parquet')
rng = np.random.default_rng(0)

# ---- H1/H2 diagnostic: is the second half thinner? ----
for per in ['H1','H2']:
    s = fp[fp.period==per]
    print(f'{per}: total weight {s.w.sum():,.0f}; rows {len(s):,}; distinct tracts {s.tract.nunique():,}; '
          f'weighted mean SVI {np.average(s.svi, weights=s.w):.3f}')

# ---- per-site origin bootstrap (full period = H1 + H2 rows), 300 draws ----
full = fp[fp.period.isin(['H1','H2'])].groupby(['_bld','tract'], as_index=False).agg(w=('w','sum'), svi=('svi','first'))
B = 300
rec = {}
for b, g in full.groupby('_bld'):
    w, v = g.w.to_numpy(), g.svi.to_numpy(); n = len(w)
    if n < 2: continue
    p = w / w.sum()
    idx = rng.choice(n, size=(B, n), replace=True, p=p)          # resample origins by weight
    Vb = v[idx].mean(axis=1)                                     # each draw: mean SVI of resampled origins
    rec[b] = (Vb.mean(), Vb.std(), (Vb >= MP).mean(), np.percentile(Vb, [2.5, 97.5]))
site = pd.DataFrame({k: dict(V_boot=v[0], se_V=v[1], p_above_MP=v[2], lo=v[3][0], hi=v[3][1]) for k, v in rec.items()}).T
site = site.reindex(BN.index)
site['L'] = BN.site_svi; site['V_full'] = BN.patient_svi
site['p_HH'] = np.where(site.L < MS, site.p_above_MP, 0.0)
site['ex'] = nb.excess; site['n_origins'] = full.groupby('_bld').size().reindex(BN.index)
print(f'\nper-site SE of V: median {site.se_V.median():.3f}, p10 {site.se_V.quantile(.1):.3f}, p90 {site.se_V.quantile(.9):.3f}')
print(f'observed hidden-high {int(BN.hidden_high.sum())}: P(HH) >= 0.90: {(site.p_HH>=.9).sum()}, '
      f'0.50-0.90: {((site.p_HH>=.5)&(site.p_HH<.9)).sum()}, < 0.50 among observed HH: {((site.p_HH<.5)&BN.hidden_high).sum()}')
print(f'expected number of hidden-high sites (sum of P): {site.p_HH.sum():.0f}')
site['flag'] = np.select([site.p_HH>=.9, site.p_HH>=.5], ['robust','uncertain'], default='nonqualifying')
print(site.flag.value_counts().to_string())

# does the excess survive per-site noise?  share of sites whose |excess| exceeds 2 SE
print(f'\nshare of sites with |excess over null| > 2*SE(V): {(site.ex.abs() > 2*site.se_V).mean():.1%}')
site.to_parquet(f'{PAPER_DIR}/site_uncertainty.parquet'); print('saved site_uncertainty.parquet')

# ---- aggregate: spatial block bootstrap over sites, 500 draws, 500 m and 1000 m blocks ----
xy = nas.local_xy(BN.lat.to_numpy(), BN.lon.to_numpy())
L, V, D = BN.site_svi.to_numpy(), BN.patient_svi.to_numpy(), BN.gap.to_numpy()
def stats(ix):
    return [nas.spearman(L[ix], V[ix]), np.median(D[ix]), (D[ix]>.2).mean(), (D[ix]>0).mean(),
            ((L[ix]<MS)&(V[ix]>=MP)).sum(), nas.spearman(V[ix], nb.V_null_b.to_numpy()[ix])]
names = ['rho(L,V)','median Delta','share >0.20','share Delta>0','hidden-high n','rho(V, V_null)']
print(f'\n{"":18}{"point":>8}', end='')
for blk in [500, 1000]:
    cell = np.floor(xy/blk).astype(int); key = cell[:,0]*100000 + cell[:,1]
    ukeys = np.unique(key); groups = [np.where(key==k)[0] for k in ukeys]
    draws = np.array([stats(np.concatenate([groups[j] for j in rng.integers(0, len(groups), len(groups))])) for _ in range(500)])
    print(f'{"CI95 blk"+str(blk)+"m":>26}', end='')
    if blk == 1000: print()
    globals()[f'ci{blk}'] = np.percentile(draws, [2.5, 97.5], axis=0)
pt = stats(np.arange(len(L)))
for i, n in enumerate(names):
    print(f'{n:<18}{pt[i]:>8.3f}{"":>6}[{ci500[0,i]:.3f}, {ci500[1,i]:.3f}]{"":>6}[{ci1000[0,i]:.3f}, {ci1000[1,i]:.3f}]')

In [ ]:
fp = pd.read_parquet(f'{PAPER_DIR}/flows_by_period.parquet')
q = fp[fp.period.str.startswith('Q')].groupby('period').agg(weight=('w','sum'), rows=('w','size'),
        tracts=('tract','nunique'), mean_svi=('svi', lambda s: np.average(s, weights=fp.loc[s.index,'w'])))
print(q.round(3).to_string())

In [ ]:
# ============ STEP 5 | Q1 odd/even split-half reliability + break-week locator ============
import re, json
from collections import defaultdict
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
MS, MP = BN.site_svi.median(), BN.patient_svi.median()
xw = pd.read_parquet(f'{PAPER_DIR}/site_crosswalk.parquet')
pk2bld, pk_w = xw.set_index('placekey')['_bld'], xw.set_index('placekey')['annual_visits']
cohort_pk = set(xw.placekey)
nb = pd.read_parquet(f'{PAPER_DIR}/null_b.parquet').set_index('_bld')
us = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False, usecols=['FIPS','RPL_THEMES'])
us['FIPS'] = us.FIPS.astype(str).str.zfill(11)
svi_map = us.assign(R=us.RPL_THEMES.replace(-999, np.nan)).set_index('FIPS')['R']
def parse_j(js):
    if not isinstance(js, str): return {}
    s = re.sub(r'"{2,}', '"', js.strip()).strip("'").strip()
    try: return {str(k): float(v) for k, v in json.loads(s).items() if isinstance(v,(int,float))}
    except Exception: return {}

pairs = bf.find_weekly_files(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns')
acc = defaultdict(float); wk = []
for i, (d_, pth) in enumerate(pairs, 1):
    if i > 16: break                                       # weeks 1-16 cover Q1 and the break
    n_rows = 0; wsum = 0.0
    for ch in pd.read_csv(pth, usecols=['placekey','visitor_home_cbgs'], chunksize=200_000, low_memory=False):
        ch = ch[ch.placekey.isin(cohort_pk)]
        for pk, js in zip(ch.placekey, ch.visitor_home_cbgs):
            d = parse_j(js); n_rows += len(d); wsum += sum(d.values())
            if i <= 13:
                lab = 'odd' if i % 2 else 'even'
                for cbg, w in d.items(): acc[(pk, lab, cbg)] += w
    wk.append((i, str(d_), n_rows, wsum)); print(f'  week {i:>2} {d_}: origin rows {n_rows:>7,}  weight {wsum:>10,.0f}')
fl = pd.DataFrame([(k[0],k[1],k[2],v) for k,v in acc.items()], columns=['placekey','period','home_cbg','w'])
fl['tract'] = fl.home_cbg.astype(str).str.zfill(12).str[:11]; fl['svi'] = fl.tract.map(svi_map)
fl = fl.dropna(subset=['svi']); fl['_bld'] = fl.placekey.map(pk2bld)
def V_site(sub):
    g = sub.groupby('placekey')
    pv = pd.DataFrame({'V': g.apply(lambda x: np.average(x.svi, weights=x.w), include_groups=False), 'n_tr': g.tract.nunique()})
    pv['_bld'] = pv.index.map(pk2bld); pv['wt'] = pv.index.map(pk_w)
    gb = pv.groupby('_bld')
    return gb.apply(lambda x: np.average(x.V, weights=x.wt), include_groups=False).reindex(BN.index), gb.n_tr.sum().reindex(BN.index).fillna(0)
Vo, no = V_site(fl[fl.period=='odd']); Ve, ne = V_site(fl[fl.period=='even'])
ok = (no>=10)&(ne>=10)
print(f'\nsites with >=10 matched tracts in BOTH Q1 halves: {ok.sum()} of {len(BN)}')
r_V = nas.spearman(Vo[ok], Ve[ok]); r_ex = nas.spearman((Vo-nb.V_null_b)[ok], (Ve-nb.V_null_b)[ok])
sb = lambda r: 2*r/(1+r)
print(f'split-half rho: V {r_V:.3f} (Spearman-Brown full-Q1 reliability {sb(r_V):.2f}); '
      f'excess over null {r_ex:.3f} (reliability {sb(r_ex):.2f})')
hho, hhe = ((BN.site_svi<MS)&(Vo>=MP))[ok], ((BN.site_svi<MS)&(Ve>=MP))[ok]
print(f'hidden-high: odd {hho.sum()}, even {hhe.sum()}, both {(hho&hhe).sum()}, full-period among these sites {BN.hidden_high[ok].sum()}')
# Q1-only estimate vs full-period estimate: does the primary window matter?
Vq1 = pd.concat([fl[fl.period=='odd'], fl[fl.period=='even']]); Vq1, nq1 = V_site(Vq1)
okq = nq1 >= 10
print(f'\nQ1-only vs full-period, sites with >=10 tracts in Q1 ({okq.sum()}): rho(V_Q1, V_full) {nas.spearman(Vq1[okq], BN.patient_svi[okq]):.3f}; '
      f'median Delta Q1 {(Vq1-BN.site_svi)[okq].median():+.3f} vs full {BN.gap[okq].median():+.3f}; '
      f'hidden-high Q1 {((BN.site_svi<MS)&(Vq1>=MP))[okq].sum()} vs full {BN.hidden_high[okq].sum()}')
pd.DataFrame(wk, columns=['week','start','origin_rows','weight']).to_csv(f'{OUT_DIR}/weekly_origin_density.csv', index=False)

In [ ]:
# ============ PACKAGE B | week-by-week reporting audit on the fixed cohort + window comparisons ============
import re, json
from collections import defaultdict
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
MS, MP = BN.site_svi.median(), BN.patient_svi.median()
xw = pd.read_parquet(f'{PAPER_DIR}/site_crosswalk.parquet')
pk2bld, pk_w = xw.set_index('placekey')['_bld'], xw.set_index('placekey')['annual_visits']
cohort_pk = set(xw.placekey)
us = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False, usecols=['FIPS','RPL_THEMES'])
us['FIPS'] = us.FIPS.astype(str).str.zfill(11)
svi_map = us.assign(R=us.RPL_THEMES.replace(-999, np.nan)).set_index('FIPS')['R']
def parse_j(js):
    if not isinstance(js, str): return {}
    s = re.sub(r'"{2,}', '"', js.strip()).strip("'").strip()
    try: return {str(k): float(v) for k, v in json.loads(s).items() if isinstance(v,(int,float))}
    except Exception: return {}

pairs = bf.find_weekly_files(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns')
hdr = bf.header_of(pairs[0][1])
cols = [c for c in ['placekey','raw_visit_counts','raw_visitor_counts','visitor_home_cbgs'] if c in hdr]
print('columns used:', cols)
weekly = []; acc = defaultdict(float)                       # (placekey, week, cbg) -> device count
for i, (d_, pth) in enumerate(pairs, 1):
    pois = pois_nonempty = rows = 0; dev = vis = visitors = 0.0; matched_w = 0.0; svi_w = 0.0
    for ch in pd.read_csv(pth, usecols=cols, chunksize=200_000, low_memory=False):
        ch = ch[ch.placekey.isin(cohort_pk)]
        pois += len(ch)
        vis += pd.to_numeric(ch.get('raw_visit_counts', 0), errors='coerce').fillna(0).sum()
        visitors += pd.to_numeric(ch.get('raw_visitor_counts', 0), errors='coerce').fillna(0).sum()
        for pk, js in zip(ch.placekey, ch.visitor_home_cbgs):
            d = parse_j(js)
            if d: pois_nonempty += 1
            for cbg, w in d.items():
                rows += 1; dev += w; acc[(pk, i, cbg)] += w
                v = svi_map.get(str(cbg).zfill(12)[:11])
                if v is not None and v == v: matched_w += w; svi_w += w * v
    weekly.append(dict(week=i, start=str(d_), pois=pois, pois_with_origins=pois_nonempty, origin_rows=rows,
                       origin_devices=dev, raw_visits=vis, raw_visitors=visitors,
                       origin_per_visitor=dev/visitors if visitors else np.nan,
                       matched_share=matched_w/dev if dev else np.nan,
                       mean_origin_svi=svi_w/matched_w if matched_w else np.nan))
    print(f'  [{i:>2}/48] {d_}  pois {pois:>5}  with origins {pois_nonempty:>5}  rows {rows:>7,}  devices {dev:>9,.0f}  '
          f'visitors {visitors:>9,.0f}  origin/visitor {dev/visitors if visitors else float("nan"):.3f}')
wk = pd.DataFrame(weekly); wk.to_csv(f'{OUT_DIR}/B_weekly_audit.csv', index=False)
print('\nweekly audit saved: B_weekly_audit.csv')

# ---- window comparisons on common sites ----
fl = pd.DataFrame([(k[0],k[1],k[2],v) for k,v in acc.items()], columns=['placekey','week','home_cbg','w'])
fl['tract'] = fl.home_cbg.astype(str).str.zfill(12).str[:11]; fl['svi'] = fl.tract.map(svi_map)
fl = fl.dropna(subset=['svi']); fl['_bld'] = fl.placekey.map(pk2bld)
fl.to_parquet(f'{PAPER_DIR}/flows_by_week.parquet', index=False)
def V_site(sub):
    g = sub.groupby('placekey')
    pv = pd.DataFrame({'V': g.apply(lambda x: np.average(x.svi, weights=x.w), include_groups=False), 'n_tr': g.tract.nunique()})
    pv['_bld'] = pv.index.map(pk2bld); pv['wt'] = pv.index.map(pk_w)
    gb = pv.groupby('_bld')
    return gb.apply(lambda x: np.average(x.V, weights=x.wt), include_groups=False).reindex(BN.index), gb.n_tr.sum().reindex(BN.index).fillna(0)
windows = {'dense w1-3': range(1,4), 'sparse w4-48': range(4,49), 'Q1 sparse w4-13': range(4,14),
           'Q2 w14-26': range(14,27), 'Q3 w27-39': range(27,40), 'Q4 w40-48': range(40,49), 'full': range(1,49)}
W = {k: V_site(fl[fl.week.isin(list(v))]) for k, v in windows.items()}
def hh(V): return (BN.site_svi < MS) & (V >= MP)
print(f'\n{"window":<18}{"sites>=10tr":>12}{"median D":>10}{"share>0.20":>11}{"HH":>6}')
for k, (V, n) in W.items():
    ok = n >= 10; D = (V - BN.site_svi)[ok]
    print(f'{k:<18}{int(ok.sum()):>12}{D.median():>+10.3f}{(D>.2).mean():>11.1%}{int(hh(V)[ok].sum()):>6}')
print('\nnon-overlapping regime comparisons on common sites (>=10 tracts in both):')
for a, b in [('dense w1-3','sparse w4-48'), ('dense w1-3','Q1 sparse w4-13'), ('Q2 w14-26','Q3 w27-39'), ('Q3 w27-39','Q4 w40-48')]:
    Va, na = W[a]; Vb, nb_ = W[b]; ok = (na>=10)&(nb_>=10)
    ha, hb = hh(Va)[ok], hh(Vb)[ok]
    print(f'  {a:<16} vs {b:<16} N={int(ok.sum()):>5}  rho(V) {nas.spearman(Va[ok], Vb[ok]):.3f}  '
          f'median D {(Va-BN.site_svi)[ok].median():+.3f} vs {(Vb-BN.site_svi)[ok].median():+.3f}  '
          f'HH {int(ha.sum())}/{int(hb.sum())} both {int((ha&hb).sum())}  '
          f'V diff median {(Va-Vb)[ok].median():+.3f}')

In [ ]:
# ============ PACKAGE B2 | are weeks 1-3 three measurements or one? ============
fl = pd.read_parquet(f'{PAPER_DIR}/flows_by_week.parquet')
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
xw = pd.read_parquet(f'{PAPER_DIR}/site_crosswalk.parquet')
pk2bld, pk_w = xw.set_index('placekey')['_bld'], xw.set_index('placekey')['annual_visits']
def V_site(sub):
    g = sub.groupby('placekey')
    pv = pd.DataFrame({'V': g.apply(lambda x: np.average(x.svi, weights=x.w), include_groups=False), 'n_tr': g.tract.nunique()})
    pv['_bld'] = pv.index.map(pk2bld); pv['wt'] = pv.index.map(pk_w); gb = pv.groupby('_bld')
    return gb.apply(lambda x: np.average(x.V, weights=x.wt), include_groups=False).reindex(BN.index), gb.n_tr.sum().reindex(BN.index).fillna(0)
Vw = {w: V_site(fl[fl.week==w]) for w in [1,2,3]}
for a, b in [(1,2),(2,3),(1,3)]:
    ok = (Vw[a][1]>=10)&(Vw[b][1]>=10)
    print(f'week {a} vs {b}: N={int(ok.sum())}  rho(V) {nas.spearman(Vw[a][0][ok], Vw[b][0][ok]):.3f}  '
          f'median |V diff| {(Vw[a][0]-Vw[b][0])[ok].abs().median():.4f}')
# identical origin dictionaries across the three weeks would mean a repeated aggregate, not weekly data
p3 = fl[fl.week.isin([1,2,3])].pivot_table(index=['placekey','home_cbg'], columns='week', values='w')
both = p3.dropna()
print(f'\nplacekey-origin pairs present in all three weeks: {len(both):,}; identical counts in all three: {(both[1]==both[2]).mul(both[2]==both[3]).mean():.1%}')
print(f'week-to-week rank corr of origin counts within pair: {both[1].corr(both[2], method="spearman"):.3f}, {both[2].corr(both[3], method="spearman"):.3f}')
sparse = fl[fl.week>=4].groupby(['placekey','home_cbg']).w.sum()
print(f'origin pairs in week 1 also seen anywhere in weeks 4-48: {p3[1].dropna().index.isin(sparse.index).mean():.1%}')

In [ ]:
fl = pd.read_parquet(f'{PAPER_DIR}/flows_by_week.parquet')
for lab, m in [('weeks 1-3', fl.week<=3), ('weeks 4-48', fl.week>=4)]:
    s = fl[m].w
    print(f'{lab}: min {s.min():.0f}  share ==4: {(s==4).mean():.1%}  share <=4: {(s<=4).mean():.1%}  '
          f'median {s.median():.0f}  p90 {s.quantile(.9):.0f}  share of total weight from values ==4: {s[s==4].sum()/s.sum():.1%}')

In [ ]:
# ============ PACKAGE C+E | two estimands, unified pooled estimator, benchmark with diagnostics ============
import json, glob, os
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
fl = pd.read_parquet(f'{PAPER_DIR}/flows_by_week.parquet')
gz = [f for f in glob.glob(f'{PAPER_DIR}/*') if 'gaz' in os.path.basename(f).lower()][0]
gaz = pd.read_csv(gz, sep='\t', dtype=str); gaz.columns = [c.strip() for c in gaz.columns]
cent = gaz.assign(GEOID=gaz.GEOID.str.zfill(11)).set_index('GEOID')[['INTPTLAT','INTPTLONG']].astype(float)
rng = np.random.default_rng(0)
def hav(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2); dl = np.radians(lon2 - lon1)
    a = np.sin((p2-p1)/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*6371.0*np.arcsin(np.sqrt(a))

# ---- the two estimands, pooled at site level (one estimator for V and for the benchmark) ----
E = {}
d1 = fl[fl.week<=3].groupby(['_bld','tract'], as_index=False).agg(svi=('svi','first'))          # presence set (union over 3 weeks)
d1['w'] = 1.0
E['E1 origin-set (weeks 1-3)'] = d1
d2 = fl[(fl.week>=4)&(fl.w>0)].groupby(['_bld','tract'], as_index=False).agg(w=('w','sum'), svi=('svi','first'))
E['E2 censored-volume (weeks 4-48)'] = d2
d2b = d2.copy(); d2b['w'] = np.where(d2b.w<=4, 2.5, d2b.w)                                         # floor sensitivity
E['E2b censored-volume, floor 4 -> 2.5'] = d2b

ledger = {}
for name, d in E.items():
    d = d[d._bld.isin(BN.index) & d.tract.isin(cent.index)].copy()
    g = d.groupby('_bld')
    V = g.apply(lambda x: np.average(x.svi, weights=x.w), include_groups=False)
    ntr = g.tract.nunique(); ok = ntr >= 10
    sites = V[ok].index
    L = BN.site_svi.reindex(sites); Vs = V.reindex(sites); D = Vs - L
    MS, MP = L.median(), Vs.median()
    hh = (L<MS)&(Vs>=MP); nlow = int((L<MS).sum())
    r = dict(N=int(len(sites)), rho_LV=nas.spearman(L, Vs), median_L=L.median(), median_V=Vs.median(),
             median_D=D.median(), share_D_pos=(D>0).mean(), share_gt20=(D>.2).mean(), share_lt20=(D<-.2).mean(),
             same_fifth=(pd.qcut(L,5,labels=False,duplicates='drop')==pd.qcut(Vs,5,labels=False,duplicates='drop')).mean(),
             HH=int(hh.sum()), n_lowL=nlow, MS=MS, MP=MP)
    # ---- block bootstrap (500 m) ----
    xy = nas.local_xy(BN.lat.reindex(sites).to_numpy(), BN.lon.reindex(sites).to_numpy())
    key = np.floor(xy/500).astype(int); key = key[:,0]*100000+key[:,1]
    groups = [np.where(key==k)[0] for k in np.unique(key)]
    Ln, Vn, Dn = L.to_numpy(), Vs.to_numpy(), D.to_numpy()
    bs = np.array([[nas.spearman(Ln[ix], Vn[ix]), np.median(Dn[ix]), (Dn[ix]>.2).mean(), ((Ln[ix]<MS)&(Vn[ix]>=MP)).sum()]
                   for ix in (np.concatenate([groups[j] for j in rng.integers(0,len(groups),len(groups))]) for _ in range(500))])
    r['ci_rho'], r['ci_medD'], r['ci_gt20'], r['ci_HH'] = [np.percentile(bs[:,k],[2.5,97.5]).round(3).tolist() for k in range(4)]
    # ---- origin-resampling classification frequency (fixed cut) ----
    freq = {}
    for b, x in d[d._bld.isin(sites)].groupby('_bld'):
        w, v = x.w.to_numpy(), x.svi.to_numpy(); p = w/w.sum()
        Vb = v[rng.choice(len(w), size=(300, len(w)), replace=True, p=p)].mean(1)
        freq[b] = (Vb>=MP).mean() if L[b] < MS else 0.0
    fq = pd.Series(freq).reindex(sites).fillna(0)
    r['HH_freq_ge90'] = int((fq>=.9).sum()); r['HH_freq_ge50'] = int((fq>=.5).sum()); r['sum_freq'] = float(fq.sum())
    # ---- benchmark: kernel fitted, IPF, diagnostics, alternative kernel ----
    T = d[d._bld.isin(sites)].groupby('tract').w.sum(); S = d[d._bld.isin(sites)].groupby('_bld').w.sum().reindex(sites)
    tr = T.index.to_numpy(); tv = d.drop_duplicates('tract').set_index('tract').svi.reindex(tr).to_numpy()
    Dk = np.empty((len(sites), len(tr)), np.float32)
    slat, slon = BN.lat.reindex(sites).to_numpy(), BN.lon.reindex(sites).to_numpy()
    for i in range(len(sites)): Dk[i] = hav(slat[i], slon[i], cent.loc[tr,'INTPTLAT'].to_numpy(), cent.loc[tr,'INTPTLONG'].to_numpy())
    dd = d[d._bld.isin(sites)].merge(pd.DataFrame({'tract':tr,'ti':np.arange(len(tr))}), on='tract')
    dd['si'] = dd._bld.map({b:i for i,b in enumerate(sites)})
    dobs = Dk[dd.si.to_numpy(), dd.ti.to_numpy()]; target = np.average(np.log(dobs+.5), weights=dd.w)
    def ipf(K0, iters=200, tol=1e-6):
        K = K0.copy()
        for it in range(iters):
            K *= (S.to_numpy()/np.maximum(K.sum(1),1e-12))[:,None]; K *= (T.to_numpy()/np.maximum(K.sum(0),1e-12))[None,:]
            if np.abs(K.sum(1)-S.to_numpy()).max()/S.max() < tol: break
        return K, it+1
    mlog = lambda K: np.average(np.log(Dk+.5), weights=K)
    def fit_and_diag(kern, grid, lab):
        best = None
        for th in grid:                                            # calibrate on the BALANCED matrix
            K, it = ipf(kern(th) * T.to_numpy()[None,:])
            m = mlog(K)
            if best is None or abs(m-target) < abs(best[2]-target): best = (th, K, m, it)
        th, K, m, it = best
        Vb = pd.Series((K*np.nan_to_num(tv)[None,:]).sum(1)/np.maximum(K.sum(1),1e-12), index=sites)
        rel_row = np.abs(K.sum(1)-S.to_numpy())/S.to_numpy(); rel_col = np.abs(K.sum(0)-T.to_numpy())/T.to_numpy()
        qs_obs = np.percentile(np.repeat(dobs, np.maximum(1,(dd.w/dd.w.min()).astype(int).clip(upper=20))), [25,50,75,90])
        flatK = K.ravel(); idx = rng.choice(flatK.size, 200000, p=flatK/flatK.sum()); qs_b = np.percentile(Dk.ravel()[idx], [25,50,75,90])
        hb = (L<MS)&(Vb>=MP)
        return dict(param=float(th), balanced_mean_logd=float(m), target_mean_logd=float(target), ipf_iters=int(it),
                    max_rel_row_err=float(rel_row.max()), max_rel_col_err=float(rel_col.max()),
                    dist_quartiles_obs_km=qs_obs.round(1).tolist(), dist_quartiles_bench_km=qs_b.round(1).tolist(),
                    rho_L_Vb=nas.spearman(L, Vb), median_D_b=float((Vb-L).median()), share_gt20_b=float(((Vb-L)>.2).mean()),
                    share_Dpos_b=float(((Vb-L)>0).mean()), rho_V_Vb=nas.spearman(Vs, Vb),
                    resid_median=float((Vs-Vb).median()), resid_abs_gt10=float(((Vs-Vb).abs()>.1).mean()),
                    HH_b=int(hb.sum()), overlap=int((hh&hb).sum()), chance_overlap=float(hh.sum()*hb.sum()/nlow),
                    recall=float((hh&hb).sum()/max(hh.sum(),1)), precision=float((hh&hb).sum()/max(hb.sum(),1)))
    r['bench_power'] = fit_and_diag(lambda b: (Dk+.5)**(-b), np.linspace(0.5,3.5,13), 'power')
    r['bench_exp']   = fit_and_diag(lambda lam: np.exp(-Dk/lam), np.array([2,3,5,8,12,20,35,60], float), 'exp')
    ledger[name] = r
    pd.DataFrame({'L':L,'V':Vs,'D':D,'HH':hh,'freq':fq}).to_parquet(f'{PAPER_DIR}/site_table_{name.split()[0]}.parquet')
    print(f'\n=== {name} ===  N={r["N"]}  rho(L,V) {r["rho_LV"]:.3f} {r["ci_rho"]}  medD {r["median_D"]:+.3f} {r["ci_medD"]}  '
          f'>0.20 {r["share_gt20"]:.1%}  D>0 {r["share_D_pos"]:.1%}  HH {r["HH"]} {r["ci_HH"]}  freq>=.9 {r["HH_freq_ge90"]}')
    for k in ['bench_power','bench_exp']:
        b = r[k]; print(f'  {k:<12} param {b["param"]:.2f}  logd {b["balanced_mean_logd"]:.3f}/{b["target_mean_logd"]:.3f}  iters {b["ipf_iters"]}  '
                        f'maxRelErr row {b["max_rel_row_err"]:.1e} col {b["max_rel_col_err"]:.1e}  distQ obs {b["dist_quartiles_obs_km"]} bench {b["dist_quartiles_bench_km"]}')
        print(f'  {"":<12} rho(L,Vb) {b["rho_L_Vb"]:.3f}  medDb {b["median_D_b"]:+.3f}  >0.20b {b["share_gt20_b"]:.1%}  rho(V,Vb) {b["rho_V_Vb"]:.3f}  '
              f'resid med {b["resid_median"]:+.3f} |resid|>.10 {b["resid_abs_gt10"]:.1%}  HH_b {b["HH_b"]}  overlap {b["overlap"]} (chance {b["chance_overlap"]:.1f})')
json.dump(ledger, open(f'{OUT_DIR}/results_ledger.json','w'), indent=1, default=float); print('\nsaved results_ledger.json + site tables')

In [ ]:
# ============ RANK STABILITY | E1 vs E2 on common sites; top-group retention across January periods ============
E1 = pd.read_parquet(f'{PAPER_DIR}/site_table_E1.parquet'); E2 = pd.read_parquet(f'{PAPER_DIR}/site_table_E2.parquet')
c = E1.join(E2, lsuffix='_1', rsuffix='_2', how='inner')
print(f'common sites {len(c)}: rank corr V {nas.spearman(c.V_1, c.V_2):.3f}, rank corr Delta {nas.spearman(c.D_1, c.D_2):.3f}; '
      f'top-quartile-by-Delta overlap {(((c.D_1>=c.D_1.quantile(.75))&(c.D_2>=c.D_2.quantile(.75))).sum())} of {int(len(c)*.25)} expected {len(c)*.25*.25:.1f} by chance')
fl = pd.read_parquet(f'{PAPER_DIR}/flows_by_week.parquet'); BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
Vw = {}
for w in [1,2,3]:
    d = fl[fl.week==w].groupby(['_bld','tract'], as_index=False).agg(svi=('svi','first')); g = d.groupby('_bld')
    V = g.svi.mean(); n = g.size(); Vw[w] = (V[n>=10] - BN.site_svi.reindex(V[n>=10].index))
ok = Vw[1].index.intersection(Vw[2].index).intersection(Vw[3].index)
top = {w: set(Vw[w].reindex(ok).nlargest(int(len(ok)*.25)).index) for w in [1,2,3]}
print(f'January periods, {len(ok)} common sites: top-quartile-by-Delta in all three periods {len(top[1]&top[2]&top[3])} of {int(len(ok)*.25)}; '
      f'in at least two {len((top[1]&top[2])|(top[1]&top[3])|(top[2]&top[3]))}; pairwise Delta rank corr '
      f'{nas.spearman(Vw[1].reindex(ok), Vw[2].reindex(ok)):.2f} / {nas.spearman(Vw[2].reindex(ok), Vw[3].reindex(ok)):.2f} / {nas.spearman(Vw[1].reindex(ok), Vw[3].reindex(ok)):.2f}')


In [ ]:
# ============ STEP 6 | does V change the selection? Delta ranking vs -L ranking ============
E1 = pd.read_parquet(f'{PAPER_DIR}/site_table_E1.parquet'); E2 = pd.read_parquet(f'{PAPER_DIR}/site_table_E2.parquet')
for name, S in [('E1', E1), ('E2', E2)]:
    rD, rL = S.D.rank(ascending=False), (-S.L).rank(ascending=False); q = int(len(S)*.25)
    topD, topL = set(S.D.nlargest(q).index), set((-S.L).nlargest(q).index)
    print(f'{name}: N={len(S)}  rank corr(Delta, -L) {nas.spearman(rD, rL):.3f}  top-quarter overlap {len(topD&topL)} of {q} (chance {q*q/len(S):.1f})  '
          f'enter top quarter only with V: {len(topD-topL)}  leave: {len(topL-topD)}')
S = E1.assign(rank_D=E1.D.rank(ascending=False).astype(int), rank_L=(-E1.L).rank(ascending=False).astype(int))
S['shift'] = S.rank_L - S.rank_D; S['top_quarter'] = S.rank_D <= int(len(S)*.25)
S['V_E2'] = E2.V.reindex(S.index)
ex = pd.concat([S.nlargest(3,'shift'), S.nsmallest(3,'shift')]).reset_index(drop=True)
ex.index = [f'S{i+1}' for i in ex.index]
print('\nTable 7 rows (anonymised):'); print(ex[['L','V','V_E2','D','rank_D','rank_L','top_quarter','freq']].round(3).to_string())

In [ ]:
# ============ FINAL (complete) | F1-F4, null comparison, decomposition, S7 robustness, Table 5 ============
import glob, os, matplotlib.pyplot as plt
FIG = f'{OUT_DIR}/figures'; os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
rng = np.random.default_rng(0)

E1 = pd.read_parquet(f'{PAPER_DIR}/site_table_E1.parquet')
E2 = pd.read_parquet(f'{PAPER_DIR}/site_table_E2.parquet')
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
fl = pd.read_parquet(f'{PAPER_DIR}/flows_by_week.parquet')
wk = pd.read_csv(f'{OUT_DIR}/B_weekly_audit.csv')

# ---------------- F1: reporting audit ----------------
fig, ax = plt.subplots(3, 1, figsize=(7, 6.5), sharex=True)
ax[0].bar(wk.week, wk.origin_rows, color='#7f9fbf'); ax[0].set_ylabel('origin entries')
ax0b = ax[0].twinx(); ax0b.plot(wk.week, wk.raw_visitors, 'k-', lw=1); ax0b.set_ylabel('raw visitors'); ax0b.spines['right'].set_visible(True)
ax[1].plot(wk.week, wk.pois_with_origins / wk.pois, 'o-', ms=3, color='#1f77b4'); ax[1].set_ylabel('share of POIs\nwith origins'); ax[1].set_ylim(0, 1.05)
ax[2].plot(wk.week, wk.origin_per_visitor, 'o-', ms=3, color='#d62728'); ax[2].set_ylabel('summed origin counts\n/ raw visitors'); ax[2].set_xlabel('weekly period of 2024')
for a in ax: a.axvline(3.5, ls='--', c='0.4', lw=.8)
fig.suptitle('Origin-field reporting audit, 4,750 practices', fontsize=10)
fig.tight_layout(); fig.savefig(f'{FIG}/F1_reporting_audit.png', dpi=300); plt.close()
print('F1 saved')

# ---------------- E1 spatial null model (per-site V_null) ----------------
gz = [f for f in glob.glob(f'{PAPER_DIR}/*') if 'gaz' in os.path.basename(f).lower()][0]
gaz = pd.read_csv(gz, sep='\t', dtype=str); gaz.columns = [c.strip() for c in gaz.columns]
cent = gaz.assign(GEOID=gaz.GEOID.str.zfill(11)).set_index('GEOID')[['INTPTLAT', 'INTPTLONG']].astype(float)
d1 = fl[fl.week <= 3].groupby(['_bld', 'tract'], as_index=False).agg(svi=('svi', 'first')); d1['w'] = 1.0
d1 = d1[d1._bld.isin(E1.index) & d1.tract.isin(cent.index)]
T = d1.groupby('tract').w.sum(); S = d1.groupby('_bld').w.sum().reindex(E1.index).fillna(0)
tr = T.index.to_numpy(); tv = d1.drop_duplicates('tract').set_index('tract').svi.reindex(tr).to_numpy()
def hav(a1, o1, a2, o2):
    p1, p2 = np.radians(a1), np.radians(a2); dl = np.radians(o2 - o1)
    x = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(x))
slat, slon = BN.lat.reindex(E1.index).to_numpy(), BN.lon.reindex(E1.index).to_numpy()
tlat, tlon = cent.loc[tr, 'INTPTLAT'].to_numpy(), cent.loc[tr, 'INTPTLONG'].to_numpy()
Dk = np.empty((len(E1), len(tr)), np.float32)
for i in range(len(E1)): Dk[i] = hav(slat[i], slon[i], tlat, tlon)
K = (Dk + .5) ** (-1.25) * T.to_numpy()[None, :]
for _ in range(40):
    K *= (S.to_numpy() / np.maximum(K.sum(1), 1e-12))[:, None]
    K *= (T.to_numpy() / np.maximum(K.sum(0), 1e-12))[None, :]
E1['V_null'] = (K * np.nan_to_num(tv)[None, :]).sum(1) / np.maximum(K.sum(1), 1e-12)
E1['D_null'] = E1.V_null - E1.L
print('null model computed')

# ---------------- F2: gap distributions ----------------
fig, ax = plt.subplots(figsize=(6.5, 3.4)); bins = np.linspace(-0.6, 0.8, 57)
ax.hist(E1.D, bins, alpha=.6, color='#7f9fbf', label=f'E1 observed (n={len(E1)}, median {E1.D.median():+.3f})')
ax.hist(E2.D, bins, alpha=.6, color='#ff9f5f', label=f'E2 observed (n={len(E2)}, median {E2.D.median():+.3f})')
ax.hist(E1.D_null, bins, histtype='step', color='k', lw=1.2, label=f'E1 spatial null (median {E1.D_null.median():+.3f})')
ax.axvline(0, c='0.3', lw=.8); ax.set_xlabel('gap  Δ = V − L  (national SVI percentile points)'); ax.set_ylabel('sites')
ax.legend(fontsize=7, frameon=False); fig.tight_layout(); fig.savefig(f'{FIG}/F2_delta_by_estimand.png', dpi=300); plt.close()
print('F2 saved')

# ---------------- F3: observed vs null V ----------------
MS, MP = E1.L.median(), E1.V.median()
hh_o = (E1.L < MS) & (E1.V >= MP); hh_n = (E1.L < MS) & (E1.V_null >= MP)
fig, ax = plt.subplots(figsize=(4.6, 4.6))
col = np.where(hh_o & hh_n, '#1f77b4', np.where(hh_o | hh_n, '#ff7f0e', '0.6'))
ax.scatter(E1.V_null, E1.V, c=col, s=8, linewidths=0); ax.plot([0, 1], [0, 1], 'k--', lw=.7); ax.axhline(MP, c='0.5', lw=.5)
ax.set_xlabel('spatial null model V'); ax.set_ylabel('observed V (E1)'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title(f'rank corr {nas.spearman(E1.V, E1.V_null):.2f}; blue = hidden-high in both, orange = one only', fontsize=8)
fig.tight_layout(); fig.savefig(f'{FIG}/F3_observed_vs_benchmark_E1.png', dpi=300); plt.close()
print('F3 saved')

# ---------------- ranks, top quarters, moved sets ----------------
q = int(len(E1) * .25)
E1['rank_D'] = E1.D.rank(ascending=False).astype(int)
E1['rank_L'] = (-E1.L).rank(ascending=False).astype(int)
topD = set(E1.D.nlargest(q).index); topL = set((-E1.L).nlargest(q).index); topN = set(E1.D_null.nlargest(q).index)
moved = {'promoted': topD - topL, 'demoted': topL - topD}
cutD = E1.D.nlargest(q).min()

# ---------------- null comparison + overlap uncertainty ----------------
low = np.where(E1.L < MS)[0]
perm = np.array([len(set(rng.choice(low, hh_o.sum(), replace=False)) & set(rng.choice(low, hh_n.sum(), replace=False))) for _ in range(2000)])
print(f'\nNULL vs OBSERVED: rank corr of gap rankings {nas.spearman(E1.D, E1.D_null):.3f}; '
      f'top-quarter overlap {len(topD & topN)} of {q} (chance {q*q/len(E1):.1f}); '
      f'hidden-high overlap {int((hh_o & hh_n).sum())} vs permutation 95% interval [{np.percentile(perm, 2.5):.0f}, {np.percentile(perm, 97.5):.0f}]')

# ---------------- null-adjusted quantity ----------------
E1['adj'] = E1.V - E1.V_null; topA = set(E1.adj.nlargest(q).index)
print(f'NULL-ADJUSTED (V_obs - V_null): rank corr with Delta {nas.spearman(E1.adj, E1.D):.3f}; with -L {nas.spearman(E1.adj, -E1.L):.3f}; '
      f'top-quarter overlap with Delta {len(topA & topD)}, with -L {len(topA & topL)} (chance {q*q/len(E1):.1f})')

# ---------------- decomposition ----------------
sd_L, sd_V = E1.L.std(), E1.V.std()
print(f'DECOMPOSITION: SD(L) {sd_L:.3f}, SD(V) {sd_V:.3f}; corr(L,V) {nas.spearman(E1.L, E1.V):.3f}; '
      f'expected Delta agreement if V were pure noise = {sd_L**2/(sd_L**2+sd_V**2):.2f}; observed 0.94')

# ---------------- S7: rank-shift robustness ----------------
E1['shift_keep'] = np.nan
for kind, sites in moved.items():
    keep = {}
    for b in sites:
        v = d1[d1._bld == b].svi.to_numpy()
        Db = v[rng.choice(len(v), size=(300, len(v)))].mean(1) - E1.L[b]
        keep[b] = ((Db >= cutD) if kind == 'promoted' else (Db < cutD)).mean()
    k = pd.Series(keep); E1.loc[k.index, 'shift_keep'] = k
    print(f'S7 {kind}: {len(k)} sites; median share of resamples on new side {k.median():.2f}; '
          f'>=90% {(k >= .9).sum()}, 50-90% {((k >= .5) & (k < .9)).sum()}, <50% {(k < .5).sum()}')

# ---------------- Table 5: most robust actual entrants and leavers ----------------
tab5 = pd.concat([E1.loc[E1.index.isin(s)].nlargest(3, 'shift_keep').assign(kind=k) for k, s in moved.items()])
tab5.index = [f'S{i+1}' for i in range(len(tab5))]
print('\nTABLE 5 (three most robust actual entrants, then three most robust actual leavers):')
print(tab5[['kind', 'L', 'V', 'D', 'rank_D', 'rank_L', 'shift_keep']].round(3).to_string())

# ---------------- F4: rank by Delta vs rank by -L ----------------
fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.scatter(E1.rank_L, E1.rank_D, s=6, c='0.7', linewidths=0)
for kind, c in [('promoted', '#1f77b4'), ('demoted', '#d62728')]:
    s = list(moved[kind]); ax.scatter(E1.rank_L[s], E1.rank_D[s], s=14, c=c, label=f'{kind} by V (n={len(s)})', linewidths=0)
ax.axhline(q, c='k', lw=.6, ls='--'); ax.axvline(q, c='k', lw=.6, ls='--'); ax.plot([1, len(E1)], [1, len(E1)], 'k:', lw=.6)
ax.set_xlabel('rank by −L (address only)'); ax.set_ylabel('rank by Δ (with mobility data)')
ax.invert_xaxis(); ax.invert_yaxis(); ax.legend(fontsize=7, frameon=False)
fig.tight_layout(); fig.savefig(f'{FIG}/F4_rank_delta_vs_negL.png', dpi=300); plt.close()
print('F4 saved')

E1.to_parquet(f'{PAPER_DIR}/site_table_E1_with_null.parquet')
print(f'\nall four figures in {FIG}; site table with null, ranks, and robustness saved')

In [ ]:
# ============ S10 | sensitivity of the Delta threshold and of the tract minimum ============
E1 = pd.read_parquet(f'{PAPER_DIR}/site_table_E1_with_null.parquet'); E2 = pd.read_parquet(f'{PAPER_DIR}/site_table_E2.parquet')
for name, S in [('E1', E1), ('E2', E2), ('E1 null', E1.assign(D=E1.D_null))]:
    print(f'{name}: share D>0.10 {(S.D>.1).mean():.1%}  >0.20 {(S.D>.2).mean():.1%}  >0.30 {(S.D>.3).mean():.1%}')
fl = pd.read_parquet(f'{PAPER_DIR}/flows_by_week.parquet'); BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
d1 = fl[fl.week<=3].groupby(['_bld','tract'], as_index=False).agg(svi=('svi','first')); g = d1.groupby('_bld'); V = g.svi.mean(); n = g.size()
for m in (5, 10, 20):
    ok = n >= m; D = (V[ok] - BN.site_svi.reindex(V[ok].index)).dropna()
    print(f'E1 with minimum {m} tracts: N={len(D)}  median D {D.median():+.3f}  share>0 {(D>0).mean():.1%}  share>0.20 {(D>.2).mean():.1%}')

In [ ]:
# ============ S11 (clean) | maps, spatial autocorrelation, zonal summaries ============
# !pip install -q geopandas esda libpysal
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt, os
from libpysal.weights import KNN; from esda.moran import Moran, Moran_Local, Moran_Local_BV
FIG = f'{OUT_DIR}/figures'; os.makedirs(FIG, exist_ok=True)
E1 = pd.read_parquet(f'{PAPER_DIR}/site_table_E1_with_null.parquet')
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
xw = pd.read_parquet(f'{PAPER_DIR}/site_crosswalk.parquet')
S = E1.join(BN[['lat','lon']]); S['S'] = S.V - S.V_null; S['n_pr'] = xw.groupby('_bld').size().reindex(S.index)
q = int(len(S)*.25); MS, MP = S.L.median(), S.V.median()
S['HH'] = (S.L < MS) & (S.V >= MP); S['topD'] = S.D >= S.D.nlargest(q).min(); S['topL'] = (-S.L) >= (-S.L).nlargest(q).min(); S['topS'] = S.S >= S.S.nlargest(q).min()
pts = gpd.GeoDataFrame(S, geometry=gpd.points_from_xy(S.lon, S.lat), crs='EPSG:4326').to_crs('EPSG:2263')

# ---- tracts with SVI (merged once) ----
tr = gpd.read_file('https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_36_tract.zip'); tr = tr[tr.COUNTYFP=='061'].to_crs('EPSG:2263')
svi = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False, usecols=['FIPS','RPL_THEMES']); svi['FIPS'] = svi.FIPS.astype(str).str.zfill(11)
tr = tr.merge(svi, left_on='GEOID', right_on='FIPS', how='left'); tr.loc[tr.RPL_THEMES < 0, 'RPL_THEMES'] = np.nan
print('tracts:', len(tr), '| with SVI:', tr.RPL_THEMES.notna().sum())

# ---- community districts ----
def load_cd():
    for url in ['https://data.cityofnewyork.us/api/geospatial/5crt-au7u?method=export&format=GeoJSON',
                'https://data.cityofnewyork.us/resource/5crt-au7u.geojson?$limit=100']:
        try:
            g = gpd.read_file(url); col = [c for c in g.columns if c.lower().replace('_','') == 'borocd'][0]
            g = g.rename(columns={col:'boro_cd'}); g = g[g.boro_cd.astype(int).between(101,112)].copy(); g['cd'] = g.boro_cd.astype(int)
            print('community districts loaded from', url); return g.to_crs('EPSG:2263')[['cd','geometry']]
        except Exception as e: print('  failed:', url[-40:], type(e).__name__)
    raise RuntimeError('community districts unavailable')
cd = load_cd()
pts = gpd.sjoin(pts, cd, how='left', predicate='within').drop(columns='index_right')
print('sites assigned to a district:', pts.cd.notna().sum(), 'of', len(pts))

# ---- F0 study area ----
fig, ax = plt.subplots(1, 2, figsize=(10, 6.5))
tr.plot(column='RPL_THEMES', cmap='YlOrRd', ax=ax[0], legend=True, legend_kwds={'shrink':.6,'label':'SVI 2022 national percentile'}, edgecolor='0.85', linewidth=.2, missing_kwds={'color':'0.9'})
pts.plot(ax=ax[0], markersize=np.clip(pts.n_pr.fillna(1), 1, 30)*1.2, color='k', alpha=.5); cd.boundary.plot(ax=ax[0], color='0.2', linewidth=.6)
ax[0].set_title('A  Tract SVI and 1,180 clinic sites', loc='left'); ax[0].set_axis_off()
cd['sites'] = cd.cd.map(pts.groupby('cd').size()).fillna(0)
cd.plot(column='sites', cmap='Blues', ax=ax[1], legend=True, legend_kwds={'shrink':.6,'label':'clinic sites'}, edgecolor='0.2', linewidth=.6)
for _, r in cd.iterrows(): ax[1].annotate(f'CD{int(r.cd)-100}', (r.geometry.centroid.x, r.geometry.centroid.y), ha='center', fontsize=7)
ax[1].set_title('B  Sites by community district', loc='left'); ax[1].set_axis_off(); fig.tight_layout(); fig.savefig(f'{FIG}/F0_study_area.png', dpi=300); plt.close()

# ---- F5 geography of Delta ----
fig, ax = plt.subplots(1, 3, figsize=(15, 6))
tr.plot(ax=ax[0], color='0.93', edgecolor='white', linewidth=.2); cd.boundary.plot(ax=ax[0], color='0.3', linewidth=.5)
pts.plot(column='D', cmap='RdBu_r', vmin=-.5, vmax=.5, markersize=9, ax=ax[0], legend=True, legend_kwds={'shrink':.6,'label':'Δ = V − L'}); ax[0].set_title('A  Δ at each site (E1)', loc='left')
cd['medD'] = cd.cd.map(pts.groupby('cd').D.median())
cd.plot(column='medD', cmap='RdBu_r', vmin=-.3, vmax=.3, ax=ax[1], legend=True, edgecolor='0.2', legend_kwds={'shrink':.6,'label':'median Δ'}); ax[1].set_title('B  Median Δ by community district', loc='left')
tr.plot(column='RPL_THEMES', cmap='YlOrRd', ax=ax[2], edgecolor='white', linewidth=.2, missing_kwds={'color':'0.9'}); pts[pts.HH].plot(ax=ax[2], color='navy', markersize=12)
ax[2].set_title(f'C  Hidden-high sites (n={int(pts.HH.sum())}) on tract SVI', loc='left')
for a in ax: a.set_axis_off()
fig.tight_layout(); fig.savefig(f'{FIG}/F5_map_delta_E1.png', dpi=300); plt.close()

# ---- Moran's I and LISA ----
w = KNN.from_dataframe(pts, k=8); w.transform = 'r'
print('\nMORAN TABLE (k=8, 999 permutations):')
for v in ['L','V','D','S']:
    m = Moran(pts[v].to_numpy(), w, permutations=999); ml = Moran_Local(pts[v].to_numpy(), w, permutations=999, seed=0)
    sig = ml.p_sim < .05; pts[f'lisa_{v}'] = np.where(sig, ml.q, 0)
    print(f'  {v}: I = {m.I:.3f}  pseudo-p = {m.p_sim:.3f}  significant local = {int(sig.sum())}  (HH {int(((ml.q==1)&sig).sum())}, LL {int(((ml.q==3)&sig).sum())}, HL {int(((ml.q==4)&sig).sum())}, LH {int(((ml.q==2)&sig).sum())})')
bv = Moran_Local_BV(pts.L.to_numpy(), pts.V.to_numpy(), w, permutations=999, seed=0); sigb = bv.p_sim < .05; pts['bv'] = np.where(sigb, bv.q, 0)
print(f'BIVARIATE L vs lag(V): HH {int(((bv.q==1)&sigb).sum())}, LL {int(((bv.q==3)&sigb).sum())}, low-L/high-lagV {int(((bv.q==2)&sigb).sum())}, high-L/low-lagV {int(((bv.q==4)&sigb).sum())}')

# ---- F6 LISA maps ----
cols = {0:'0.85', 1:'#d7191c', 2:'#abd9e9', 3:'#2c7bb6', 4:'#fdae61'}; lab = {0:'n.s.',1:'high-high',2:'low-high',3:'low-low',4:'high-low'}
fig, ax = plt.subplots(1, 3, figsize=(15, 6))
for a, col, ttl in zip(ax, ['lisa_D','bv','lisa_S'], ['A  Local Moran of Δ', 'B  Bivariate: L vs spatial lag of V', 'C  Local Moran of S = V_obs − V_null']):
    tr.plot(ax=a, color='0.95', edgecolor='white', linewidth=.2); cd.boundary.plot(ax=a, color='0.3', linewidth=.5)
    for k in [0,1,2,3,4]:
        sub = pts[pts[col]==k]
        if len(sub): sub.plot(ax=a, color=cols[k], markersize=6 if k==0 else 14, label=f'{lab[k]} ({len(sub)})')
    a.legend(fontsize=7, loc='lower right', frameon=False); a.set_title(ttl, loc='left'); a.set_axis_off()
fig.tight_layout(); fig.savefig(f'{FIG}/F6_lisa_maps.png', dpi=300); plt.close()

# ---- F7 shortlists ----
fig, ax = plt.subplots(1, 4, figsize=(18, 5.5))
for a, col, ttl in zip(ax[:3], ['topD','topL','topS'], ['A  Top quarter by Δ', 'B  Top quarter by −L (address only)', 'C  Top quarter by S']):
    tr.plot(column='RPL_THEMES', cmap='YlOrRd', ax=a, edgecolor='white', linewidth=.2, missing_kwds={'color':'0.9'}, alpha=.8); pts[pts[col]].plot(ax=a, color='navy', markersize=10); a.set_title(ttl, loc='left'); a.set_axis_off()
tr.plot(ax=ax[3], color='0.93', edgecolor='white', linewidth=.2); pts.plot(column='S', cmap='PuOr_r', vmin=-.3, vmax=.3, markersize=9, ax=ax[3], legend=True, legend_kwds={'shrink':.6,'label':'S'}); ax[3].set_title('D  S at each site', loc='left'); ax[3].set_axis_off()
fig.tight_layout(); fig.savefig(f'{FIG}/F7_map_shortlists.png', dpi=300); plt.close()

# ---- zonal table and shortlist geography ----
z = pts.groupby('cd').agg(sites=('D','size'), med_L=('L','median'), med_V=('V','median'), med_D=('D','median'), hidden_high=('HH','sum'), top_S=('topS','sum'), top_D=('topD','sum'), top_L=('topL','sum')).round(3)
z.index = [f'CD{int(c)-100}' for c in z.index]; print('\nZONAL TABLE:'); print(z.to_string())
print('\nSHORTLIST GEOGRAPHY (share of each 295-site shortlist in its three most-represented districts):')
for col in ['topD','topL','topS']:
    top3 = pts[pts[col]].groupby('cd').size().nlargest(3); print(f'  {col}: {", ".join(f"CD{int(c)-100} {n}" for c, n in top3.items())}  = {top3.sum()/q:.0%}')
print(f'\nsaved F0, F5, F6, F7 to {FIG}')

In [ ]:
# ============ S9 | robustness of the S-shortlist under origin resampling (V_null held fixed) ============
E1 = pd.read_parquet(f'{PAPER_DIR}/site_table_E1_with_null.parquet'); fl = pd.read_parquet(f'{PAPER_DIR}/flows_by_week.parquet')
d1 = fl[fl.week<=3].groupby(['_bld','tract'], as_index=False).agg(svi=('svi','first')); rng = np.random.default_rng(0)
q = int(len(E1)*.25); E1['S'] = E1.V - E1.V_null; cutS = E1.S.nlargest(q).min(); topS = E1.S.nlargest(q).index
keep = {}
for b in topS:
    v = d1[d1._bld==b].svi.to_numpy(); Sb = v[rng.choice(len(v), size=(300, len(v)))].mean(1) - E1.V_null[b]; keep[b] = (Sb >= cutS).mean()
k = pd.Series(keep)
print(f'S-shortlist ({len(k)} sites): median share of resamples in top quarter {k.median():.2f}; >=90% {(k>=.9).sum()}, 50-90% {((k>=.5)&(k<.9)).sum()}, <50% {(k<.5).sum()}')

In [ ]:
# ============ TABLE 7 | most robust actual shortlist entrants and leavers (E1) ============
E1 = pd.read_parquet(f'{PAPER_DIR}/site_table_E1_with_null.parquet')
q = int(len(E1)*.25); cutD = E1.D.nlargest(q).min()
E1['rank_D'] = E1.D.rank(ascending=False).astype(int); E1['rank_L'] = (-E1.L).rank(ascending=False).astype(int)
topD = set(E1.D.nlargest(q).index); topL = set((-E1.L).nlargest(q).index)
moved = {'entrant (promoted by V)': topD - topL, 'leaver (demoted by V)': topL - topD}
if 'shift_keep' not in E1 or E1.shift_keep.isna().all():                       # recompute robustness if not saved
    fl = pd.read_parquet(f'{PAPER_DIR}/flows_by_week.parquet'); d1 = fl[fl.week<=3].groupby(['_bld','tract'], as_index=False).agg(svi=('svi','first'))
    rng = np.random.default_rng(0); E1['shift_keep'] = np.nan
    for kind, sites in moved.items():
        for b in sites:
            v = d1[d1._bld==b].svi.to_numpy(); Db = v[rng.choice(len(v), size=(300, len(v)))].mean(1) - E1.L[b]
            E1.loc[b, 'shift_keep'] = ((Db >= cutD) if kind.startswith('entrant') else (Db < cutD)).mean()
rows = pd.concat([E1.loc[E1.index.isin(s)].nlargest(3, 'shift_keep').assign(kind=k) for k, s in moved.items()])
rows.index = [f'S{i+1}' for i in range(len(rows))]
print('TABLE 7 rows (paste this block):')
for sid, r in rows.iterrows():
    print(f'{sid} & {r.L:.3f} & {r.V:.3f} & {r.D:+.3f} & {int(r.rank_D)} & {int(r.rank_L):,} & {r.shift_keep:.2f} \\\\   % {r.kind}')

In [ ]:
# ============ FIGURES (all) | regenerate F0-F7 with final labels ============
# !pip install -q geopandas esda libpysal
import os, glob, numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from libpysal.weights import KNN
from esda.moran import Moran_Local, Moran_Local_BV

PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data'))
OUT_DIR   = globals().get('OUT_DIR',   f'{PAPER_DIR}/R_series_results')
FIG = f'{OUT_DIR}/figures'; os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
rho = lambda a, b: spearmanr(a, b, nan_policy='omit').correlation

# ---------- inputs already saved by earlier cells ----------
E1 = pd.read_parquet(f'{PAPER_DIR}/site_table_E1_with_null.parquet')      # L, V, D, V_null, D_null
E2 = pd.read_parquet(f'{PAPER_DIR}/site_table_E2.parquet')
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
xw = pd.read_parquet(f'{PAPER_DIR}/site_crosswalk.parquet')
wk = pd.read_csv(f'{OUT_DIR}/B_weekly_audit.csv')
E1['S'] = E1.V - E1.V_null
q = int(len(E1) * .25); MS, MP = E1.L.median(), E1.V.median()

# ---------- F1 reporting audit ----------
fig, ax = plt.subplots(3, 1, figsize=(7, 6.5), sharex=True)
ax[0].bar(wk.week, wk.origin_rows, color='#7f9fbf', label='origin entries'); ax[0].set_ylabel('origin entries')
a0 = ax[0].twinx(); a0.plot(wk.week, wk.raw_visitors, 'k-', lw=1, label='raw visitors'); a0.set_ylabel('raw visitors'); a0.spines['right'].set_visible(True)
ax[1].plot(wk.week, wk.pois_with_origins / wk.pois, 'o-', ms=3, color='#1f77b4'); ax[1].set_ylabel('share of POIs\nwith origins'); ax[1].set_ylim(0, 1.05)
ax[2].plot(wk.week, wk.origin_per_visitor, 'o-', ms=3, color='#d62728'); ax[2].set_ylabel('summed origin counts\n/ raw visitors'); ax[2].set_xlabel('weekly period of 2024')
for a in ax: a.axvline(3.5, ls='--', c='0.4', lw=.8)
fig.suptitle('Visitor-origin field, week by week (4,750 practices)', fontsize=10)
fig.tight_layout(); fig.savefig(f'{FIG}/F1_reporting_audit.png', dpi=300); plt.close(); print('F1 saved')

# ---------- F2 gap distributions (densities) ----------
fig, ax = plt.subplots(figsize=(6.5, 3.4)); bins = np.linspace(-0.6, 0.8, 57)
ax.hist(E1.D, bins, density=True, alpha=.6, color='#7f9fbf', label=f'E1 observed (n={len(E1)}, median {E1.D.median():+.3f})')
ax.hist(E2.D, bins, density=True, alpha=.6, color='#ff9f5f', label=f'E2 observed (n={len(E2)}, median {E2.D.median():+.3f})')
ax.hist(E1.D_null, bins, density=True, histtype='step', color='k', lw=1.2, label=f'E1 benchmark model (median {E1.D_null.median():+.3f})')
ax.axvline(0, c='0.3', lw=.8); ax.set_xlabel('gap  Δ = V − L  (SVI percentile points)'); ax.set_ylabel('density')
ax.legend(fontsize=7, frameon=False); fig.tight_layout(); fig.savefig(f'{FIG}/F2_delta_by_estimand.png', dpi=300); plt.close(); print('F2 saved')

# ---------- F3 observed vs benchmark V, four classes ----------
hh_o = (E1.L < MS) & (E1.V >= MP); hh_b = (E1.L < MS) & (E1.V_null >= MP)
cls = [('hidden-high in both', hh_o & hh_b, '#1f77b4'), ('observed only', hh_o & ~hh_b, '#ff7f0e'),
       ('benchmark only', ~hh_o & hh_b, '#2ca02c'), ('neither', ~hh_o & ~hh_b, '0.75')]
fig, ax = plt.subplots(figsize=(4.8, 4.8))
for lab, m, c in reversed(cls):
    ax.scatter(E1.V_null[m], E1.V[m], s=8, c=c, linewidths=0, label=f'{lab} ({int(m.sum())})')
ax.plot([0, 1], [0, 1], 'k--', lw=.7); ax.axhline(MP, c='0.4', lw=.6)
ax.text(0.02, MP + .01, f'median observed V = {MP:.2f} (hidden-high threshold)', fontsize=6.5, color='0.3')
ax.set_xlabel('benchmark model V'); ax.set_ylabel('observed V (E1)'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title(f'rank correlation {rho(E1.V, E1.V_null):.2f}', fontsize=8); ax.legend(fontsize=6.5, frameon=False, loc='lower right')
fig.tight_layout(); fig.savefig(f'{FIG}/F3_observed_vs_benchmark_E1.png', dpi=300); plt.close(); print('F3 saved')

# ---------- F4 rank by gap vs rank by address ----------
E1['rank_D'] = E1.D.rank(ascending=False); E1['rank_L'] = (-E1.L).rank(ascending=False)
topD = set(E1.D.nlargest(q).index); topL = set((-E1.L).nlargest(q).index)
moved = {'promoted by phone data': topD - topL, 'demoted by phone data': topL - topD}
fig, ax = plt.subplots(figsize=(4.8, 4.8))
ax.scatter(E1.rank_L, E1.rank_D, s=6, c='0.75', linewidths=0)
for (lab, s), c in zip(moved.items(), ['#1f77b4', '#d62728']):
    s = list(s); ax.scatter(E1.rank_L[s], E1.rank_D[s], s=14, c=c, linewidths=0, label=f'{lab} ({len(s)})')
ax.axhline(q, c='k', lw=.6, ls='--'); ax.axvline(q, c='k', lw=.6, ls='--'); ax.plot([1, len(E1)], [1, len(E1)], 'k:', lw=.6)
ax.text(q + 15, len(E1) * .97, f'rank {q}', fontsize=7); ax.text(len(E1) * .97, q + 15, f'rank {q}', fontsize=7, ha='right')
ax.set_xlabel('rank by −L (address only; 1 = lowest tract SVI)'); ax.set_ylabel('rank by Δ (with phone data; 1 = largest gap)')
ax.invert_xaxis(); ax.invert_yaxis(); ax.legend(fontsize=7, frameon=False, loc='lower left')
fig.tight_layout(); fig.savefig(f'{FIG}/F4_rank_delta_vs_negL.png', dpi=300); plt.close(); print('F4 saved')

# ---------- boundaries for the maps ----------
pts = gpd.GeoDataFrame(E1.join(BN[['lat', 'lon']]), geometry=gpd.points_from_xy(BN.lon.reindex(E1.index), BN.lat.reindex(E1.index)), crs='EPSG:4326').to_crs('EPSG:2263')
pts['n_pr'] = xw.groupby('_bld').size().reindex(pts.index).fillna(1)
tr = gpd.read_file('https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_36_tract.zip'); tr = tr[tr.COUNTYFP == '061'].to_crs('EPSG:2263')
svi = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False, usecols=['FIPS', 'RPL_THEMES']); svi['FIPS'] = svi.FIPS.astype(str).str.zfill(11)
tr = tr.merge(svi, left_on='GEOID', right_on='FIPS', how='left'); tr.loc[tr.RPL_THEMES < 0, 'RPL_THEMES'] = np.nan
cd = gpd.read_file('https://data.cityofnewyork.us/api/geospatial/5crt-au7u?method=export&format=GeoJSON')
col = [c for c in cd.columns if c.lower().replace('_', '') == 'borocd'][0]
cd = cd[cd[col].astype(int).between(101, 112)].copy(); cd['cd'] = cd[col].astype(int); cd = cd[['cd', 'geometry']].to_crs('EPSG:2263')
pts = gpd.sjoin(pts, cd, how='left', predicate='within').drop(columns='index_right')
pts['HH'] = (pts.L < MS) & (pts.V >= MP)
pts['topD'] = pts.index.isin(topD); pts['topL'] = pts.index.isin(topL); pts['topS'] = pts.index.isin(set(E1.S.nlargest(q).index))

# ---------- F0 study area ----------
fig, ax = plt.subplots(1, 2, figsize=(10, 6.5))
tr.plot(column='RPL_THEMES', cmap='YlOrRd', ax=ax[0], legend=True, legend_kwds={'shrink': .6, 'label': 'SVI 2022 national percentile'}, edgecolor='0.85', linewidth=.2, missing_kwds={'color': '0.9'})
pts.plot(ax=ax[0], markersize=np.clip(pts.n_pr, 1, 30) * 1.2, color='k', alpha=.5); cd.boundary.plot(ax=ax[0], color='0.2', linewidth=.6)
ax[0].set_title(f'A  Tract SVI and {len(pts):,} clinic sites', loc='left')
cd['sites'] = cd.cd.map(pts.groupby('cd').size()).fillna(0)
cd.plot(column='sites', cmap='Blues', ax=ax[1], legend=True, legend_kwds={'shrink': .6, 'label': 'clinic sites'}, edgecolor='0.2', linewidth=.6)
for _, r in cd.iterrows(): ax[1].annotate(f'CD{r.cd - 100}', (r.geometry.centroid.x, r.geometry.centroid.y), ha='center', fontsize=7)
ax[1].set_title('B  Sites by community district', loc='left')
for a in ax: a.set_axis_off()
fig.tight_layout(); fig.savefig(f'{FIG}/F0_study_area.png', dpi=300); plt.close(); print('F0 saved')

# ---------- F5 geography of the gap ----------
fig, ax = plt.subplots(1, 3, figsize=(15, 6))
tr.plot(ax=ax[0], color='0.93', edgecolor='white', linewidth=.2); cd.boundary.plot(ax=ax[0], color='0.3', linewidth=.5)
pts.plot(column='D', cmap='RdBu_r', vmin=-.5, vmax=.5, markersize=9, ax=ax[0], legend=True, legend_kwds={'shrink': .6, 'label': 'Δ = V − L'}); ax[0].set_title('A  Gap at each site (E1)', loc='left')
cd['medD'] = cd.cd.map(pts.groupby('cd').D.median())
cd.plot(column='medD', cmap='RdBu_r', vmin=-.3, vmax=.3, ax=ax[1], legend=True, edgecolor='0.2', legend_kwds={'shrink': .6, 'label': 'median Δ'}); ax[1].set_title('B  Median gap by community district', loc='left')
tr.plot(column='RPL_THEMES', cmap='YlOrRd', ax=ax[2], edgecolor='white', linewidth=.2, missing_kwds={'color': '0.9'}); pts[pts.HH].plot(ax=ax[2], color='navy', markersize=12)
ax[2].set_title(f'C  Hidden-high sites (n={int(pts.HH.sum())}) on tract SVI', loc='left')
for a in ax: a.set_axis_off()
fig.tight_layout(); fig.savefig(f'{FIG}/F5_map_delta_E1.png', dpi=300); plt.close(); print('F5 saved')

# ---------- F6 local spatial association ----------
w = KNN.from_dataframe(pts, k=8); w.transform = 'r'
for v in ['D', 'S']:
    ml = Moran_Local(pts[v].to_numpy(), w, permutations=999, seed=0); pts[f'lisa_{v}'] = np.where(ml.p_sim < .05, ml.q, 0)
bv = Moran_Local_BV(pts.L.to_numpy(), pts.V.to_numpy(), w, permutations=999, seed=0); pts['bv'] = np.where(bv.p_sim < .05, bv.q, 0)
cols = {0: '0.85', 1: '#d7191c', 2: '#abd9e9', 3: '#2c7bb6', 4: '#fdae61'}; lab = {0: 'not significant', 1: 'high–high', 2: 'low–high', 3: 'low–low', 4: 'high–low'}
fig, ax = plt.subplots(1, 3, figsize=(15, 6))
for a, c, ttl in zip(ax, ['lisa_D', 'bv', 'lisa_S'], ['A  Local Moran of the gap Δ', 'B  Bivariate: L against spatial lag of V', 'C  Local Moran of S = V_obs − V_bench']):
    tr.plot(ax=a, color='0.95', edgecolor='white', linewidth=.2); cd.boundary.plot(ax=a, color='0.3', linewidth=.5)
    for k in [0, 1, 2, 3, 4]:
        sub = pts[pts[c] == k]
        if len(sub): sub.plot(ax=a, color=cols[k], markersize=6 if k == 0 else 14, label=f'{lab[k]} ({len(sub)})')
    a.legend(fontsize=7, loc='lower right', frameon=False); a.set_title(ttl, loc='left'); a.set_axis_off()
fig.tight_layout(); fig.savefig(f'{FIG}/F6_lisa_maps.png', dpi=300); plt.close(); print('F6 saved')

# ---------- F7 where the three shortlists fall ----------
fig, ax = plt.subplots(1, 4, figsize=(18, 5.5))
for a, c, ttl in zip(ax[:3], ['topD', 'topL', 'topS'], ['A  Top quarter by gap Δ', 'B  Top quarter by address (−L)', 'C  Top quarter by S (phone-specific)']):
    tr.plot(column='RPL_THEMES', cmap='YlOrRd', ax=a, edgecolor='white', linewidth=.2, missing_kwds={'color': '0.9'}, alpha=.8)
    pts[pts[c]].plot(ax=a, color='navy', markersize=10); a.set_title(ttl, loc='left'); a.set_axis_off()
tr.plot(ax=ax[3], color='0.93', edgecolor='white', linewidth=.2)
pts.plot(column='S', cmap='PuOr_r', vmin=-.3, vmax=.3, markersize=9, ax=ax[3], legend=True, legend_kwds={'shrink': .6, 'label': 'S = V_obs − V_bench'}); ax[3].set_title('D  S at each site', loc='left'); ax[3].set_axis_off()
fig.tight_layout(); fig.savefig(f'{FIG}/F7_map_shortlists.png', dpi=300); plt.close(); print('F7 saved')

# ---------- consistency check against the numbers in the paper ----------
print(f'\nchecks: E1 n={len(E1)} median Δ {E1.D.median():+.3f} | benchmark median {E1.D_null.median():+.3f} | '
      f'ρ(V, V_bench) {rho(E1.V, E1.V_null):.2f} | hidden-high {int(hh_o.sum())}, benchmark {int(hh_b.sum())}, both {int((hh_o & hh_b).sum())} | '
      f'moved {len(moved["promoted by phone data"])}+{len(moved["demoted by phone data"])}')
print(f'all eight figures saved to {FIG}')

In [ ]:
# ============ CHECKS | setup: data, distances, geography-only benchmark ============
import os, glob, numpy as np, pandas as pd
from scipy.stats import spearmanr, fisher_exact
PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data'))
rho = lambda a, b: spearmanr(a, b, nan_policy='omit').correlation

E1 = pd.read_parquet(f'{PAPER_DIR}/site_table_E1_with_null.parquet')          # L, V, D, V_null, D_null
E1['S'] = E1.V - E1.V_null
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
fl = pd.read_parquet(f'{PAPER_DIR}/flows_by_week.parquet')
gz = [f for f in glob.glob(f'{PAPER_DIR}/*') if 'gaz' in os.path.basename(f).lower()][0]
gaz = pd.read_csv(gz, sep='\t', dtype=str); gaz.columns = [c.strip() for c in gaz.columns]
cent = gaz.assign(GEOID=gaz.GEOID.str.zfill(11)).set_index('GEOID')[['INTPTLAT', 'INTPTLONG']].astype(float)

def hav(a1, o1, a2, o2):
    p1, p2 = np.radians(a1), np.radians(a2); dl = np.radians(o2 - o1)
    return 2 * 6371 * np.arcsin(np.sqrt(np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2))

def presence(weeks):
    d = fl[fl.week.isin(weeks)].groupby(['_bld', 'tract'], as_index=False).agg(svi=('svi', 'first')); d['w'] = 1.0
    return d[d.tract.isin(cent.index)]

def benchmark(d, sites, beta, iters=40):
    """Geography-only benchmark: distance kernel (d+0.5)^-beta, balanced to observed site and tract totals.
    beta can be a number or an array with one value per site. Returns benchmark V, balanced and observed mean log-distance."""
    d = d[d._bld.isin(sites)]
    T = d.groupby('tract').w.sum(); S = d.groupby('_bld').w.sum().reindex(sites).fillna(0)
    tr = T.index.to_numpy(); tv = d.drop_duplicates('tract').set_index('tract').svi.reindex(tr).to_numpy()
    slat, slon = BN.lat.reindex(sites).to_numpy(), BN.lon.reindex(sites).to_numpy()
    Dk = np.vstack([hav(slat[i], slon[i], cent.loc[tr, 'INTPTLAT'].to_numpy(), cent.loc[tr, 'INTPTLONG'].to_numpy()) for i in range(len(sites))]).astype(np.float32)
    b = np.broadcast_to(np.asarray(beta, float).reshape(-1, 1) if np.ndim(beta) else np.full((len(sites), 1), beta), (len(sites), 1))
    K = (Dk + .5) ** (-b) * T.to_numpy()[None, :]
    for _ in range(iters):
        K *= (S.to_numpy() / np.maximum(K.sum(1), 1e-12))[:, None]; K *= (T.to_numpy() / np.maximum(K.sum(0), 1e-12))[None, :]
    Vb = pd.Series((K * np.nan_to_num(tv)[None, :]).sum(1) / np.maximum(K.sum(1), 1e-12), index=sites)
    si = pd.Series(np.arange(len(sites)), index=sites); ti = pd.Series(np.arange(len(tr)), index=tr)
    dobs = Dk[si[d._bld].to_numpy(), ti[d.tract].to_numpy()]
    return Vb, K, Dk, np.log(dobs + .5)

d1 = presence([1, 2, 3]); sites = E1.index
Vb0, K0, Dk0, _ = benchmark(d1, sites, 1.25)
print(f'setup ok: {len(sites)} E1 sites | reproduces saved benchmark: rank corr {rho(Vb0, E1.V_null):.3f} (should be ~1.000)')

In [ ]:
# ============ CHECK 1 | week-to-week stability of the proposed ranking S (vs gap and V) ============
W = {}
for wk in [1, 2, 3]:
    dw = presence([wk]); n = dw.groupby('_bld').size(); s_w = n[n >= 10].index.intersection(E1.index)
    Vw = dw[dw._bld.isin(s_w)].groupby('_bld').svi.mean()
    Vbw, *_ = benchmark(dw, s_w, 1.25)
    W[wk] = pd.DataFrame({'V': Vw, 'S': Vw - Vbw, 'D': Vw - E1.L.reindex(s_w)})
    print(f'week {wk}: {len(s_w)} sites with 10+ origin tracts')
common = W[1].index.intersection(W[2].index).intersection(W[3].index); q = int(len(common) * .25)
print(f'\nsites eligible in all three weeks: {len(common)}; top quarter = {q}')
for v, lab in [('S', 'proposed ranking S'), ('D', 'gap ranking'), ('V', 'visitor score V')]:
    r = [rho(W[a].loc[common, v], W[b].loc[common, v]) for a, b in [(1, 2), (2, 3), (1, 3)]]
    tops = [set(W[k].loc[common, v].nlargest(q).index) for k in [1, 2, 3]]
    print(f'{lab:<20} week-to-week rank corr {r[0]:.2f} / {r[1]:.2f} / {r[2]:.2f} | top quarter in all three weeks: {len(tops[0] & tops[1] & tops[2])} of {q}')
print(f'\nchance expectation for "top quarter in all three weeks" if unrelated: about {q * .25 * .25:.0f}')

In [ ]:
# ============ CHECK 2 | sensitivity of S to the benchmark specification ============
q = int(len(sites) * .25); topS0 = set(E1.S.nlargest(q).index)
north = (BN.lat.reindex(sites) > 40.787).to_numpy()          # roughly north of 96th Street
share_north = lambda top: np.mean([BN.lat[b] > 40.787 for b in top])
print(f'base (beta 1.25, one kernel): {share_north(topS0):.0%} of the S top quarter lies north of 96th St '
      f'(north has {north.mean():.0%} of all sites)\n')

res = {}
for b in [1.0, 1.5]:                                           # weaker / stronger distance effect
    Vb, *_ = benchmark(d1, sites, b); res[f'uniform beta {b}'] = E1.V - Vb
# separate kernels for north and south, each calibrated to its own observed travel distances
_, _, Dk, logd_obs = benchmark(d1, sites, 1.25)
si = pd.Series(np.arange(len(sites)), index=sites); rowN = north[si[d1[d1._bld.isin(sites)]._bld].to_numpy()]
target = {True: logd_obs[rowN].mean(), False: logd_obs[~rowN].mean()}
best, grid = None, np.arange(0.75, 2.26, 0.25)
for bn in grid:
    for bs in grid:
        Vb, K, Dk, _ = benchmark(d1, sites, np.where(north, bn, bs))
        L = np.log(Dk + .5); mN = (K[north] * L[north]).sum() / K[north].sum(); mS = (K[~north] * L[~north]).sum() / K[~north].sum()
        err = (mN - target[True]) ** 2 + (mS - target[False]) ** 2
        if best is None or err < best[0]: best = (err, bn, bs, Vb, mN, mS)
_, bn, bs, Vb, mN, mS = best
res[f'regional beta (north {bn}, south {bs})'] = E1.V - Vb
print(f'regional calibration: beta north {bn}, south {bs} | mean log-distance north {mN:.3f} vs observed {target[True]:.3f}, '
      f'south {mS:.3f} vs observed {target[False]:.3f}\n')
for lab, S_alt in res.items():
    top = set(S_alt.nlargest(q).index)
    print(f'{lab:<34} rank corr with base S {rho(S_alt, E1.S):.3f} | top-quarter overlap {len(top & topS0)} of {q} | '
          f'share of top quarter north of 96th St {share_north(top):.0%}')

In [ ]:
# ============ CHECK 3 | external check against HRSA health-centre (safety-net) sites ============
# !pip install -q geopandas statsmodels
import geopandas as gpd, statsmodels.api as sm
url = 'https://data.hrsa.gov/DataDownload/DD_Files/Health_Center_Service_Delivery_and_LookAlike_Sites.csv'
hr = pd.read_csv(url, low_memory=False, encoding='latin-1')
xcol = [c for c in hr.columns if 'x coordinate' in c.lower() or c.lower().endswith('longitude')][0]
ycol = [c for c in hr.columns if 'y coordinate' in c.lower() or c.lower().endswith('latitude')][0]
print('HRSA columns used for location:', xcol, '|', ycol, '| rows nationally:', len(hr))
hr = hr.dropna(subset=[xcol, ycol])
hr = hr[(hr[ycol].between(40.68, 40.88)) & (hr[xcol].between(-74.03, -73.90))]            # Manhattan bounding box
hg = gpd.GeoDataFrame(hr, geometry=gpd.points_from_xy(hr[xcol], hr[ycol]), crs='EPSG:4326').to_crs('EPSG:2263')
tr = gpd.read_file('https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_36_tract.zip')
man = tr[tr.COUNTYFP == '061'].to_crs('EPSG:2263').dissolve().geometry.iloc[0]
hg = hg[hg.within(man)]
print(f'HRSA health-centre sites inside Manhattan: {len(hg)}')

pts = gpd.GeoDataFrame(E1, geometry=gpd.points_from_xy(BN.lon.reindex(E1.index), BN.lat.reindex(E1.index)), crs='EPSG:4326').to_crs('EPSG:2263')
near = gpd.sjoin_nearest(pts, hg[['geometry']], how='left', max_distance=200, distance_col='dist_ft')   # 200 ft is about 60 m
E1['fqhc'] = near.groupby(level=0).dist_ft.min().reindex(E1.index).notna()
print(f'clinic sites within about 60 m of an HRSA site: {int(E1.fqhc.sum())} of {len(E1)} ({E1.fqhc.mean():.1%})\n')

q = int(len(E1) * .25)
for lab, score in [('proposed ranking S', E1.S), ('gap ranking', E1.D), ('address ranking (-L)', -E1.L)]:
    top = E1.index.isin(score.nlargest(q).index)
    a, b = int((top & E1.fqhc).sum()), int((top & ~E1.fqhc).sum()); c, d = int((~top & E1.fqhc).sum()), int((~top & ~E1.fqhc).sum())
    orr, p = fisher_exact([[a, b], [c, d]])
    print(f'{lab:<22} top quarter: {a} of {q} are HRSA sites ({a/q:.1%}) | odds ratio {orr:.2f}, Fisher p = {p:.3g}')
print(f'all E1 sites: {E1.fqhc.mean():.1%} are HRSA sites\n')

# fair comparison: does S predict HRSA status once the clinic's own tract score is accounted for?
X = sm.add_constant(pd.DataFrame({'S_z': (E1.S - E1.S.mean()) / E1.S.std(), 'L_z': (E1.L - E1.L.mean()) / E1.L.std()}))
m = sm.Logit(E1.fqhc.astype(int), X).fit(disp=0)
print('logistic regression, HRSA site ~ S + L (both standardised):')
print(f'  S: odds ratio per SD {np.exp(m.params.S_z):.2f} (95% CI {np.exp(m.conf_int().loc["S_z", 0]):.2f}-{np.exp(m.conf_int().loc["S_z", 1]):.2f}), p = {m.pvalues.S_z:.3g}')
print(f'  L: odds ratio per SD {np.exp(m.params.L_z):.2f} (95% CI {np.exp(m.conf_int().loc["L_z", 0]):.2f}-{np.exp(m.conf_int().loc["L_z", 1]):.2f}), p = {m.pvalues.L_z:.3g}')

In [ ]:
# ============ CHECK 3b | wider HRSA matching + comparison within tract-score bands ============
# !pip install -q geopandas statsmodels
import os, re, glob, difflib, numpy as np, pandas as pd, geopandas as gpd, statsmodels.api as sm
from scipy.stats import fisher_exact
from statsmodels.stats.contingency_tables import StratifiedTable
PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data'))

# ---------- inputs ----------
E1 = pd.read_parquet(f'{PAPER_DIR}/site_table_E1_with_null.parquet'); E1['S'] = E1.V - E1.V_null
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
xw = pd.read_parquet(f'{PAPER_DIR}/site_crosswalk.parquet')
pr = pd.read_parquet(f'{PAPER_DIR}/practitioners_raw.parquet')
ncol = next(c for c in ['location_name', 'name', 'poi_name'] if c in pr.columns)
acol = next((c for c in ['street_address', 'address'] if c in pr.columns), None)
poi = pr[['placekey', ncol] + ([acol] if acol else [])].drop_duplicates('placekey').merge(xw[['placekey', '_bld']], on='placekey')
poi = poi[poi._bld.isin(E1.index)]
print(f'practice names available for {poi._bld.nunique()} of {len(E1)} sites | name column: {ncol} | address column: {acol}')

hr = pd.read_csv('https://data.hrsa.gov/DataDownload/DD_Files/Health_Center_Service_Delivery_and_LookAlike_Sites.csv', low_memory=False, encoding='latin-1')
X = 'Geocoding Artifact Address Primary X Coordinate'; Y = 'Geocoding Artifact Address Primary Y Coordinate'
hname = next(c for c in hr.columns if c.lower().strip() == 'site name')
haddr = next((c for c in hr.columns if c.lower().strip() in ('site address', 'site address line 1')), None)
hr = hr.dropna(subset=[X, Y]); hr = hr[hr[Y].between(40.68, 40.88) & hr[X].between(-74.03, -73.90)]
hg = gpd.GeoDataFrame(hr, geometry=gpd.points_from_xy(hr[X], hr[Y]), crs='EPSG:4326').to_crs('EPSG:2263')
tr = gpd.read_file('https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_36_tract.zip')
man = tr[tr.COUNTYFP == '061'].to_crs('EPSG:2263').dissolve().geometry.iloc[0]
hg = hg[hg.within(man)].reset_index(drop=True)
print(f'HRSA sites in Manhattan: {len(hg)} | name column: {hname} | address column: {haddr}\n')

pts = gpd.GeoDataFrame(E1[[]].copy(), geometry=gpd.points_from_xy(BN.lon.reindex(E1.index), BN.lat.reindex(E1.index)), crs='EPSG:4326').to_crs('EPSG:2263')

# ---------- helpers ----------
GENERIC = {'the','of','and','inc','llc','pc','pllc','md','nyc','new','york','ny','center','centre','health','healthcare','medical','clinic','services','service','care','at','a'}
ABBR = {'street':'st','avenue':'ave','boulevard':'blvd','place':'pl','road':'rd','east':'e','west':'w','north':'n','south':'s','first':'1st','second':'2nd','third':'3rd'}
def norm_tokens(s):
    return [ABBR.get(w, w) for w in re.sub(r'[^a-z0-9 ]', ' ', str(s).lower()).split()]
def addr_key(s):
    t = norm_tokens(s); return ' '.join(t[:3]) if t and t[0][0].isdigit() else None
def name_sim(a, b):
    ta = set(norm_tokens(a)) - GENERIC; tb = set(norm_tokens(b)) - GENERIC
    jac = len(ta & tb) / max(len(ta | tb), 1)
    return max(jac, difflib.SequenceMatcher(None, ' '.join(sorted(ta)), ' '.join(sorted(tb))).ratio() if ta and tb else 0)

def matches_by_distance(ft):
    j = gpd.sjoin_nearest(pts, hg[['geometry']], how='left', max_distance=ft, distance_col='d')
    return set(j[j.d.notna()].index)

# candidate pairs within 400 m for address and name matching
cand = gpd.sjoin(pts.assign(geometry=pts.buffer(1312)), hg[['geometry', hname] + ([haddr] if haddr else [])], how='inner', predicate='intersects')
names = poi.groupby('_bld')[ncol].apply(list); addrs = poi.groupby('_bld')[acol].apply(list) if acol else None
addr_match, name_match = set(), set()
for b, r in cand.iterrows():
    if acol and haddr and b in addrs.index:
        hk = addr_key(r[haddr])
        if hk and any(addr_key(a) == hk for a in addrs[b]): addr_match.add(b)
    if b in names.index and max(name_sim(n, r[hname]) for n in names[b]) >= 0.6: name_match.add(b)

rules = {'distance 60 m': matches_by_distance(197), 'distance 100 m': matches_by_distance(328), 'distance 150 m': matches_by_distance(492),
         'address (within 400 m)': addr_match, 'name (within 400 m)': name_match,
         'address or name or 60 m': addr_match | name_match | matches_by_distance(197)}

# ---------- tests for each matching rule ----------
q = int(len(E1) * .25); bands = pd.qcut(E1.L, 5, labels=False)
for lab, m in rules.items():
    y = E1.index.isin(m); n = int(y.sum())
    print(f'=== {lab}: {n} clinics matched to an HRSA site ({n/len(E1):.1%}) ===')
    if n < 5: print('   too few matches to test\n'); continue
    for rl, score in [('proposed S', E1.S), ('gap', E1.D), ('address (-L)', -E1.L)]:
        top = E1.index.isin(score.nlargest(q).index); a = int((top & y).sum())
        orr, p = fisher_exact([[a, q - a], [n - a, len(E1) - q - (n - a)]])
        print(f'   {rl:<13} top quarter holds {a:>2} of {n} HRSA clinics (random: {n*q/len(E1):.1f}) | OR {orr:.2f}, p = {p:.3g}')
    Xr = sm.add_constant(pd.DataFrame({'S_z': (E1.S - E1.S.mean()) / E1.S.std(), 'L_z': (E1.L - E1.L.mean()) / E1.L.std()}))
    fit = sm.Logit(y.astype(int), Xr).fit(disp=0); ci = np.exp(fit.conf_int().loc['S_z'])
    print(f'   logistic, S adjusted for L: OR per SD {np.exp(fit.params.S_z):.2f} (95% CI {ci[0]:.2f}-{ci[1]:.2f}), p = {fit.pvalues.S_z:.3g}')
    tabs = []
    for k in range(5):
        idx = bands == k; topk = E1.S[idx] >= E1.S[idx].quantile(.75); yk = pd.Series(y, index=E1.index)[idx]
        t = np.array([[int((topk & yk).sum()), int((topk & ~yk).sum())], [int((~topk & yk).sum()), int((~topk & ~yk).sum())]])
        if t[:, 0].sum() > 0: tabs.append(t + 0.5 * (t.min() == 0))
    st = StratifiedTable(tabs)
    print(f'   within tract-score fifths (top quarter of S in each fifth): pooled OR {st.oddsratio_pooled:.2f}, '
          f'Mantel-Haenszel p = {st.test_null_odds().pvalue:.3g} ({len(tabs)} of 5 fifths contain HRSA clinics)\n')

In [ ]:
# ============ CHECK 3c | exact street-address matching to HRSA health centres ============
# !pip install -q geopandas statsmodels
import os, re, glob, numpy as np, pandas as pd, statsmodels.api as sm
from scipy.stats import fisher_exact
from statsmodels.stats.contingency_tables import StratifiedTable
PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data'))

E1 = pd.read_parquet(f'{PAPER_DIR}/site_table_E1_with_null.parquet'); E1['S'] = E1.V - E1.V_null
xw = pd.read_parquet(f'{PAPER_DIR}/site_crosswalk.parquet')
want = set(xw[xw._bld.isin(E1.index)].placekey)
print(f'practices to find: {len(want)}')

# ---------- 1. pull street addresses from the raw weekly files ----------
files = sorted(glob.glob(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns/**/*.csv*', recursive=True) + glob.glob(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns/**/*.parquet', recursive=True))
print(f'weekly files found: {len(files)}')
found = []
for f in files:
    if f.endswith('.parquet'):
        cols = pd.read_parquet(f, columns=None).columns if False else __import__('pyarrow.parquet', fromlist=['x']).ParquetFile(f).schema.names
    else:
        cols = pd.read_csv(f, nrows=0).columns
    keep = [c for c in ['placekey', 'location_name', 'street_address', 'postal_code', 'city'] if c in cols]
    if 'street_address' not in keep: print('  no street_address column in', os.path.basename(f)); continue
    it = [pd.read_parquet(f, columns=keep)] if f.endswith('.parquet') else pd.read_csv(f, usecols=keep, chunksize=200_000, low_memory=False)
    for ch in it:
        found.append(ch[ch.placekey.isin(want)].drop_duplicates('placekey'))
    got = pd.concat(found).drop_duplicates('placekey')
    print(f'  {os.path.basename(f)}: {len(got)} of {len(want)} practices have an address so far')
    if len(got) >= len(want): break
addr = pd.concat(found).drop_duplicates('placekey').merge(xw[['placekey', '_bld']], on='placekey')
addr.to_parquet(f'{PAPER_DIR}/practice_addresses.parquet', index=False)
print(f'addresses saved for {len(addr)} practices in {addr._bld.nunique()} of {len(E1)} sites\n')

# ---------- 2. normalise addresses on both sides ----------
ABBR = {'street':'st','str':'st','avenue':'ave','av':'ave','boulevard':'blvd','place':'pl','road':'rd','drive':'dr','plaza':'plz','square':'sq','terrace':'ter',
        'east':'e','west':'w','north':'n','south':'s','first':'1st','second':'2nd','third':'3rd','fourth':'4th','fifth':'5th','sixth':'6th',
        'seventh':'7th','eighth':'8th','ninth':'9th','tenth':'10th','eleventh':'11th','twelfth':'12th','saint':'st','fort':'ft'}
STOP = {'ste','suite','fl','floor','rm','room','unit','apt','bldg','building','level','lvl','frnt','front'}
def akey(s):
    t = re.sub(r'[^a-z0-9 ]', ' ', str(s).lower()).split()
    out = []
    for w in t:
        if w in STOP: break
        w = ABBR.get(w, w)
        w = re.sub(r'^(\d+)(st|nd|rd|th)$', r'\1', w)          # 42nd -> 42
        out.append(w)
    if not out or not out[0][0].isdigit(): return None
    out[0] = re.match(r'\d+', out[0]).group(0)                  # 10-12 -> 10
    return ' '.join(out[:5])

hr = pd.read_csv('https://data.hrsa.gov/DataDownload/DD_Files/Health_Center_Service_Delivery_and_LookAlike_Sites.csv', low_memory=False, encoding='latin-1')
sc = next(c for c in hr.columns if c.lower().strip() in ('site state abbreviation', 'site state'))
cc = next((c for c in hr.columns if 'county' in c.lower() and 'name' in c.lower()), None)
hr = hr[hr[sc] == 'NY']
if cc: hr = hr[hr[cc].astype(str).str.contains('New York', case=False)]
hr['key'] = hr['Site Address'].map(akey)
addr['key'] = addr.street_address.map(akey)
hkeys = set(hr.key.dropna())
m_addr = set(addr[addr.key.isin(hkeys)]._bld)
print(f'HRSA sites in New York County: {len(hr)} | with a usable address key: {hr.key.notna().sum()}')
print(f'clinic sites sharing an exact street address with an HRSA site: {len(m_addr)}')
print('examples:', addr[addr.key.isin(hkeys)][['location_name', 'street_address']].head(8).to_string(index=False), '\n')

# ---------- 3. tests ----------
import geopandas as gpd
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet').set_index('_bld')
X, Y = 'Geocoding Artifact Address Primary X Coordinate', 'Geocoding Artifact Address Primary Y Coordinate'
hg = gpd.GeoDataFrame(hr.dropna(subset=[X, Y]), geometry=gpd.points_from_xy(hr.dropna(subset=[X, Y])[X], hr.dropna(subset=[X, Y])[Y]), crs='EPSG:4326').to_crs('EPSG:2263')
pts = gpd.GeoDataFrame(E1[[]].copy(), geometry=gpd.points_from_xy(BN.lon.reindex(E1.index), BN.lat.reindex(E1.index)), crs='EPSG:4326').to_crs('EPSG:2263')
j = gpd.sjoin_nearest(pts, hg[['geometry']], how='left', max_distance=197, distance_col='d'); m60 = set(j[j.d.notna()].index)
q = int(len(E1) * .25); bands = pd.qcut(E1.L, 5, labels=False)
for lab, m in {'exact address': m_addr, 'exact address or within 60 m': m_addr | m60}.items():
    y = E1.index.isin(m); n = int(y.sum())
    print(f'=== {lab}: {n} clinics matched ({n/len(E1):.1%}) ===')
    if n < 5: print('   too few to test\n'); continue
    for rl, score in [('proposed S', E1.S), ('gap', E1.D), ('address (-L)', -E1.L)]:
        top = E1.index.isin(score.nlargest(q).index); a = int((top & y).sum())
        orr, p = fisher_exact([[a, q - a], [n - a, len(E1) - q - (n - a)]])
        print(f'   {rl:<13} top quarter holds {a:>2} of {n} (random: {n*q/len(E1):.1f}) | OR {orr:.2f}, p = {p:.3g}')
    Xr = sm.add_constant(pd.DataFrame({'S_z': (E1.S - E1.S.mean()) / E1.S.std(), 'L_z': (E1.L - E1.L.mean()) / E1.L.std()}))
    fit = sm.Logit(y.astype(int), Xr).fit(disp=0); ci = np.exp(fit.conf_int().loc['S_z'])
    print(f'   logistic, S adjusted for L: OR per SD {np.exp(fit.params.S_z):.2f} (95% CI {ci[0]:.2f}-{ci[1]:.2f}), p = {fit.pvalues.S_z:.3g}')
    tabs = []
    for k in range(5):
        idx = bands == k; tk = E1.S[idx] >= E1.S[idx].quantile(.75); yk = pd.Series(y, index=E1.index)[idx]
        t = np.array([[int((tk & yk).sum()), int((tk & ~yk).sum())], [int((~tk & yk).sum()), int((~tk & ~yk).sum())]])
        if t[:, 0].sum() > 0: tabs.append(t + 0.5 * (t.min() == 0))
    st = StratifiedTable(tabs)
    print(f'   within tract-score fifths: pooled OR {st.oddsratio_pooled:.2f}, Mantel-Haenszel p = {st.test_null_odds().pvalue:.3g}\n')

## Expected visitors and the Medicare patient check

These cells produce `R_series_results/expected_visitors_E1.parquet` and `R_series_results/medicare_site_validation.parquet`, which the Table 7 and figure cells below read. The Medicare cell downloads 2023 provider data from data.cms.gov (a few minutes) unless `R_series_results/cms_providers_manhattan_2023.parquet` already exists.

In [ ]:
# ============ EXPECTED VISITORS | what location, access, specialty and nearby population predict, and what is left ============
# Self-contained: reloads the saved files, rebuilds the geography-only benchmark, then
#  1 adds clinic specialty (NAICS), subway access, and the resident population around each clinic
#  2 predicts each clinic's visitor-origin score V from those, out of fold with 1 km spatial blocks
#  3 ranks clinics on the unexpected part R = V - expected V, and compares R with S, the gap, and the address
#  4 compares whole origin distributions (top and bottom SVI quintiles, Wasserstein distance), not only means
#  5 compares clinics within specialty, tests week-to-week reliability, and checks R against HRSA health centres
# Everything runs on site-by-tract presence tables; no device-level data exist or are created.
import os, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt, statsmodels.api as sm
from scipy.stats import spearmanr, pearsonr, fisher_exact, kruskal, wasserstein_distance, norm, chi2
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neighbors import NearestNeighbors
from statsmodels.stats.multitest import multipletests
if str(globals().get('DATA_DIR', 'data')).startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data'))
OUT_DIR = f'{PAPER_DIR}/R_series_results'; FIG = f'{OUT_DIR}/figures'; os.makedirs(FIG, exist_ok=True)
rho = lambda a, b: spearmanr(a, b, nan_policy='omit').correlation
zs = lambda s: (s - s.mean()) / s.std()

# ---------- 0. setup (same as the CHECKS setup cell) ----------
E1 = pd.read_parquet(f'{PAPER_DIR}/site_table_E1_with_null.parquet'); E1['S'] = E1.V - E1.V_null
BN = pd.read_parquet(f'{PAPER_DIR}/building_svi_quadrants_national.parquet')
if '_bld' in BN.columns: BN = BN.set_index('_bld')
fl = pd.read_parquet(f'{PAPER_DIR}/flows_by_week.parquet')
gz = [f for f in glob.glob(f'{PAPER_DIR}/*') if 'gaz' in os.path.basename(f).lower()][0]
gaz = pd.read_csv(gz, sep='\t', dtype=str); gaz.columns = [c.strip() for c in gaz.columns]
cent = gaz.assign(GEOID=gaz.GEOID.str.zfill(11)).set_index('GEOID')[['INTPTLAT', 'INTPTLONG']].astype(float)

def hav(a1, o1, a2, o2):
    p1, p2 = np.radians(a1), np.radians(a2); dl = np.radians(o2 - o1)
    return 2 * 6371 * np.arcsin(np.sqrt(np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2))

def presence(weeks):
    d = fl[fl.week.isin(weeks)].groupby(['_bld', 'tract'], as_index=False).agg(svi=('svi', 'first')); d['w'] = 1.0
    return d[d.tract.isin(cent.index)]

def benchmark(d, sites, beta, iters=40):
    d = d[d._bld.isin(sites)]
    T = d.groupby('tract').w.sum(); S = d.groupby('_bld').w.sum().reindex(sites).fillna(0)
    tr = T.index.to_numpy(); tv = d.drop_duplicates('tract').set_index('tract').svi.reindex(tr).to_numpy()
    slat, slon = BN.lat.reindex(sites).to_numpy(), BN.lon.reindex(sites).to_numpy()
    Dk = np.vstack([hav(slat[i], slon[i], cent.loc[tr, 'INTPTLAT'].to_numpy(), cent.loc[tr, 'INTPTLONG'].to_numpy()) for i in range(len(sites))]).astype(np.float32)
    b = np.broadcast_to(np.asarray(beta, float).reshape(-1, 1) if np.ndim(beta) else np.full((len(sites), 1), beta), (len(sites), 1))
    K = (Dk + .5) ** (-b) * T.to_numpy()[None, :]
    for _ in range(iters):
        K *= (S.to_numpy() / np.maximum(K.sum(1), 1e-12))[:, None]; K *= (T.to_numpy() / np.maximum(K.sum(0), 1e-12))[None, :]
    Vb = pd.Series((K * np.nan_to_num(tv)[None, :]).sum(1) / np.maximum(K.sum(1), 1e-12), index=sites)
    return Vb, K, tr, tv

sites = E1.index; n = len(sites); q = int(n * .25)
slat, slon = BN.lat.reindex(sites).to_numpy(), BN.lon.reindex(sites).to_numpy()
X = pd.DataFrame(index=sites)
X['L'], X['V'], X['V_bench'] = E1.L, E1.V, E1.V_null
X['D'], X['S'] = X.V - X.L, X.V - X.V_bench

d1 = presence([1, 2, 3]); dd = d1[d1._bld.isin(sites)]
Vb, K, tr_, tv = benchmark(d1, sites, 1.25)
ok = ~np.isnan(tv)
obs = dd.dropna(subset=['svi']).groupby('_bld').svi
print(f'[0] {n} E1 sites | benchmark rebuilt: rank corr with saved V_bench {rho(Vb, X.V_bench):.3f} | '
      f'V from presence table vs saved V: {rho(obs.mean().reindex(sites), X.V):.3f} (both should be ~1.000)')
nan_w = K[:, ~ok].sum() / K.sum()
Vb_fix = pd.Series((K[:, ok] * tv[ok][None, :]).sum(1) / K[:, ok].sum(1), index=sites)
print(f'    benchmark weight on origin tracts with no SVI: {nan_w:.2%}; S recomputed without them: rank corr with S {rho(X.V - Vb_fix, X.S):.3f}, '
      f'median change {np.median((X.V - Vb_fix) - X.S):+.4f}\n')

# ---------- 1. specialty from NAICS codes in the raw weekly files ----------
xw = pd.read_parquet(f'{PAPER_DIR}/site_crosswalk.parquet'); xw = xw[xw._bld.isin(sites)]
want = set(xw.placekey); got = []
files = sorted(glob.glob(f'{PAPER_DIR}/NYC-2024-Weekly-Patterns/**/*.csv*', recursive=True))
for k, f in enumerate(files):
    hdr = {c.lower(): c for c in pd.read_csv(f, nrows=0).columns}
    assert 'naics_code' in hdr, f'no naics_code column in {os.path.basename(f)}: {list(hdr)[:15]}'
    use = [hdr[c] for c in ['placekey', 'naics_code', 'sub_category'] if c in hdr]
    for ch in pd.read_csv(f, usecols=use, chunksize=200_000, low_memory=False, dtype=str):
        ch.columns = [c.lower() for c in ch.columns]; got.append(ch[ch.placekey.isin(want)])
    nx = pd.concat(got).drop_duplicates('placekey')
    if len(nx) >= len(want): break
print(f'[1] NAICS code found for {len(nx)} of {len(want)} practices (read {k + 1} weekly files)')
SIX = {'621111': 'Physician', '621112': 'Mental health', '621330': 'Mental health', '621420': 'Mental health',
       '621310': 'Other practitioner', '621320': 'Other practitioner', '621340': 'Other practitioner',
       '621391': 'Other practitioner', '621399': 'Other practitioner', '621410': 'Outpatient centre',
       '621491': 'Outpatient centre', '621492': 'Outpatient centre', '621493': 'Outpatient centre', '621498': 'Outpatient centre'}
FOUR = {'6211': 'Physician', '6213': 'Other practitioner', '6214': 'Outpatient centre'}
CLS = ['Physician', 'Mental health', 'Other practitioner', 'Outpatient centre']
nx['naics'] = nx.naics_code.astype(str).str.extract(r'(\d{4,6})')[0]
nx['spec'] = nx.naics.map(lambda c: SIX.get(str(c)[:6], FOUR.get(str(c)[:4], 'Unknown')) if isinstance(c, str) else 'Unknown')
pr = xw[['placekey', '_bld']].merge(nx[['placekey', 'naics', 'spec']], on='placekey', how='left').fillna({'spec': 'Unknown'})
tab = pd.crosstab(pr._bld, pr.spec).reindex(columns=CLS, fill_value=0).reindex(sites, fill_value=0)
X['n_pr'] = pr.groupby('_bld').size().reindex(sites).fillna(1)
X['spec'] = np.where(tab.sum(axis=1) > 0, tab.idxmax(axis=1), 'Unknown')
for c, s in [('Mental health', 'sh_mental'), ('Other practitioner', 'sh_other'), ('Outpatient centre', 'sh_centre')]:
    X[s] = (tab[c] / tab.sum(axis=1).replace(0, np.nan)).fillna(0)
print('    practices by 6-digit NAICS:', pr.naics.value_counts().head(8).to_dict())
print('    sites by dominant specialty:', X.spec.value_counts().to_dict(), '| single-specialty sites:', f'{(tab.gt(0).sum(axis=1) == 1).mean():.0%}\n')

# ---------- 2. accessibility: MTA subway stations ----------
st = None
for u in ['https://data.ny.gov/api/views/39hk-dx4f/rows.csv?accessType=DOWNLOAD', 'https://data.ny.gov/resource/39hk-dx4f.csv?$limit=5000']:
    try:
        st = pd.read_csv(u); break
    except Exception as e:
        print('    station download failed:', u, type(e).__name__)
latc = next(c for c in st.columns if 'latitude' in c.lower()); lonc = next(c for c in st.columns if 'longitude' in c.lower())
rtc = next(c for c in st.columns if 'route' in c.lower())
st = st.dropna(subset=[latc, lonc])
Dst = hav(slat[:, None], slon[:, None], st[latc].to_numpy(float)[None, :], st[lonc].to_numpy(float)[None, :])
routes = st[rtc].fillna('').astype(str).str.split().to_list()
X['d_subway'] = Dst.min(1)
X['routes_500'] = [len(set().union(*[set(routes[j]) for j in np.where(Dst[i] <= .5)[0]])) for i in range(n)]
print(f'[2] {len(st)} subway station records ({latc}, {lonc}, {rtc}) | median distance to nearest station {X.d_subway.median() * 1000:.0f} m | '
      f'median routes within 500 m {X.routes_500.median():.0f}\n')

# ---------- 3. surrounding resident population (census, no phone data) ----------
svi = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False, usecols=['FIPS', 'RPL_THEMES', 'E_TOTPOP'], dtype={'FIPS': str})
svi['FIPS'] = svi.FIPS.str.zfill(11)
res = cent.join(svi.set_index('FIPS'), how='inner')
res = res[(res.RPL_THEMES >= 0) & (res.E_TOTPOP > 0) & res.INTPTLAT.between(40.45, 41.05) & res.INTPTLONG.between(-74.35, -73.65)]
Dres = hav(slat[:, None], slon[:, None], res.INTPTLAT.to_numpy()[None, :], res.INTPTLONG.to_numpy()[None, :])
P, v = res.E_TOTPOP.to_numpy(float), res.RPL_THEMES.to_numpy(float); near = Dres <= 1.0
X['pop_1km'] = (near * P).sum(1)
X['L_1km'] = np.where(X.pop_1km > 0, (near * P * v).sum(1) / np.maximum(X.pop_1km, 1), X.L)
G = P[None, :] * (Dres + .5) ** -1.25
X['V_pop'] = (G * v).sum(1) / G.sum(1)     # residents within reach, weighted by the same distance decay: an address-only expected V
print(f'[3] {len(res)} populated tracts in the metropolitan core | median residents within 1 km {X.pop_1km.median():,.0f} | '
      f'rank corr V_pop with V {rho(X.V_pop, X.V):.2f}, with V_bench {rho(X.V_pop, X.V_bench):.2f}\n')

# ---------- 4. expected V: nested models, out-of-fold predictions on 1 km spatial blocks ----------
x_km = (slon + 73.979192) * 111.32 * np.cos(np.radians(40.754834)); y_km = (slat - 40.754834) * 110.57
blk = pd.factorize(pd.Series(np.floor(x_km).astype(int)).astype(str) + '_' + pd.Series(np.floor(y_km).astype(int)).astype(str))[0]
X['log_pop_1km'], X['log_d_subway'], X['log_n_pr'] = np.log1p(X.pop_1km), np.log(X.d_subway + .05), np.log(X.n_pr)
X['lat'], X['lon'] = slat, slon
FA = ['L']
FB = FA + ['L_1km', 'V_pop', 'log_pop_1km', 'log_d_subway', 'routes_500', 'log_n_pr', 'sh_mental', 'sh_other', 'sh_centre']
FC = FB + ['V_bench']
FD = FC + ['lat', 'lon']

def oof(y, F, groups, kind='ols'):
    y = np.asarray(y, float); F = np.asarray(F, float); pred = np.full(len(y), np.nan)
    for a, b in GroupKFold(n_splits=5).split(F, y, groups):
        m = LinearRegression() if kind == 'ols' else HistGradientBoostingRegressor(max_iter=300, learning_rate=.05, min_samples_leaf=20, random_state=0)
        pred[b] = m.fit(F[a], y[a]).predict(F[b])
    return pred
r2 = lambda y, p: 1 - np.sum((np.asarray(y) - p) ** 2) / np.sum((np.asarray(y) - np.mean(y)) ** 2)
top = lambda s: set(s.nlargest(q).index)
chance = q * q / n
print(f'[4] expected visitor-origin score V, predicted out of fold (5 folds of 1 km blocks, {blk.max() + 1} blocks); top quarter = {q} sites, chance overlap {chance:.0f}')
print(f'    {"model":<58}{"R2":>6}{"rho(resid,S)":>14}{"top-q overlap with S":>22}')
print(f'    {"geography-only benchmark as is (V - V_bench = S)":<58}{r2(X.V, X.V_bench):>6.2f}{1:>14.2f}{q:>22}')
PRED = {}
for lab, F, kind in [('A  address tract SVI only (L)', FA, 'ols'),
                     ('B  A + residents, subway, specialty, size (no phone data)', FB, 'ols'),
                     ('C  B + geography-only benchmark (main expanded model)', FC, 'ols'),
                     ('D  C + coordinates, gradient boosting (most flexible)', FD, 'gbm')]:
    p = oof(X.V, X[F], blk, kind); PRED[lab[0]] = p; R_ = X.V - p
    print(f'    {lab:<58}{r2(X.V, p):>6.2f}{rho(R_, X.S):>14.2f}{len(top(R_) & top(X.S)):>22}')
X['V_hat'] = PRED['C']; X['R'] = X.V - X.V_hat; R2C = r2(X.V, X.V_hat)
X['R_B'] = X.V - PRED['B']; X['R_D'] = X.V - PRED['D']

Fz = X[FC].copy(); cont = [c for c in FC if not c.startswith('sh_')]; Fz[cont] = (Fz[cont] - Fz[cont].mean()) / Fz[cont].std()
mC = sm.OLS(X.V, sm.add_constant(Fz)).fit(cov_type='HC3')
print('\n    model C fitted to all sites (continuous terms per SD; specialty terms per unit share vs physician offices):')
for t in FC:
    lo, hi = mC.conf_int().loc[t]
    print(f'      {t:<14}{mC.params[t]:+.3f}  (95% CI {lo:+.3f} to {hi:+.3f})  p = {mC.pvalues[t]:.3g}')

# ---------- 5. the proposed residual ranking R against S, the gap, and the address ----------
xy = np.c_[x_km, y_km]; nbr = NearestNeighbors(n_neighbors=9).fit(xy).kneighbors(xy, return_distance=False)[:, 1:]
def moran(z, perm=999, seed=0):
    z = np.asarray(z, float) - np.mean(z); I = (z * z[nbr].mean(1)).sum() / (z ** 2).sum()
    rng = np.random.default_rng(seed); sim = np.empty(perm)
    for k in range(perm):
        zp = rng.permutation(z); sim[k] = (zp * zp[nbr].mean(1)).sum() / (zp ** 2).sum()
    return I, (1 + (sim >= I).sum()) / (perm + 1)
north = pd.Series(slat > 40.787, index=sites)          # roughly north of 96th Street
print(f'\n[5] rankings compared (n = {n}; north of 96th St holds {north.mean():.0%} of sites)')
for lab, s in [('R  expanded residual (model C)', X.R), ('R_B  residual, no phone-derived terms', X.R_B), ('R_D  residual, flexible model', X.R_D),
               ('S  benchmark signal (paper)', X.S), ('gap (V - L)', X.D), ('address (-L)', -X.L)]:
    I, p = moran(s)
    print(f'    {lab:<40} rho with R {rho(s, X.R):+.2f} | with -L {rho(s, -X.L):+.2f} | top-q overlap with R {len(top(s) & top(X.R)):>3}, '
          f'with -L {len(top(s) & top(-X.L)):>3} | north share of top q {north[list(top(s))].mean():.0%} | Moran I {I:.3f} (p {p:.3f})')

# ---------- 6. whole distributions, not only means ----------
HI, LO = .80, .20
tvo = tv[ok]; Kk = K[:, ok] / K[:, ok].sum(1, keepdims=True)
X['hi_obs'], X['lo_obs'] = obs.apply(lambda s: (s >= HI).mean()).reindex(sites), obs.apply(lambda s: (s <= LO).mean()).reindex(sites)
X['hi_bench'], X['lo_bench'] = Kk[:, tvo >= HI].sum(1), Kk[:, tvo <= LO].sum(1)
for c in ['p25', 'p50', 'p75']: X[c + '_obs'] = obs.quantile(int(c[1:]) / 100).reindex(sites)
grp = {k: g.to_numpy() for k, g in obs}
X['W1'] = [wasserstein_distance(grp[s], tvo, None, Kk[i]) if s in grp else np.nan for i, s in enumerate(sites)]
X['tail_excess'] = X.hi_obs - X.hi_bench; X['low_deficit'] = X.lo_bench - X.lo_obs
X['shift_share'] = (X.S.abs() / X.W1).clip(upper=1)
pH = oof(X.hi_obs, X[FB + ['hi_bench']], blk); X['R_hi'] = X.hi_obs - pH
print(f'\n[6] origin-tract distributions (E1 counts every origin tract once, so these are distributions of origin tracts, not of visitors)')
print(f'    share of origin tracts in the top national SVI quintile (>= {HI}): observed median {X.hi_obs.median():.2f}, benchmark {X.hi_bench.median():.2f}; '
      f'bottom quintile (<= {LO}): observed {X.lo_obs.median():.2f}, benchmark {X.lo_bench.median():.2f}')
print(f'    Wasserstein distance observed vs benchmark: median {X.W1.median():.3f}; |S| / W1 (share of the difference that is a pure shift): median {X.shift_share.median():.2f}')
for lab, s in [('top-quintile excess', X.tail_excess), ('bottom-quintile deficit', X.low_deficit), ('top-quintile residual (model B + tail benchmark)', X.R_hi), ('Wasserstein distance', X.W1)]:
    print(f'    {lab:<48} rho with S {rho(s, X.S):+.2f}, with R {rho(s, X.R):+.2f} | top-q overlap with S {len(top(s) & top(X.S)):>3}, with R {len(top(s) & top(X.R)):>3}')
for lab, s in [('S', X.S), ('R', X.R)]:
    t = list(top(s))
    print(f'    {lab} top quarter: top-quintile origins {X.hi_obs[t].mean():.2f} observed vs {X.hi_bench[t].mean():.2f} expected; '
          f'bottom-quintile origins {X.lo_obs[t].mean():.2f} vs {X.lo_bench[t].mean():.2f}; sites with more top-quintile origins than expected {np.mean(X.tail_excess[t] > 0):.0%}')

# ---------- 7. within specialty ----------
print(f'\n[7] by dominant specialty (top quarter by R = {q} sites)')
print(f'    {"specialty":<20}{"sites":>6}{"median gap":>12}{"median S":>10}{"median R":>10}{"in top q by R":>15}{"expected":>10}')
for c in CLS + ['Unknown']:
    g = X[X.spec == c]
    if len(g) == 0: continue
    print(f'    {c:<20}{len(g):>6}{g.D.median():>+12.3f}{g.S.median():>+10.3f}{g.R.median():>+10.3f}{g.index.isin(list(top(X.R))).sum():>15}{len(g) / n * q:>10.0f}')
gs = [g.R.to_numpy() for _, g in X.groupby('spec') if len(g) >= 10]
print(f'    Kruskal-Wallis, R across specialties: p = {kruskal(*gs).pvalue:.3g}; same for S: p = {kruskal(*[g.S.to_numpy() for _, g in X.groupby("spec") if len(g) >= 10]).pvalue:.3g}')
X['R_within'] = X.groupby('spec').R.rank(pct=True)
tw = set(X.index[X.R_within > .75])
print(f'    ranking within specialty vs overall: rho {rho(X.R_within, X.R):.2f} | top quarter within each specialty ({len(tw)} sites) shares {len(tw & top(X.R))} with the overall top quarter')

# ---------- 8. is R reproducible from week to week? ----------
Wk = {}
for wk in [1, 2, 3]:
    dw = presence([wk]); nn = dw.groupby('_bld').size(); s_w = nn[nn >= 10].index.intersection(sites)
    Vw = dw[dw._bld.isin(s_w)].groupby('_bld').svi.mean().reindex(s_w)
    Vbw, *_ = benchmark(dw, s_w, 1.25)
    Fw = X.loc[s_w, FB].assign(V_bench=Vbw.to_numpy())[FC]
    pw = oof(Vw, Fw, pd.Series(blk, index=sites).loc[s_w].to_numpy())
    Wk[wk] = pd.DataFrame({'R': Vw - pw, 'S': Vw - Vbw, 'V': Vw}, index=s_w)
cm = Wk[1].index.intersection(Wk[2].index).intersection(Wk[3].index); qc = int(len(cm) * .25)
print(f'\n[8] week-to-week reproducibility on {len(cm)} sites with 10+ origin tracts in each January week (top quarter = {qc}; chance in all three ~{qc / 16:.0f})')
rel = {}
for v_ in ['R', 'S', 'V']:
    pairs = [(1, 2), (2, 3), (1, 3)]
    rs = [rho(Wk[a].loc[cm, v_], Wk[b].loc[cm, v_]) for a, b in pairs]; rp = [pearsonr(Wk[a].loc[cm, v_], Wk[b].loc[cm, v_])[0] for a, b in pairs]
    tops = [set(Wk[k].loc[cm, v_].nlargest(qc).index) for k in [1, 2, 3]]; rel[v_] = np.mean(rp)
    print(f'    {v_}: rank corr {rs[0]:.2f} / {rs[1]:.2f} / {rs[2]:.2f} | Pearson mean {np.mean(rp):.2f} | top quarter in all three weeks {len(tops[0] & tops[1] & tops[2])} of {qc}')
print(f'    share of variance in V not predicted by model C: {1 - R2C:.2f}; reproducible share of that residual (mean week-to-week r): {rel["R"]:.2f}; '
      f'so about {(1 - R2C) * rel["R"]:.0%} of the variance in V is both unexpected and reproducible')

# ---------- 9. which clinics differ beyond sampling noise? ----------
X['n_orig'] = obs.size().reindex(sites); X['se'] = obs.std(ddof=1).reindex(sites) / np.sqrt(X.n_orig)
X['z'] = X.R / X.se; pv = 2 * norm.sf(X.z.abs()); X['q_bh'] = multipletests(pv, method='fdr_bh')[1]
sig = X.q_bh < .05
print(f'\n[9] clinics whose observed V differs from expected V beyond origin-sampling noise (two-sided, Benjamini-Hochberg q < 0.05; model error not included):')
print(f'    more vulnerable than expected: {int((sig & (X.R > 0)).sum())} | less vulnerable than expected: {int((sig & (X.R < 0)).sum())} | '
      f'of these, |R| >= 0.10: {int((sig & (X.R >= .10)).sum())} and {int((sig & (X.R <= -.10)).sum())}')

# ---------- 10. external check: HRSA health-centre sites ----------
hr = pd.read_csv('https://data.hrsa.gov/DataDownload/DD_Files/Health_Center_Service_Delivery_and_LookAlike_Sites.csv', low_memory=False, encoding='latin-1')
xcol = [c for c in hr.columns if 'x coordinate' in c.lower() or c.lower().endswith('longitude')][0]
ycol = [c for c in hr.columns if 'y coordinate' in c.lower() or c.lower().endswith('latitude')][0]
hr = hr.dropna(subset=[xcol, ycol]); hr = hr[hr[ycol].between(40.68, 40.88) & hr[xcol].between(-74.03, -73.90)]
dh = hav(slat[:, None], slon[:, None], hr[ycol].to_numpy(float)[None, :], hr[xcol].to_numpy(float)[None, :]).min(1)
print(f'\n[10] HRSA health-centre sites (bounding box): {len(hr)}')
for lab, cut in [('60 m', .06096), ('100 m', .09997)]:
    h = pd.Series(dh <= cut, index=sites); print(f'    within {lab}: {int(h.sum())} clinic sites matched')
    for nm, s in [('R expanded residual', X.R), ('S benchmark signal', X.S), ('gap', X.D), ('address (-L)', -X.L)]:
        tq = sites.isin(list(top(s))); a, b, c, d = int((tq & h).sum()), int((tq & ~h).sum()), int((~tq & h).sum()), int((~tq & ~h).sum())
        orr, p = fisher_exact([[a, b], [c, d]]); print(f'      {nm:<22} top quarter holds {a:>2} (chance {h.mean() * q:.1f}) | OR {orr:.2f}, Fisher p = {p:.3g}')
    for nm, cols in [('expected V (model C) + R', ['V_hat', 'R']), ('address L + R', ['L', 'R'])]:
        Z = sm.add_constant(pd.DataFrame({c: zs(X[c]) for c in cols}))
        m1 = sm.Logit(h.astype(int), Z).fit(disp=0); m0 = sm.Logit(h.astype(int), Z.drop(columns='R')).fit(disp=0)
        lr = 2 * (m1.llf - m0.llf); ci = np.exp(m1.conf_int().loc['R'])
        print(f'      logit HRSA ~ {nm:<26} OR per SD of R {np.exp(m1.params.R):.2f} ({ci[0]:.2f}-{ci[1]:.2f}), LR test p = {chi2.sf(lr, 1):.3g} | '
              f'OR per SD of {cols[0]} {np.exp(m1.params[cols[0]]):.2f}')
X['hrsa60'] = dh <= .06096

# ---------- 11. the clinics that differ most from expectation ----------
try:
    import geopandas as gpd
    cd = gpd.read_file('https://data.cityofnewyork.us/api/geospatial/5crt-au7u?method=export&format=GeoJSON')
    col = [c for c in cd.columns if c.lower().replace('_', '') == 'borocd'][0]
    cd = cd[cd[col].astype(int).between(101, 112)].assign(cd=lambda g: g[col].astype(int) - 100)[['cd', 'geometry']]
    pts = gpd.GeoDataFrame(index=sites, geometry=gpd.points_from_xy(slon, slat), crs='EPSG:4326').to_crs(cd.crs)
    X['cd'] = gpd.sjoin(pts, cd, how='left', predicate='within').groupby(level=0).cd.first().reindex(sites)
except Exception as e:
    print('community districts not loaded:', type(e).__name__); X['cd'] = np.nan
CDN = {1: 'Financial District/Tribeca', 2: 'Greenwich Village/SoHo', 3: 'Lower East Side/Chinatown', 4: 'Chelsea/Clinton', 5: 'Midtown',
       6: 'Stuyvesant Town/Turtle Bay', 7: 'Upper West Side', 8: 'Upper East Side', 9: 'Morningside Heights/Hamilton Heights',
       10: 'Central Harlem', 11: 'East Harlem', 12: 'Washington Heights/Inwood'}
X['district'] = X.cd.map(CDN)
try:
    nm = pd.read_parquet(f'{PAPER_DIR}/practice_addresses.parquet').groupby('_bld').location_name.apply(lambda s: '; '.join(s.astype(str).head(2)))
except Exception:
    nm = pd.Series(dtype=str)
X['names (for your checking only)'] = nm.reindex(sites)
cols = ['district', 'spec', 'n_pr', 'L', 'V', 'V_hat', 'R', 'z', 'hi_obs', 'hi_bench', 'hrsa60', 'names (for your checking only)']
with pd.option_context('display.width', 250, 'display.max_columns', 20, 'display.max_colwidth', 60, 'display.float_format', '{:.2f}'.format):
    print('\n[11] 15 clinics whose visitors are most MORE vulnerable than expected:'); print(X.sort_values('R', ascending=False)[cols].head(15).to_string())
    print('\n     10 clinics whose visitors are most LESS vulnerable than expected:'); print(X.sort_values('R')[cols].head(10).to_string())
tq = list(top(X.R)); print('\n     top quarter by R, by district (count of', q, '/ district total):')
print('    ', (X.loc[tq].district.value_counts().astype(str) + ' / ' + X.district.value_counts().reindex(X.loc[tq].district.value_counts().index).astype(str)).to_dict())

# ---------- 12. save and draw ----------
X.drop(columns=['names (for your checking only)']).to_parquet(f'{OUT_DIR}/expected_visitors_E1.parquet')
fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.6), gridspec_kw={'width_ratios': [1.1, 1]})
tR = X.index.isin(tq); bR = X.index.isin(list(set(X.R.nsmallest(q).index)))
ax[0].scatter(X.V_hat[~tR & ~bR], X.V[~tR & ~bR], s=6, c='0.75', lw=0, label='other sites')
ax[0].scatter(X.V_hat[tR], X.V[tR], s=8, c='#d62728', lw=0, label=f'top quarter by R ({q})')
ax[0].scatter(X.V_hat[bR], X.V[bR], s=8, c='#1f77b4', lw=0, label=f'bottom quarter by R ({q})')
lim = [min(X.V_hat.min(), X.V.min()) - .02, max(X.V_hat.max(), X.V.max()) + .02]; ax[0].plot(lim, lim, 'k:', lw=.8)
ax[0].set_xlabel('expected visitor-origin SVI (model C, out of fold)'); ax[0].set_ylabel('observed visitor-origin SVI, V')
ax[0].set_title(f'A  Observed vs expected (out-of-fold R$^2$ = {R2C:.2f})', loc='left', fontsize=10); ax[0].legend(fontsize=7, frameon=False)
order = [c for c in CLS if (X.spec == c).sum() >= 10]
ax[1].boxplot([X.R[X.spec == c] for c in order], showfliers=False); ax[1].axhline(0, c='k', lw=.6, ls=':')
ax[1].set_xticks(range(1, len(order) + 1)); ax[1].set_xticklabels([f'{c}\n(n = {(X.spec == c).sum()})' for c in order], fontsize=8)
ax[1].set_ylabel('R = observed minus expected'); ax[1].set_title('B  Unexpected part by specialty', loc='left', fontsize=10)
fig.tight_layout(); fig.savefig(f'{FIG}/F8_expected_vs_observed.png', dpi=300); plt.close()
print(f'\nsaved {OUT_DIR}/expected_visitors_E1.parquet and {FIG}/F8_expected_vs_observed.png')

In [ ]:
# ============ EXTERNAL CHECK WITH PATIENT DATA | Medicare patients of the physicians practising at each clinic site ============
# Source: CMS "Medicare Physician & Other Practitioners - by Provider", 2023 (public, provider-level, small counts suppressed).
# For each provider it gives the practice street address and how many of their Medicare patients are dual-eligible
# (also on Medicaid, i.e. low income), plus race/ethnicity counts and an average risk score.
# We match providers to clinic sites by street address and ask:
#   does the visitor-origin score V track the patients' disadvantage better than the address L does,
#   and does the unexpected part U add anything beyond the expectation V_hat?
import os, re, time, requests, numpy as np, pandas as pd, statsmodels.api as sm
from scipy.stats import spearmanr
if str(globals().get('DATA_DIR', 'data')).startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data'))
OUT = f'{PAPER_DIR}/R_series_results'
X = pd.read_parquet(f'{OUT}/expected_visitors_E1.parquet')
pa = pd.read_parquet(f'{PAPER_DIR}/practice_addresses.parquet')
rho = lambda a, b: spearmanr(a, b, nan_policy='omit').correlation
API = 'https://data.cms.gov/data-api/v1/dataset/8ba584c6-a43a-4b0b-a35a-eb9a59e3a571/data'   # data year 2023 (pinned; the paper's run used the 'latest' alias, which was 2023)

# ---------- 1. download Medicare providers practising in Manhattan ZIP codes ----------
cache = f'{OUT}/cms_providers_manhattan_2023.parquet'
if os.path.exists(cache):
    cms = pd.read_parquet(cache)
else:
    rows = []
    for z in range(10001, 10283):
        off = 0
        while True:
            for attempt in range(3):
                try:
                    r = requests.get(API, params={'filter[Rndrng_Prvdr_Zip5]': f'{z:05d}', 'size': 5000, 'offset': off}, timeout=60); r.raise_for_status(); page = r.json(); break
                except Exception as e:
                    page = None; time.sleep(2)
            if not page: break
            rows += page; off += len(page)
            if len(page) < 5000: break
    cms = pd.DataFrame(rows); cms.to_parquet(cache)
print(f'[1] Medicare providers with a Manhattan practice ZIP (2023): {len(cms):,}')

num = ['Tot_Benes', 'Bene_Dual_Cnt', 'Bene_Ndual_Cnt', 'Bene_Avg_Risk_Scre', 'Bene_Race_Wht_Cnt', 'Bene_Race_Black_Cnt', 'Bene_Race_API_Cnt',
       'Bene_Race_Hspnc_Cnt', 'Bene_Race_NatInd_Cnt', 'Bene_Race_Othr_Cnt']
for c in num: cms[c] = pd.to_numeric(cms[c], errors='coerce')
HOSPITAL_BASED = {'Diagnostic Radiology', 'Pathology', 'Anesthesiology', 'Emergency Medicine', 'Hospitalist', 'Critical Care (Intensivists)',
                  'Interventional Radiology', 'Nuclear Medicine', 'Clinical Laboratory', 'Ambulance Service Provider', 'Certified Registered Nurse Anesthetist (CRNA)',
                  'Anesthesiology Assistant', 'Independent Diagnostic Testing Facility (IDTF)', 'Portable X-Ray Supplier', 'Mass Immunizer Roster Biller'}
cms = cms[(cms.Tot_Benes >= 11) & ~cms.Rndrng_Prvdr_Type.isin(HOSPITAL_BASED)]
print(f'    after dropping hospital-based specialties and providers with fewer than 11 patients: {len(cms):,}')

# ---------- 2. match by normalised street address ----------
ABBR = {'street': 'st', 'str': 'st', 'avenue': 'ave', 'av': 'ave', 'boulevard': 'blvd', 'place': 'pl', 'road': 'rd', 'drive': 'dr', 'plaza': 'plz',
        'square': 'sq', 'terrace': 'ter', 'east': 'e', 'west': 'w', 'north': 'n', 'south': 's', 'first': '1st', 'second': '2nd', 'third': '3rd',
        'fourth': '4th', 'fifth': '5th', 'sixth': '6th', 'seventh': '7th', 'eighth': '8th', 'ninth': '9th', 'tenth': '10th', 'eleventh': '11th',
        'twelfth': '12th', 'saint': 'st', 'fort': 'ft', 'avenue of the americas': '6th ave'}
STOP = {'ste', 'suite', 'fl', 'floor', 'rm', 'room', 'unit', 'apt', 'bldg', 'building', 'level', 'lvl', 'frnt', 'front', 'ph', 'lbby', 'lobby'}
def akey(s):
    s = re.sub(r'avenue of the americas', '6th ave', str(s).lower())
    t = re.sub(r'[^a-z0-9 ]', ' ', s).split(); out = []
    for w in t:
        if w in STOP: break
        w = ABBR.get(w, w); w = re.sub(r'^(\d+)(st|nd|rd|th)$', r'\1', w); out.append(w)
    if not out or not out[0][0].isdigit(): return None
    out[0] = re.match(r'\d+', out[0]).group(0)
    return ' '.join(out[:4])
pa = pa[pa._bld.isin(X.index)].copy(); pa['k'] = pa.street_address.map(akey)
k2b = pa.dropna(subset=['k']).groupby('k')._bld.agg(lambda s: s.value_counts().index[0])       # address -> most common site
cms['k'] = cms.Rndrng_Prvdr_St1.map(akey); cms['_bld'] = cms.k.map(k2b)
m = cms.dropna(subset=['_bld'])
print(f'[2] providers matched to a clinic site by street address: {len(m):,} ({m._bld.nunique()} sites)')

# ---------- 3. site-level patient measures ----------
def site(g):
    ok = g.Bene_Dual_Cnt.notna() & g.Bene_Ndual_Cnt.notna()
    d, nd = g.Bene_Dual_Cnt[ok].sum(), g.Bene_Ndual_Cnt[ok].sum()
    race = g[['Bene_Race_Wht_Cnt', 'Bene_Race_Black_Cnt', 'Bene_Race_API_Cnt', 'Bene_Race_Hspnc_Cnt', 'Bene_Race_NatInd_Cnt', 'Bene_Race_Othr_Cnt']].sum(min_count=1)
    rk = g.dropna(subset=['Bene_Avg_Risk_Scre'])
    return pd.Series({'n_prov': len(g), 'benes': g.Tot_Benes.sum(), 'w': d + nd, 'dual': d / (d + nd) if d + nd > 0 else np.nan,
                      'black_hisp': (race.Bene_Race_Black_Cnt + race.Bene_Race_Hspnc_Cnt) / race.sum() if race.sum() > 0 else np.nan,
                      'risk': np.average(rk.Bene_Avg_Risk_Scre, weights=rk.Tot_Benes) if len(rk) else np.nan})
P = m.groupby('_bld').apply(site)
D = X.join(P, how='inner'); D = D[D.w >= 50].copy()
north = D.lat > 40.787
print(f'[3] sites with at least 50 Medicare patients of known dual status: {len(D)} of {len(X)} '
      f'(median L {D.L.median():.2f} vs {X.L.median():.2f} for all sites; north of 96th St {north.mean():.0%} vs {(X.lat > 40.787).mean():.0%})')
print(f'    dual-eligible share across these sites: median {D.dual.median():.2f}, IQR {D.dual.quantile(.25):.2f}-{D.dual.quantile(.75):.2f}\n')

# ---------- 4. which score tracks the patients? ----------
print('[4] rank correlation of each site score with the Medicare patients of the site')
print(f'    {"score":<34}{"dual-eligible share":>20}{"Black or Hispanic":>20}{"risk score":>12}')
for lab, s in [('address tract SVI, L', D.L), ('visitor-origin SVI, V', D.V), ('expected V (full expectation)', D.V_hat),
               ('resident-based score, V_pop', D.V_pop), ('gap, V - L', D.D), ('distance-only residual, S', D.S), ('unexpected part, U', D.R)]:
    print(f'    {lab:<34}{rho(s, D.dual):>20.2f}{rho(s, D.black_hisp):>20.2f}{rho(s, D.risk):>12.2f}')

# ---------- 5. does V add to the address, and does U add to the expectation? (weighted, block bootstrap) ----------
x_km = (D.lon + 73.979192) * 111.32 * np.cos(np.radians(40.754834)); y_km = (D.lat - 40.754834) * 110.57
blk = pd.factorize(np.floor(x_km).astype(int).astype(str) + '_' + np.floor(y_km).astype(int).astype(str))[0]
z = lambda s: (s - s.mean()) / s.std()
def wls(y, cols, idx=None):
    d = D if idx is None else D.iloc[idx]
    A = np.column_stack([np.ones(len(d))] + [z(D[c]).iloc[idx].values if idx is not None else z(D[c]).values for c in cols])
    w = np.sqrt(d.w.values); b = np.linalg.lstsq(A * w[:, None], d[y].values * w, rcond=None)[0]
    res = d[y].values - A @ b; r2 = 1 - np.sum(d.w.values * res ** 2) / np.sum(d.w.values * (d[y].values - np.average(d[y].values, weights=d.w.values)) ** 2)
    return b[1:], r2
rng = np.random.default_rng(0); ub = np.unique(blk); by = {g: np.where(blk == g)[0] for g in ub}
def boot(y, cols, B=1000):
    out = []
    for _ in range(B):
        i = np.concatenate([by[g] for g in rng.choice(ub, len(ub), replace=True)]); out.append(wls(y, cols, i)[0])
    return np.percentile(np.array(out), [2.5, 97.5], axis=0)
print('\n[5] weighted regressions of the dual-eligible share (coefficients per SD; 95% block-bootstrap intervals)')
for cols in [['L'], ['V'], ['L', 'V'], ['V_hat'], ['V_hat', 'R'], ['L', 'R']]:
    b, r2 = wls('dual', cols); ci = boot('dual', cols)
    terms = ', '.join(f'{c} {b[j]:+.3f} ({ci[0, j]:+.3f} to {ci[1, j]:+.3f})' for j, c in enumerate(cols))
    print(f'    dual ~ {" + ".join(cols):<12} R2 {r2:.2f} | {terms}')

# ---------- 6. shortlists: are the patients of top-quarter sites more disadvantaged? ----------
q = int(len(X) * .25)
print('\n[6] dual-eligible share of patients (weighted) at matched sites in each top quarter vs the rest')
for lab, s in [('address (-L)', -X.L), ('gap', X.D), ('distance-only residual S', X.S), ('unexpected part U', X.R)]:
    t = D.index.isin(s.nlargest(q).index)
    a, b = np.average(D.dual[t], weights=D.w[t]), np.average(D.dual[~t], weights=D.w[~t])
    print(f'    {lab:<26} top quarter {a:.3f} ({t.sum()} sites) vs rest {b:.3f} ({(~t).sum()} sites)')
D.to_parquet(f'{OUT}/medicare_site_validation.parquet')
print(f'\nsaved {OUT}/medicare_site_validation.parquet')

In [ ]:
# ============ MEDICARE CHECK, ROBUSTNESS | is the patient-based result solid enough to headline? ============
# Uses the files saved by the Medicare cell (no new downloads). About 1-2 minutes.
import os, re, numpy as np, pandas as pd
from scipy.stats import spearmanr
if str(globals().get('DATA_DIR', 'data')).startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data')); OUT = f'{PAPER_DIR}/R_series_results'
X = pd.read_parquet(f'{OUT}/expected_visitors_E1.parquet'); X['V_hatB'] = X.V - X.R_B          # public-information model, no benchmark
cms0 = pd.read_parquet(f'{OUT}/cms_providers_manhattan_2023.parquet')
pa = pd.read_parquet(f'{PAPER_DIR}/practice_addresses.parquet')
rho = lambda a, b: spearmanr(a, b, nan_policy='omit').correlation

num = ['Tot_Benes', 'Bene_Dual_Cnt', 'Bene_Ndual_Cnt', 'Bene_Race_Wht_Cnt', 'Bene_Race_Black_Cnt', 'Bene_Race_API_Cnt', 'Bene_Race_Hspnc_Cnt', 'Bene_Race_NatInd_Cnt', 'Bene_Race_Othr_Cnt']
for c in num: cms0[c] = pd.to_numeric(cms0[c], errors='coerce')
HOSPITAL_BASED = {'Diagnostic Radiology', 'Pathology', 'Anesthesiology', 'Emergency Medicine', 'Hospitalist', 'Critical Care (Intensivists)', 'Interventional Radiology',
                  'Nuclear Medicine', 'Clinical Laboratory', 'Ambulance Service Provider', 'Certified Registered Nurse Anesthetist (CRNA)', 'Anesthesiology Assistant',
                  'Independent Diagnostic Testing Facility (IDTF)', 'Portable X-Ray Supplier', 'Mass Immunizer Roster Biller'}
PRIMARY = {'Internal Medicine', 'Family Practice', 'General Practice', 'Geriatric Medicine', 'Nurse Practitioner', 'Physician Assistant', 'Obstetrics & Gynecology', 'Preventive Medicine'}
ABBR = {'street': 'st', 'str': 'st', 'avenue': 'ave', 'av': 'ave', 'boulevard': 'blvd', 'place': 'pl', 'road': 'rd', 'drive': 'dr', 'plaza': 'plz', 'square': 'sq', 'terrace': 'ter',
        'east': 'e', 'west': 'w', 'north': 'n', 'south': 's', 'first': '1st', 'second': '2nd', 'third': '3rd', 'fourth': '4th', 'fifth': '5th', 'sixth': '6th', 'seventh': '7th',
        'eighth': '8th', 'ninth': '9th', 'tenth': '10th', 'eleventh': '11th', 'twelfth': '12th', 'saint': 'st', 'fort': 'ft'}
STOP = {'ste', 'suite', 'fl', 'floor', 'rm', 'room', 'unit', 'apt', 'bldg', 'building', 'level', 'lvl', 'frnt', 'front', 'ph', 'lbby', 'lobby'}
def akey(s):
    s = re.sub(r'avenue of the americas', '6th ave', str(s).lower()); t = re.sub(r'[^a-z0-9 ]', ' ', s).split(); out = []
    for w in t:
        if w in STOP: break
        w = ABBR.get(w, w); w = re.sub(r'^(\d+)(st|nd|rd|th)$', r'\1', w); out.append(w)
    if not out or not out[0][0].isdigit(): return None
    out[0] = re.match(r'\d+', out[0]).group(0); return ' '.join(out[:4])
pa = pa[pa._bld.isin(X.index)].copy(); pa['k'] = pa.street_address.map(akey)
k2b = pa.dropna(subset=['k']).groupby('k')._bld.agg(lambda s: s.value_counts().index[0])
cms0['_bld'] = cms0.Rndrng_Prvdr_St1.map(akey).map(k2b)

def build(mask, min_w=50):
    m = cms0[mask & cms0._bld.notna()]
    ok = m.Bene_Dual_Cnt.notna() & m.Bene_Ndual_Cnt.notna(); m = m[ok]
    g = m.groupby('_bld')[['Bene_Dual_Cnt', 'Bene_Ndual_Cnt', 'Bene_Race_Black_Cnt', 'Bene_Race_Hspnc_Cnt', 'Bene_Race_Wht_Cnt', 'Bene_Race_API_Cnt', 'Bene_Race_NatInd_Cnt', 'Bene_Race_Othr_Cnt']].sum(min_count=1)
    g['w'] = g.Bene_Dual_Cnt + g.Bene_Ndual_Cnt; g['dual'] = g.Bene_Dual_Cnt / g.w
    race = g[['Bene_Race_Black_Cnt', 'Bene_Race_Hspnc_Cnt', 'Bene_Race_Wht_Cnt', 'Bene_Race_API_Cnt', 'Bene_Race_NatInd_Cnt', 'Bene_Race_Othr_Cnt']].sum(axis=1)
    g['black_hisp'] = (g.Bene_Race_Black_Cnt.fillna(0) + g.Bene_Race_Hspnc_Cnt.fillna(0)) / race.replace(0, np.nan)
    D = X.join(g[['w', 'dual', 'black_hisp']], how='inner'); return D[D.w >= min_w].copy()

base = (cms0.Tot_Benes >= 11) & ~cms0.Rndrng_Prvdr_Type.isin(HOSPITAL_BASED)
D = build(base)
z = lambda s: (s - s.mean()) / s.std()
def blocks(D):
    x = (D.lon + 73.979192) * 111.32 * np.cos(np.radians(40.754834)); y = (D.lat - 40.754834) * 110.57
    return pd.factorize(np.floor(x).astype(int).astype(str) + '_' + np.floor(y).astype(int).astype(str))[0]
def wls(D, y, cols):
    A = np.column_stack([np.ones(len(D))] + [z(D[c]).values for c in cols]); w = D.w.values
    b = np.linalg.lstsq(A * np.sqrt(w)[:, None], D[y].values * np.sqrt(w), rcond=None)[0]; res = D[y].values - A @ b
    return b[1:], 1 - np.sum(w * res ** 2) / np.sum(w * (D[y].values - np.average(D[y].values, weights=w)) ** 2)
def boot(D, fn, B=1000, seed=0):
    blk = blocks(D); ub = np.unique(blk); by = {g: np.where(blk == g)[0] for g in ub}; rng = np.random.default_rng(seed); out = []
    for _ in range(B):
        i = np.concatenate([by[g] for g in rng.choice(ub, len(ub), replace=True)]); out.append(fn(D.iloc[i]))
    return np.percentile(np.array(out), [2.5, 97.5], axis=0)

print(f'[1] main set: {len(D)} sites | rank correlation with the dual-eligible share (95% block-bootstrap interval)')
for c, lab in [('L', 'address tract SVI'), ('V', 'visitor-origin SVI'), ('V_hat', 'full expectation'), ('V_hatB', 'public-information model'), ('V_pop', 'resident score'), ('R', 'unexpected part U')]:
    ci = boot(D, lambda d, c=c: rho(d[c], d.dual))
    print(f'    {lab:<26} {rho(D[c], D.dual):+.2f} ({ci[0]:+.2f} to {ci[1]:+.2f})')
print('\n[2] weighted differences in explained variance (95% block-bootstrap interval)')
for lab, f in [('V alone minus L alone', lambda d: wls(d, 'dual', ['V'])[1] - wls(d, 'dual', ['L'])[1]),
               ('L + V minus L', lambda d: wls(d, 'dual', ['L', 'V'])[1] - wls(d, 'dual', ['L'])[1]),
               ('public-information model minus L', lambda d: wls(d, 'dual', ['V_hatB'])[1] - wls(d, 'dual', ['L'])[1]),
               ('full expectation + U minus full expectation', lambda d: wls(d, 'dual', ['V_hat', 'R'])[1] - wls(d, 'dual', ['V_hat'])[1])]:
    ci = boot(D, f); print(f'    {lab:<46} {f(D):+.3f} ({ci[0]:+.3f} to {ci[1]:+.3f})')
for cols in [['V_hatB'], ['L', 'V_hatB'], ['V_hatB', 'R_B']]:
    b, r2 = wls(D, 'dual', cols); ci = boot(D, lambda d, cols=cols: wls(d, 'dual', cols)[0])
    print(f'    dual ~ {" + ".join(cols):<16} R2 {r2:.2f} | ' + ', '.join(f'{c} {b[j]:+.3f} ({ci[0, j]:+.3f} to {ci[1, j]:+.3f})' for j, c in enumerate(cols)))

print('\n[3] sensitivity: rank correlations with the dual-eligible share, and the joint weighted model L + V')
subsets = [('main (all office-based, >= 50 patients)', base, 50), ('at least 100 patients', base, 100), ('at least 200 patients', base, 200),
           ('primary care providers only', base & cms0.Rndrng_Prvdr_Type.isin(PRIMARY), 50)]
if 'Rndrng_Prvdr_Ent_Cd' in cms0.columns: subsets.append(('individual providers only', base & (cms0.Rndrng_Prvdr_Ent_Cd == 'I'), 50))
for lab, mask, mw in subsets:
    d = build(mask, mw)
    if len(d) < 30: print(f'    {lab:<42} only {len(d)} sites'); continue
    b, r2 = wls(d, 'dual', ['L', 'V'])
    print(f'    {lab:<42} n {len(d):>3} | rho L {rho(d.L, d.dual):+.2f}, V {rho(d.V, d.dual):+.2f}, V_hat {rho(d.V_hat, d.dual):+.2f}, U {rho(d.R, d.dual):+.2f} | L + V: L {b[0]:+.3f}, V {b[1]:+.3f}')

print('\n[4] second outcome: share of Medicare patients who are Black or Hispanic (main set)')
for cols in [['L'], ['V'], ['L', 'V'], ['V_hat', 'R']]:
    dd = D.dropna(subset=['black_hisp']); b, r2 = wls(dd, 'black_hisp', cols); ci = boot(dd, lambda d, cols=cols: wls(d, 'black_hisp', cols)[0])
    print(f'    black_hisp ~ {" + ".join(cols):<10} R2 {r2:.2f} | ' + ', '.join(f'{c} {b[j]:+.3f} ({ci[0, j]:+.3f} to {ci[1, j]:+.3f})' for j, c in enumerate(cols)))

In [ ]:
# ============ HEADLINE CHECK | intervals for "the gap ranks clinics the wrong way" ============
# Uses only files saved by the Medicare cell (no downloads). About 1 minute.
# Prints: [1] rank correlations with 95% block-bootstrap intervals, [2] shortlist differences with intervals,
#         [3] the weighted model of dual eligibility on the gap, [4] the R2 missing from Table 9 (L + public-information model),
#         [5] the same patient check on the count-weighted later-week (E2) sites.
import numpy as np, pandas as pd
from scipy.stats import spearmanr
if str(globals().get('DATA_DIR', 'data')).startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data')); OUT = f'{PAPER_DIR}/R_series_results'
X = pd.read_parquet(f'{OUT}/expected_visitors_E1.parquet')                 # all 1,180 E1 sites
M = pd.read_parquet(f'{OUT}/medicare_site_validation.parquet')             # 348 matched sites (gap is column D)
M['V_hatB'] = M.V - M.R_B
rho = lambda a, b: spearmanr(a, b, nan_policy='omit').correlation
z = lambda s: (s - s.mean()) / s.std()

x_km = (M.lon + 73.979192) * 111.32 * np.cos(np.radians(40.754834)); y_km = (M.lat - 40.754834) * 110.57
blk = pd.factorize(np.floor(x_km).astype(int).astype(str) + '_' + np.floor(y_km).astype(int).astype(str))[0]
ub = np.unique(blk); by = {g: np.where(blk == g)[0] for g in ub}
def boot(fn, B=1000, seed=0):
    rng = np.random.default_rng(seed); out = []
    for _ in range(B):
        i = np.concatenate([by[g] for g in rng.choice(ub, len(ub), replace=True)]); out.append(fn(M.iloc[i]))
    return np.nanpercentile(np.array(out, dtype=float), [2.5, 97.5], axis=0)
def wls(d, y, cols):
    d = d.dropna(subset=[y]); A = np.column_stack([np.ones(len(d))] + [z(d[c]).values for c in cols]); w = d.w.values
    b = np.linalg.lstsq(A * np.sqrt(w)[:, None], d[y].values * np.sqrt(w), rcond=None)[0]; res = d[y].values - A @ b
    return b[1:], 1 - np.sum(w * res ** 2) / np.sum(w * (d[y].values - np.average(d[y].values, weights=w)) ** 2)

print(f'[1] rank correlations with the site\'s Medicare patients, {len(M)} sites (95% block-bootstrap interval)')
for c, lab in [('D', 'gap, V - L'), ('S', 'distance-only residual S'), ('L', 'address tract SVI L'), ('V', 'visitor-origin SVI V')]:
    cells = []
    for y in ['dual', 'black_hisp', 'risk']:
        ci = boot(lambda d, c=c, y=y: rho(d[c], d[y])); cells.append(f'{y} {rho(M[c], M[y]):+.2f} ({ci[0]:+.2f} to {ci[1]:+.2f})')
    print(f'    {lab:<26} ' + ' | '.join(cells))

q = int(len(X) * .25)
print('\n[2] weighted dual-eligible share, matched sites in each top quarter minus the other matched sites (95% interval)')
for lab, s in [('address (-L)', -X.L), ('gap', X.D), ('distance-only residual S', X.S), ('unpredicted part U', X.R)]:
    top = set(s.nlargest(q).index)
    def diff(d, top=top):
        t = d.index.isin(top)
        if t.sum() == 0 or (~t).sum() == 0: return np.nan
        return np.average(d.dual[t], weights=d.w[t]) - np.average(d.dual[~t], weights=d.w[~t])
    ci = boot(diff); t = M.index.isin(top)
    print(f'    {lab:<26} {t.sum():>3} sites | difference {diff(M):+.3f} ({ci[0]:+.3f} to {ci[1]:+.3f})')

print('\n[3] weighted least squares of the dual-eligible share on the gap (coefficient per SD, 95% interval)')
b, r2 = wls(M, 'dual', ['D']); ci = boot(lambda d: wls(d, 'dual', ['D'])[0])
print(f'    dual ~ gap: R2 {r2:.2f} | gap {b[0]:+.3f} ({ci[0, 0]:+.3f} to {ci[1, 0]:+.3f})')

print('\n[4] Table 9 missing cell: dual ~ L + public-information model')
b, r2 = wls(M, 'dual', ['L', 'V_hatB']); ci = boot(lambda d: wls(d, 'dual', ['L', 'V_hatB'])[0])
print(f'    R2 {r2:.2f} | L {b[0]:+.3f} ({ci[0, 0]:+.3f} to {ci[1, 0]:+.3f}), public-information model {b[1]:+.3f} ({ci[0, 1]:+.3f} to {ci[1, 1]:+.3f})')


# ---------- [5] count-weighted counterpart of the patient check: the E2 sites (weeks 4-48) ----------
# Same provider rules as the Medicare cell; addresses are matched to the practices of the E2 sites.
import re
E2 = pd.read_parquet(f'{PAPER_DIR}/site_table_E2.parquet')
if '_bld' in E2.columns: E2 = E2.set_index('_bld')
E2 = E2[E2.L.notna() & E2.V.notna()]
cms = pd.read_parquet(f'{OUT}/cms_providers_manhattan_2023.parquet')
for c in ['Tot_Benes', 'Bene_Dual_Cnt', 'Bene_Ndual_Cnt']: cms[c] = pd.to_numeric(cms[c], errors='coerce')
HOSPITAL_BASED = {'Diagnostic Radiology', 'Pathology', 'Anesthesiology', 'Emergency Medicine', 'Hospitalist', 'Critical Care (Intensivists)', 'Interventional Radiology',
                  'Nuclear Medicine', 'Clinical Laboratory', 'Ambulance Service Provider', 'Certified Registered Nurse Anesthetist (CRNA)', 'Anesthesiology Assistant',
                  'Independent Diagnostic Testing Facility (IDTF)', 'Portable X-Ray Supplier', 'Mass Immunizer Roster Biller'}
ABBR = {'street': 'st', 'str': 'st', 'avenue': 'ave', 'av': 'ave', 'boulevard': 'blvd', 'place': 'pl', 'road': 'rd', 'drive': 'dr', 'plaza': 'plz', 'square': 'sq', 'terrace': 'ter',
        'east': 'e', 'west': 'w', 'north': 'n', 'south': 's', 'first': '1st', 'second': '2nd', 'third': '3rd', 'fourth': '4th', 'fifth': '5th', 'sixth': '6th', 'seventh': '7th',
        'eighth': '8th', 'ninth': '9th', 'tenth': '10th', 'eleventh': '11th', 'twelfth': '12th', 'saint': 'st', 'fort': 'ft'}
STOP = {'ste', 'suite', 'fl', 'floor', 'rm', 'room', 'unit', 'apt', 'bldg', 'building', 'level', 'lvl', 'frnt', 'front', 'ph', 'lbby', 'lobby'}
def akey(s):
    s = re.sub(r'avenue of the americas', '6th ave', str(s).lower()); t = re.sub(r'[^a-z0-9 ]', ' ', s).split(); out = []
    for w in t:
        if w in STOP: break
        w = ABBR.get(w, w); w = re.sub(r'^(\d+)(st|nd|rd|th)$', r'\1', w); out.append(w)
    if not out or not out[0][0].isdigit(): return None
    out[0] = re.match(r'\d+', out[0]).group(0); return ' '.join(out[:4])
pa = pd.read_parquet(f'{PAPER_DIR}/practice_addresses.parquet'); pa = pa[pa._bld.isin(E2.index)].copy(); pa['k'] = pa.street_address.map(akey)
k2b = pa.dropna(subset=['k']).groupby('k')._bld.agg(lambda s: s.value_counts().index[0])
c2 = cms[(cms.Tot_Benes >= 11) & ~cms.Rndrng_Prvdr_Type.isin(HOSPITAL_BASED)].copy(); c2['_bld'] = c2.Rndrng_Prvdr_St1.map(akey).map(k2b)
c2 = c2[c2._bld.notna() & c2.Bene_Dual_Cnt.notna() & c2.Bene_Ndual_Cnt.notna()]
g = c2.groupby('_bld')[['Bene_Dual_Cnt', 'Bene_Ndual_Cnt']].sum(); g['w'] = g.Bene_Dual_Cnt + g.Bene_Ndual_Cnt; g['dual'] = g.Bene_Dual_Cnt / g.w
E = E2[['L', 'V']].join(g[['w', 'dual']], how='inner'); E = E[E.w >= 50].copy(); E['gap'] = E.V - E.L
print(f'\n[5] E2 (count-weighted) sites with at least 50 Medicare patients of known dual status: {len(E)} of {len(E2)}')
if len(E) >= 20:
    lat = X.lat.reindex(E.index); lon = X.lon.reindex(E.index)
    if lat.isna().any() and 'lat' in E2.columns: lat = lat.fillna(E2.lat.reindex(E.index)); lon = lon.fillna(E2.lon.reindex(E.index))
    ok = lat.notna() & lon.notna(); E, lat, lon = E[ok], lat[ok], lon[ok]
    bk = pd.factorize(np.floor((lon + 73.979192) * 111.32 * np.cos(np.radians(40.754834))).astype(int).astype(str) + '_' + np.floor((lat - 40.754834) * 110.57).astype(int).astype(str))[0]
    ub2 = np.unique(bk); by2 = {k: np.where(bk == k)[0] for k in ub2}; rng = np.random.default_rng(0)
    for c, lab in [('L', 'address tract SVI L'), ('V', 'visitor-origin SVI V (E2)'), ('gap', 'gap V - L (E2)')]:
        bs = [rho(E.iloc[i][c], E.iloc[i].dual) for i in (np.concatenate([by2[k] for k in rng.choice(ub2, len(ub2), replace=True)]) for _ in range(1000))]
        lo, hi = np.nanpercentile(bs, [2.5, 97.5]); print(f'    {lab:<28} rank correlation with dual share {rho(E[c], E.dual):+.2f} ({lo:+.2f} to {hi:+.2f}), {len(E)} sites')
    sh = E.index.intersection(M.index)
    if len(sh) >= 10:
        print(f'    on the {len(sh)} sites also in the E1 patient check: gap E2 {rho(E.gap[sh], E.dual[sh]):+.2f} vs gap E1 {rho(M.D[sh], M.dual[sh]):+.2f}; '
              f'V E2 {rho(E.V[sh], E.dual[sh]):+.2f} vs V E1 {rho(M.V[sh], M.dual[sh]):+.2f}')
else:
    print('    too few sites for a count-weighted patient check')

In [ ]:
# ============ E1 vs E2, SAME PATIENTS | paired comparison of the January and later-week visitor scores ============
# On the sites in both the January (E1) and the later-week (E2) patient checks, bootstraps the difference
# rho(V_E2, dual) - rho(V_E1, dual), using the same patients (the E1 site match) for both scores. Uses saved files only.
import re, numpy as np, pandas as pd
from scipy.stats import spearmanr
if str(globals().get('DATA_DIR', 'data')).startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data')); OUT = f'{PAPER_DIR}/R_series_results'
rho = lambda a, b: spearmanr(a, b, nan_policy='omit').correlation
M = pd.read_parquet(f'{OUT}/medicare_site_validation.parquet')                      # E1 patient check, 348 sites
E2 = pd.read_parquet(f'{PAPER_DIR}/site_table_E2.parquet')
if '_bld' in E2.columns: E2 = E2.set_index('_bld')
E2 = E2[E2.L.notna() & E2.V.notna()]
P = M[['L', 'V', 'dual', 'w', 'lat', 'lon']].join(E2[['V']].rename(columns={'V': 'V2'}), how='inner')
P = P[P.w >= 50]
print(f'sites in both schemes with at least 50 Medicare patients (E1 match): {len(P)}')
bk = pd.factorize(np.floor((P.lon + 73.979192) * 111.32 * np.cos(np.radians(40.754834))).astype(int).astype(str) + '_' + np.floor((P.lat - 40.754834) * 110.57).astype(int).astype(str))[0]
ub = np.unique(bk); by = {k: np.where(bk == k)[0] for k in ub}; rng = np.random.default_rng(0)
stats = lambda d: (rho(d.L, d.dual), rho(d.V, d.dual), rho(d.V2, d.dual), rho(d.V2, d.dual) - rho(d.V, d.dual))
B = np.array([stats(P.iloc[np.concatenate([by[k] for k in rng.choice(ub, len(ub), replace=True)])]) for _ in range(2000)])
lo, hi = np.nanpercentile(B, [2.5, 97.5], axis=0); est = stats(P)
for j, lab in enumerate(['address tract SVI L', 'January visitor score V (E1)', 'count-weighted visitor score V (E2)', 'difference E2 minus E1']):
    print(f'    {lab:<38} {est[j]:+.2f} ({lo[j]:+.2f} to {hi[j]:+.2f})')
print(f'    blocks: {len(ub)}; share of bootstrap draws with E2 > E1: {np.nanmean(B[:, 3] > 0):.2f}')

In [ ]:
# ============ TABLE 7, PANEL A | rank correlations with 95% block-bootstrap intervals for every score and outcome ============
# Uses only the file saved by the Medicare cell (no downloads). About 2-3 minutes.
import numpy as np, pandas as pd
from scipy.stats import spearmanr
if str(globals().get('DATA_DIR', 'data')).startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data')); OUT = f'{PAPER_DIR}/R_series_results'
M = pd.read_parquet(f'{OUT}/medicare_site_validation.parquet')            # 348 matched sites
M['V_hatB'] = M.V - M.R_B                                                  # public-information model
rho = lambda a, b: spearmanr(a, b, nan_policy='omit').correlation
x_km = (M.lon + 73.979192) * 111.32 * np.cos(np.radians(40.754834)); y_km = (M.lat - 40.754834) * 110.57
blk = pd.factorize(np.floor(x_km).astype(int).astype(str) + '_' + np.floor(y_km).astype(int).astype(str))[0]
ub = np.unique(blk); by = {g: np.where(blk == g)[0] for g in ub}
scores = [('L', 'Address tract SVI, L'), ('V', 'Visitor-origin SVI, V'), ('V_hat', 'Full expectation'), ('V_hatB', 'Public-information model'),
          ('V_pop', 'Resident-based score'), ('D', 'Gap'), ('S', 'Distance-only residual S'), ('R', 'Unpredicted part U')]
outcomes = ['dual', 'black_hisp', 'risk']
def stats(d): return np.array([[rho(d[s], d[y]) for y in outcomes] for s, _ in scores])
rng = np.random.default_rng(0)
B = np.array([stats(M.iloc[np.concatenate([by[g] for g in rng.choice(ub, len(ub), replace=True)])]) for _ in range(1000)])
lo, hi = np.nanpercentile(B, 2.5, axis=0), np.nanpercentile(B, 97.5, axis=0); est = stats(M)
print(f'{len(M)} sites, {len(ub)} blocks | rank correlation (95% block-bootstrap interval)')
print(f'{"score":<28}' + ''.join(f'{y:>26}' for y in outcomes))
for i, (_, lab) in enumerate(scores):
    print(f'{lab:<28}' + ''.join(f'{est[i, j]:>+10.2f} ({lo[i, j]:+.2f} to {hi[i, j]:+.2f})' for j in range(3)))

In [ ]:
# ============ FIGURE F8 | observed vs expected visitor-origin vulnerability, and U by specialty ============
# Needs R_series_results/expected_visitors_E1.parquet from the EXPECTED VISITORS cell (column R is the paper's U).
# F7 (four shortlist maps) is no longer used in the paper from v68 on.
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
if str(globals().get('DATA_DIR', 'data')).startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data'))
FIG = f'{PAPER_DIR}/R_series_results/figures'; os.makedirs(FIG, exist_ok=True)
X = pd.read_parquet(f'{PAPER_DIR}/R_series_results/expected_visitors_E1.parquet')
q = int(len(X) * .25); tU = X.index.isin(X.R.nlargest(q).index); bU = X.index.isin(X.R.nsmallest(q).index)
r2 = 1 - ((X.V - X.V_hat) ** 2).sum() / ((X.V - X.V.mean()) ** 2).sum()
fig, ax = plt.subplots(1, 2, figsize=(10.5, 4.6), gridspec_kw={'width_ratios': [1.1, 1]})
ax[0].scatter(X.V_hat[~tU & ~bU], X.V[~tU & ~bU], s=6, c='0.75', lw=0, label='other sites')
ax[0].scatter(X.V_hat[tU], X.V[tU], s=8, c='#c0392b', lw=0, label=f'top quarter by $U$ ({q})')
ax[0].scatter(X.V_hat[bU], X.V[bU], s=8, c='#1f77b4', lw=0, label=f'bottom quarter by $U$ ({q})')
lim = [min(X.V_hat.min(), X.V.min()) - .02, max(X.V_hat.max(), X.V.max()) + .02]; ax[0].plot(lim, lim, 'k:', lw=.8)
ax[0].set_xlabel(r'expected visitor-origin SVI, $\hat{V}$ (out of fold)'); ax[0].set_ylabel('observed visitor-origin SVI, $V$')
ax[0].set_title(f'A  Observed vs expected (out-of-fold $R^2$ = {r2:.2f})', loc='left', fontsize=10); ax[0].legend(fontsize=7, frameon=False)
order = [c for c in ['Physician', 'Mental health', 'Other practitioner', 'Outpatient centre'] if (X.spec == c).sum() >= 10]
ax[1].boxplot([X.R[X.spec == c] for c in order], showfliers=False); ax[1].axhline(0, c='k', lw=.6, ls=':')
ax[1].set_xticks(range(1, len(order) + 1)); ax[1].set_xticklabels([f'{c}\n(n = {(X.spec == c).sum()})' for c in order], fontsize=8)
ax[1].set_ylabel('$U = V - \\hat{V}$'); ax[1].set_title('B  Unpredicted part by dominant specialty', loc='left', fontsize=10)
fig.tight_layout(); fig.savefig(f'{FIG}/F8_expected_vs_observed.png', dpi=300); plt.close()
print(f'saved {FIG}/F8_expected_vs_observed.png (out-of-fold R2 {r2:.2f}; should be 0.36)')

In [ ]:
# ============ FIGURE 4 (file F5_map_delta_E1.png) | geography of the discrepancy, redrawn at print size ============
# Three panels: (A) gap at each of the 1,180 E1 sites, (B) median gap by community district, (C) hidden-high sites on tract SVI.
# Drawn at 7.2 x 4.6 inches with 8-9 pt text, so labels stay readable at journal page width.
# Needs: R_series_results/expected_visitors_E1.parquet and SVI_US_2022.csv in PAPER_DIR; downloads tract and district boundaries.
import os, re, numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
if str(globals().get('DATA_DIR', 'data')).startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data'))
FIG = f'{PAPER_DIR}/R_series_results/figures'; os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({'font.size': 8, 'axes.titlesize': 9, 'figure.dpi': 100})

X = pd.read_parquet(f'{PAPER_DIR}/R_series_results/expected_visitors_E1.parquet')
X['gap'] = X.V - X.L
hh = (X.L < X.L.median()) & (X.V >= X.V.median())                       # hidden-high: below-median L, at-or-above-median V

# ---- boundaries: Manhattan tracts with 2022 SVI, and the 12 community districts ----
tr = gpd.read_file('https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_36_tract.zip')
tr = tr[tr.COUNTYFP == '061'].to_crs('EPSG:2263')
svi = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False, usecols=['FIPS', 'RPL_THEMES'], dtype={'FIPS': str})
svi['FIPS'] = svi.FIPS.str.zfill(11)
tr = tr.merge(svi, left_on='GEOID', right_on='FIPS', how='left'); tr.loc[tr.RPL_THEMES < 0, 'RPL_THEMES'] = np.nan
cd = None
for src in [f'{PAPER_DIR}/community_districts.geojson',                     # a local copy, if you have one
            'https://data.cityofnewyork.us/api/geospatial/5crt-au7u?method=export&format=GeoJSON',
            'https://data.cityofnewyork.us/resource/5crt-au7u.geojson?$limit=100']:
    if src.startswith('/') and not os.path.exists(src): continue
    try:
        cd = gpd.read_file(src); break
    except Exception as e:
        print(f'community districts not read from {src[:60]}...: {e}')
if cd is None:
    raise RuntimeError('Community district boundaries could not be read. Download "Community Districts" (dataset 5crt-au7u) '
                       'from NYC Open Data as GeoJSON and save it as community_districts.geojson in PAPER_DIR, then rerun.')
col = [c for c in cd.columns if c.lower().replace('_', '') == 'borocd'][0]
cd['cdn'] = cd[col].astype(int); cd = cd[cd.cdn.between(101, 112)].to_crs('EPSG:2263'); cd['cdn'] -= 100
pts = gpd.GeoDataFrame(X, geometry=gpd.points_from_xy(X.lon, X.lat), crs='EPSG:4326').to_crs('EPSG:2263')

# ---- district of each site: the saved district column if it gives 1-12, otherwise a spatial join ----
def cd_number(v):
    m = re.search(r'\d+', str(v))
    if not m: return np.nan
    n = int(m.group(0)); n = n - 100 if 100 < n < 200 else n
    return n if 1 <= n <= 12 else np.nan
used = 'spatial join'
if 'district' in X.columns:
    dn = X.district.map(cd_number)
    if dn.notna().mean() > 0.95: pts['cdn'] = dn.values; used = 'saved district column'
if used == 'spatial join':
    pts = gpd.sjoin(pts, cd[['cdn', 'geometry']], how='left', predicate='within').drop(columns='index_right')
med = pts.groupby('cdn').gap.median()
cdm = cd.merge(med.rename('med_gap'), left_on='cdn', right_index=True, how='left')

# ---- figure ----
fig, ax = plt.subplots(1, 3, figsize=(7.2, 4.6))
lim_a = 0.5; lim_b = float(np.ceil(med.abs().max() * 20) / 20)
for a in ax:
    a.set_axis_off(); a.set_aspect('equal')
# (A) gap at each site
tr.plot(ax=ax[0], color='0.93', edgecolor='white', linewidth=.2); cd.boundary.plot(ax=ax[0], color='0.45', linewidth=.4)
order = pts.gap.abs().sort_values().index
sc = ax[0].scatter(pts.loc[order].geometry.x, pts.loc[order].geometry.y, c=pts.loc[order].gap, cmap='RdBu_r',
                   norm=TwoSlopeNorm(0, -lim_a, lim_a), s=3, linewidths=0)
ax[0].set_title(f'A  Gap at each site ({len(pts):,})', loc='left')
cb = fig.colorbar(sc, ax=ax[0], orientation='horizontal', fraction=.04, pad=.01, extend='both'); cb.set_label(r'$\Delta = V - L$')
# (B) median gap by community district
cdm.plot(ax=ax[1], column='med_gap', cmap='RdBu_r', norm=TwoSlopeNorm(0, -lim_b, lim_b), edgecolor='0.3', linewidth=.4)
for _, r in cdm.iterrows():
    p = r.geometry.representative_point(); ax[1].annotate(f'{int(r.cdn)}', (p.x, p.y), ha='center', va='center', fontsize=6.5)
sm = plt.cm.ScalarMappable(cmap='RdBu_r', norm=TwoSlopeNorm(0, -lim_b, lim_b)); sm.set_array([])
cb = fig.colorbar(sm, ax=ax[1], orientation='horizontal', fraction=.04, pad=.01); cb.set_label(r'median $\Delta$ by district')
ax[1].set_title('B  Median gap by district', loc='left')
# (C) hidden-high sites on tract SVI
tr.plot(ax=ax[2], column='RPL_THEMES', cmap='YlOrRd', vmin=0, vmax=1, edgecolor='white', linewidth=.2, missing_kwds={'color': '0.9'})
cd.boundary.plot(ax=ax[2], color='0.45', linewidth=.4)
ax[2].scatter(pts.geometry.x[~hh.values], pts.geometry.y[~hh.values], s=1.5, c='0.55', linewidths=0)
ax[2].scatter(pts.geometry.x[hh.values], pts.geometry.y[hh.values], s=4, c='#08306b', linewidths=0)
sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=plt.Normalize(0, 1)); sm.set_array([])
cb = fig.colorbar(sm, ax=ax[2], orientation='horizontal', fraction=.04, pad=.01); cb.set_label('tract SVI, 2022 percentile')
ax[2].set_title(f'C  Hidden-high sites ({int(hh.sum())})', loc='left')
fig.tight_layout(); fig.savefig(f'{FIG}/F5_map_delta_E1.png', dpi=300, bbox_inches='tight', pad_inches=0.05); plt.close()

print(f'saved {FIG}/F5_map_delta_E1.png')
print(f'sites {len(pts)} (paper: 1,180) | hidden-high {int(hh.sum())} (paper: 239) | district from: {used}')
print('median gap by community district (compare with the district table in the paper, e.g. CD1 +0.431, CD12 -0.151):')
print(med.round(3).to_string())

In [ ]:
# ============ FIGURES 1, 5 and 6 redrawn at print size (files F0_study_area.png, F4_rank_delta_vs_negL.png, F8_expected_vs_observed.png) ============
# Each figure is drawn at the width it will be printed (about 6.3 in for full width), so 7-8 pt text stays 7-8 pt on the page;
# saved at 600 dpi so every figure stays above 500 ppi as placed in the manuscript.
# Replaces colab_cell_F8.py. Needs R_series_results/expected_visitors_E1.parquet and SVI_US_2022.csv in PAPER_DIR;
# Figure 1 downloads tract and community-district boundaries (or uses local copies in PAPER_DIR if present).
# Each figure is in its own try block, so one failure does not stop the others; read the printed checks against the paper.
import os, re, numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from scipy.stats import spearmanr
if str(globals().get('DATA_DIR', 'data')).startswith('/content/drive'):
    from google.colab import drive; drive.mount('/content/drive')
PAPER_DIR = globals().get('PAPER_DIR', globals().get('DATA_DIR', 'data'))
FIG = f'{PAPER_DIR}/R_series_results/figures'; os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({'font.size': 8, 'axes.titlesize': 8.5, 'axes.labelsize': 8, 'xtick.labelsize': 7.5,
                     'ytick.labelsize': 7.5, 'legend.fontsize': 7.5, 'figure.dpi': 100})
NPRAC = 'n_pr'   # Figure 1: column with the number of practices at each site (n_pr in expected_visitors_E1.parquet)

X = pd.read_parquet(f'{PAPER_DIR}/R_series_results/expected_visitors_E1.parquet')
X['gap'] = X.V - X.L
Q = int(len(X) * .25)                                                       # 295 for 1,180 sites
def save(fig, name):
    fig.savefig(f'{FIG}/{name}', dpi=600, bbox_inches='tight', pad_inches=0.05); plt.close(fig); print(f'  saved {FIG}/{name}')

# ---------------- Figure 5: rank by gap against rank by address ----------------
try:
    print('Figure 5 (F4_rank_delta_vs_negL.png)')
    ra = X.L.rank(method='first')                                          # 1 = lowest tract SVI (address-only ranking)
    rg = X.gap.rank(ascending=False, method='first')                       # 1 = largest gap
    inA, inG = ra <= Q, rg <= Q
    prom, dem = inG & ~inA, inA & ~inG
    n = len(X)
    fig, ax = plt.subplots(figsize=(3.9, 3.7))
    ax.scatter(ra[~prom & ~dem], rg[~prom & ~dem], s=2.5, c='0.72', lw=0)
    ax.scatter(ra[prom], rg[prom], s=7, c='#1f77b4', lw=0, label=f'promoted ({int(prom.sum())})')
    ax.scatter(ra[dem], rg[dem], s=7, c='#d62728', lw=0, label=f'demoted ({int(dem.sum())})')
    ax.plot([1, n], [1, n], 'k:', lw=.6)
    ax.axhline(Q + .5, c='k', ls='--', lw=.6); ax.axvline(Q + .5, c='k', ls='--', lw=.6)
    ax.set_xlim(n + 25, -25); ax.set_ylim(n + 25, -25)                     # rank 1 at the right and at the top
    ax.text(n - 10, Q - 12, f'rank {Q}', ha='left', va='bottom', fontsize=7)                  # above the horizontal line, at the left
    ax.text(Q - 12, n - 10, f'rank {Q}', ha='left', va='bottom', fontsize=7, rotation=90)     # right of the vertical line, at the bottom
    ax.set_xlabel('rank by $-L$ (address only; 1 = lowest tract SVI)')
    ax.set_ylabel(r'rank by $\Delta$ (with phone data; 1 = largest gap)')
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(loc='upper left', frameon=False, markerscale=1.6, handletextpad=.2, borderaxespad=.2)
    fig.tight_layout(); save(fig, 'F4_rank_delta_vs_negL.png')
    print(f'  top quarter {Q} (paper: 295) | shared {int((inA & inG).sum())} (paper: 234) | promoted {int(prom.sum())}, '
          f'demoted {int(dem.sum())} (paper: 61 each) | rank corr gap vs -L {spearmanr(X.gap, -X.L).correlation:.2f} (paper: 0.90)')
    print('  if shared/promoted differ slightly from the paper, ties in L are being broken differently from the original cell; keep the old figure in that case')
except Exception as e:
    print(f'  Figure 5 not redrawn: {e}')

# ---------------- Figure 6: observed against expected, and U by specialty ----------------
try:
    print('Figure 6 (F8_expected_vs_observed.png)')
    tU = X.index.isin(X.R.nlargest(Q).index); bU = X.index.isin(X.R.nsmallest(Q).index)
    r2 = 1 - ((X.V - X.V_hat) ** 2).sum() / ((X.V - X.V.mean()) ** 2).sum()
    fig, ax = plt.subplots(1, 2, figsize=(6.3, 2.9), gridspec_kw={'width_ratios': [1.05, 1]})
    ax[0].scatter(X.V_hat[~tU & ~bU], X.V[~tU & ~bU], s=3, c='0.75', lw=0, label='other sites')
    ax[0].scatter(X.V_hat[tU], X.V[tU], s=4, c='#c0392b', lw=0, label='top quarter by $U$')
    ax[0].scatter(X.V_hat[bU], X.V[bU], s=4, c='#1f77b4', lw=0, label='bottom quarter by $U$')
    lim = [min(X.V_hat.min(), X.V.min()) - .02, max(X.V_hat.max(), X.V.max()) + .02]
    ax[0].plot(lim, lim, 'k:', lw=.7); ax[0].set_xlim(lim); ax[0].set_ylim(lim)
    ax[0].set_xlabel(r'expected, $\hat{V}$ (out of fold)'); ax[0].set_ylabel('observed visitor-origin SVI, $V$')
    ax[0].set_title(f'A  Observed vs expected ($R^2$ = {r2:.2f})', loc='left')
    ax[0].legend(loc='upper left', frameon=False, fontsize=7, markerscale=2, handletextpad=.2, borderaxespad=.2)
    names = {'Physician': 'Physician', 'Mental health': 'Mental\nhealth', 'Other practitioner': 'Other\npractitioner',
             'Outpatient centre': 'Outpatient\ncentre'}
    order = [c for c in names if (X.spec == c).sum() >= 10]
    ax[1].boxplot([X.R[X.spec == c] for c in order], showfliers=False, widths=.55,
                  medianprops={'color': 'k', 'lw': 1}, boxprops={'lw': .7}, whiskerprops={'lw': .7}, capprops={'lw': .7})
    ax[1].axhline(0, c='k', lw=.5, ls=':')
    ax[1].set_xticks(range(1, len(order) + 1))
    ax[1].set_xticklabels([f'{names[c]}\n(n = {(X.spec == c).sum()})' for c in order], fontsize=7)
    ax[1].set_ylabel(r'$U = V - \hat{V}$'); ax[1].set_title('B  Unpredicted part by specialty', loc='left')
    for a in ax: a.spines[['top', 'right']].set_visible(False)
    fig.tight_layout(w_pad=1.5); save(fig, 'F8_expected_vs_observed.png')
    print(f'  out-of-fold R2 {r2:.2f} (paper: 0.36) | specialty counts ' + ', '.join(f'{c} {(X.spec == c).sum()}' for c in order))
except Exception as e:
    print(f'  Figure 6 not redrawn: {e}')

# ---------------- Figure 1: study area ----------------
try:
    print('Figure 1 (F0_study_area.png)')
    import geopandas as gpd
    cand = [c for c in X.columns if re.fullmatch(r'(n_?|num_?|count_?)(pr\w*|poi\w*|places?)|practices|n', c, re.I)]
    col = NPRAC if NPRAC in X.columns else (cand[0] if cand else None)
    if col is None:
        raise KeyError(f'no column with the number of practices per site was found; the columns are {list(X.columns)}. '
                       'Set NPRAC to the right name near the top of this cell and rerun.')
    npr = pd.to_numeric(X[col], errors='coerce').fillna(1).clip(lower=1)
    # boundaries: local copies in PAPER_DIR first, otherwise download
    tsrc = f'{PAPER_DIR}/tl_2022_36_tract.zip'
    tr = gpd.read_file(tsrc if os.path.exists(tsrc) else 'https://www2.census.gov/geo/tiger/TIGER2022/TRACT/tl_2022_36_tract.zip')
    tr = tr[tr.COUNTYFP == '061'].to_crs('EPSG:2263')
    svi = pd.read_csv(f'{PAPER_DIR}/SVI_US_2022.csv', low_memory=False, usecols=['FIPS', 'RPL_THEMES'], dtype={'FIPS': str})
    svi['FIPS'] = svi.FIPS.str.zfill(11)
    tr = tr.merge(svi, left_on='GEOID', right_on='FIPS', how='left'); tr.loc[tr.RPL_THEMES < 0, 'RPL_THEMES'] = np.nan
    cd = None
    for src in [f'{PAPER_DIR}/community_districts.geojson',
                'https://data.cityofnewyork.us/api/geospatial/5crt-au7u?method=export&format=GeoJSON',
                'https://data.cityofnewyork.us/resource/5crt-au7u.geojson?$limit=100']:
        if src.startswith('/') and not os.path.exists(src): continue
        try:
            cd = gpd.read_file(src); break
        except Exception as e:
            print(f'  community districts not read from {src[:60]}...: {e}')
    if cd is None:
        raise RuntimeError('community district boundaries could not be read; save NYC Open Data dataset 5crt-au7u as '
                           'community_districts.geojson in PAPER_DIR and rerun')
    bcol = [c for c in cd.columns if c.lower().replace('_', '') == 'borocd'][0]
    cd['cdn'] = cd[bcol].astype(int); cd = cd[cd.cdn.between(101, 112)].to_crs('EPSG:2263'); cd['cdn'] -= 100
    pts = gpd.GeoDataFrame({'npr': npr.values}, geometry=gpd.points_from_xy(X.lon, X.lat), crs='EPSG:4326').to_crs('EPSG:2263')
    pts = gpd.sjoin(pts, cd[['cdn', 'geometry']], how='left', predicate='within').drop(columns='index_right')
    cnt = pts.groupby('cdn').size()
    cdm = cd.merge(cnt.rename('sites'), left_on='cdn', right_index=True, how='left')

    size = lambda v: 1.5 + 1.6 * np.sqrt(v)
    fig, ax = plt.subplots(1, 2, figsize=(5.4, 4.9))
    for a in ax: a.set_axis_off(); a.set_aspect('equal')
    # (A) tract SVI and sites
    tr.plot(ax=ax[0], column='RPL_THEMES', cmap='YlOrRd', vmin=0, vmax=1, edgecolor='white', linewidth=.15,
            missing_kwds={'color': '0.88'})
    cd.boundary.plot(ax=ax[0], color='0.35', linewidth=.4)
    o = pts.npr.sort_values(ascending=False).index                          # large symbols underneath
    ax[0].scatter(pts.loc[o].geometry.x, pts.loc[o].geometry.y, s=size(pts.loc[o].npr), c='k', alpha=.45, linewidths=0)
    hs = [ax[0].scatter([], [], s=size(v), c='k', alpha=.45, linewidths=0, label=f'{v}') for v in (1, 10, 50) if v <= npr.max()]
    ax[0].legend(handles=hs, title='practices', loc='lower right', frameon=False, fontsize=7, title_fontsize=7, labelspacing=.6)
    sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=plt.Normalize(0, 1)); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax[0], orientation='horizontal', fraction=.04, pad=.01, shrink=.85); cb.set_label('tract SVI, 2022 national percentile')
    ax[0].set_title(f'A  Tract SVI and {len(pts):,} clinic sites', loc='left')
    # (B) sites by community district
    vmax = float(np.ceil(cnt.max() / 50) * 50)
    cdm.plot(ax=ax[1], column='sites', cmap='Blues', vmin=0, vmax=vmax, edgecolor='0.3', linewidth=.4)
    for _, r in cdm.iterrows():
        p = r.geometry.representative_point()
        ax[1].annotate(f'{int(r.cdn)}', (p.x, p.y), ha='center', va='center', fontsize=7.5,
                       path_effects=[pe.withStroke(linewidth=2, foreground='white')])
    sm = plt.cm.ScalarMappable(cmap='Blues', norm=plt.Normalize(0, vmax)); sm.set_array([])
    cb = fig.colorbar(sm, ax=ax[1], orientation='horizontal', fraction=.04, pad=.01, shrink=.85); cb.set_label('clinic sites')
    ax[1].set_title('B  Clinic sites by community district', loc='left')
    fig.tight_layout(); save(fig, 'F0_study_area.png')
    table5 = {1: 109, 2: 84, 3: 32, 4: 61, 5: 347, 6: 158, 7: 95, 8: 182, 9: 23, 10: 21, 11: 34, 12: 34}
    diff = {k: int(cnt.get(k, 0)) - v for k, v in table5.items() if int(cnt.get(k, 0)) != v}
    print(f'  practices from column "{col}" (total {int(npr.sum()):,}, at most 9,132; largest site {int(npr.max())}) | sites {len(pts)} (paper: 1,180) | '
          f'district counts ' + ('match Table 5' if not diff else f'differ from Table 5 by {diff}'))
except Exception as e:
    print(f'  Figure 1 not redrawn: {e}')